# Dataflow Estate Assessment (DEA)
**Tenant-wide inventory and migration-readiness assessment for Power BI Dataflow Gen1 and Fabric Dataflow Gen2.**

The notebook harvests metadata with Power BI / Fabric REST APIs (service principal preferred), persists a
dimensional model into Delta tables of the attached Lakehouse, then scores every dataflow for a Gen1 -> Gen2
migration: readiness class, complexity score, effort, recommendation and migration wave.

| # | Section | What it does |
|---|---|---|
| 1 | Configuration | Parameters, schema registry, rule catalog, run context, logging, storage layer |
| 2 | Authentication | Service principal / notebook identity tokens, rate-limited resilient API client, preflight |
| 3 | Workspace Discovery | Capacities, workspaces, domains, environments, Scanner API (incremental) |
| 4 | Dataflow Discovery | Gen1 + Gen2 inventory, sources, lineage edges, reconciliation |
| 5 | Dependency Collection | Definitions (model.json / mashup), M analysis, extra lineage, owners |
| 6 | Refresh History Collection | Gen1 transactions, Gen2 job instances, schedules, refresh KPIs |
| 7 | Migration Readiness Analysis | Rules, complexity score, readiness, dependency closure, waves |
| 8 | Delta Persistence | Publish Gold layer, verify, maintain tables |
| 9 | Reporting Output | KPIs, wave plan, exports (CSV/XLSX), lineage graph |
| 10 | Logging and Monitoring | Run summary, API statistics, error report, status |

**Execution model.** Cells are written to be run top to bottom (or as a pipeline notebook activity).
Collectors persist to Delta after every micro-batch and record checkpoints, so a failed or time-boxed run can
simply be re-run: finished work is skipped and API budgets are spent on what is missing or changed.
The storage layer used by the collectors is defined in section 1.4; section 8 publishes and maintains the Gold layer.

## Prerequisites
1. **Lakehouse** attached as the default Lakehouse of this notebook (schema-enabled recommended; default schema `dbo`).
2. **Service principal** (recommended) stored in Azure Key Vault - secrets: tenant id, client id, client secret.
   * Fabric admin portal > Tenant settings: *Service principals can access read-only admin APIs*,
     *Service principals can call Fabric public APIs*, and for rich Scanner output *Enhance admin APIs responses with
     detailed metadata* (+ *...with DAX and mashup expressions*). Scope each to a security group that contains the SPN.
   * The app registration must **not** have admin-consent Power BI API permissions configured.
   * Optional deep inspection (definitions, refresh history) needs the SPN as workspace member; without it those
     items are recorded as `AccessDenied` and the assessment degrades gracefully (confidence is lowered).
3. Fallback `p_auth_mode = "user"` uses the identity running the notebook (must be a Fabric administrator).
4. Optional governance input: `Files/dea/config/governance_overrides.csv` (see docs/04-runbook.md).

**Never put secrets in this notebook.** Only Key Vault secret *names* are parameters.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Parameters (pipeline-overridable). Lists accept comma/semicolon separated text; dicts accept JSON text.
# ---------------------------------------------------------------------------------------------------------------
# Run control
p_run_mode = "incremental"          # full | incremental | resume | analysis_only | refresh_only
p_resume_run_id = ""                # run id to continue when p_run_mode = "resume"
p_dry_run = False                   # collect + analyse without writing Delta tables
p_max_runtime_minutes = 170         # soft deadline: stop starting new work, flush, finish gracefully
p_log_level = "INFO"
p_fail_on_partial = False           # raise at the end when a phase failed (useful for pipelines)

# Storage
p_lakehouse_schema = "dbo"          # "" for Lakehouses without schemas
p_files_root = "Files/dea"
p_persist_raw = True                # keep raw Scanner JSON in Files for audit / reprocessing
p_raw_retention_days = 30
p_run_maintenance = True            # OPTIMIZE (+ optional VACUUM) at the end
p_vacuum_retention_hours = 0        # 0 = skip VACUUM

# Authentication
p_auth_mode = "auto"                # auto | spn | user
p_key_vault_url = ""                # https://<vault>.vault.azure.net/
p_kv_secret_tenant_id = "dea-tenant-id"
p_kv_secret_client_id = "dea-client-id"
p_kv_secret_client_secret = "dea-client-secret"
p_authority_host = "https://login.microsoftonline.com"
p_pbi_api_base = "https://api.powerbi.com/v1.0/myorg"
p_fabric_api_base = "https://api.fabric.microsoft.com/v1"
p_graph_api_base = "https://graph.microsoft.com/v1.0"

# Scope
p_only_workspaces = ""              # names or ids; empty = whole tenant
p_exclude_workspaces = ""
p_source_capacity = ""              # capacity name or id; empty = all
p_include_personal_workspaces = False
p_include_inactive_workspaces = False

# Collection switches
p_collect_gen1 = True
p_collect_gen2 = True
p_collect_definitions = True        # model.json (Gen1) - deep inspection, API-budgeted
p_definition_source = "admin"       # admin | workspace: which API reads Gen1 definitions first. The admin export is capped at 200 requests/hour per
                                    # principal; "workspace" (identity is a workspace member) is far faster and falls back to the admin export
p_collect_gen2_definitions = False  # getDefinition needs read+write on the item (Contributor+)
p_collect_compute_engine = True     # workspace dataflow list: enhanced compute engine setting (needs workspace access)
p_collect_refresh_history = True
p_collect_admin_datasources = False # admin/dataflows/{id}/datasources for dataflows the Scanner did not describe
p_verify_upstream_admin_api = False # cross-check upstream links with admin upstreamDataflows (200 req/h)
p_collect_pipeline_references = False
p_collect_item_connections = False  # Fabric items/{id}/connections (Gen2, needs read+write)
p_enrich_owners_graph = False       # Microsoft Graph owner enrichment (needs User.Read.All)

# Scanner API
p_scan_batch_size = 100             # API maximum is 100
p_scan_max_concurrent = 8           # API maximum is 16
p_scan_poll_interval_sec = 10
p_scan_timeout_sec = 900
p_scan_lookback_days = 28           # incremental window (API allows 30 min .. 30 days)
p_scan_get_users = True
p_scan_dataset_expressions = False

# Throttling / budgets
p_max_workers = 4
p_rate_limit_scale = 0.9            # use 90% of documented hourly limits
p_rate_limit_overrides = ""         # JSON, e.g. {"pbi_admin_dataflow_export": [[100, 3600]]}
p_max_retries = 6
p_retry_base_delay_sec = 2.0
p_retry_max_delay_sec = 120.0
p_max_retry_after_sec = 900         # a longer Retry-After ends the phase (resume later)
p_budget_definitions_per_run = 400    # dataflows whose definition is read per run (the budgets bound the work per run, one to three API calls each)
p_budget_refresh_dataflows_per_run = 1500   # dataflows whose refresh history (and schedule) is collected per run
p_budget_datasources_per_run = 400
p_refresh_history_ttl_hours = 24
p_refresh_history_max_records = 100
p_denied_retry_days = 7
p_failed_retry_days = 1
p_max_attempts_per_entity = 3
p_batch_flush_size = 100

# Analysis
p_stale_days = 90                   # no successful refresh for N days and no consumers -> Deferred
p_failing_consecutive = 5           # N consecutive failed refreshes -> Deferred (fix first)
p_wizard_available = True           # Upgrade Wizard (in-place, preview) may be recommended
p_preferred_path = "auto"           # auto | wizard | saveas
p_require_definition_for_ready = True
p_pilot_size = 10
p_score_weights = ""                # JSON override of default weights
p_governance_overrides_path = "Files/dea/config/governance_overrides.csv"
p_migration_results_table = "dataflowmigrations"   # optional output table of the Save-As accelerator
p_environment_patterns = ""         # JSON override: {"Dev": "regex", "Test": "regex", "Prod": "regex"}

# Reporting
p_export_reports = True
p_top_n = 25
p_render_lineage_graph = True
p_lineage_graph_max_nodes = 60

# 1. Configuration
## 1.1 Schema registry (single source of truth)
Every Delta table is declared once below. The same registry drives table creation (DDL), Spark schemas, row
coercion, `sql/lakehouse_ddl.sql` and `docs/data-dictionary.md` (generated by `tools/build_notebook.py`).

| Layer | Tables |
|---|---|
| Bronze (files) | `Files/dea/raw/<run_id>/...` raw Scanner JSON (optional) |
| Silver (current state, MERGE) | Capacities, Workspaces, Dataflows, DataflowSources, Lineage, RefreshHistory |
| Gold (snapshot per run) | UnsupportedFeatures, DataflowDependencies, MigrationReadiness, DataflowSummary, WavePlan, RunSummary |
| Control / config | RunLog, ErrorLog, Checkpoint, RuleCatalog, GovernanceOverrides |

In [ ]:
import dataclasses
import datetime as dt
import hashlib
import json
import re

_TS_RE = re.compile(r"^(\d{4}-\d{2}-\d{2})[T ](\d{2}:\d{2}:\d{2})(\.\d+)?\s*(Z|[+-]\d{2}:?\d{2})?$")


def parse_ts(value):
    """Parse API timestamps into naive UTC datetimes (Spark TimestampType compatible)."""
    if value in (None, ""):
        return None
    if isinstance(value, dt.datetime):
        if value.tzinfo is not None:
            value = value.astimezone(dt.timezone.utc).replace(tzinfo=None)
        return value
    if isinstance(value, dt.date):
        return dt.datetime(value.year, value.month, value.day)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        seconds = value / 1000.0 if value > 1e11 else float(value)
        return dt.datetime.fromtimestamp(seconds, dt.timezone.utc).replace(tzinfo=None)
    text = str(value).strip()
    match = _TS_RE.match(text)
    if not match:
        try:
            return dt.datetime.strptime(text[:10], "%Y-%m-%d")
        except ValueError:
            return None
    day, clock, frac, tz = match.groups()
    micro = int(((frac or ".0")[1:] + "000000")[:6])
    moment = dt.datetime.strptime(f"{day} {clock}", "%Y-%m-%d %H:%M:%S").replace(microsecond=micro)
    if tz and tz != "Z":
        digits = tz[1:].replace(":", "")
        offset = dt.timedelta(hours=int(digits[:2]), minutes=int(digits[2:]))
        moment = moment - offset if tz[0] == "+" else moment + offset
    return moment


def utcnow():
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)


def stable_hash64(text):
    """Deterministic signed 63-bit surrogate key (stable across runs and engines)."""
    digest = hashlib.blake2b(str(text).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") & 0x7FFFFFFFFFFFFFFF


def short_hash(text, length=16):
    return hashlib.blake2b(str(text).encode("utf-8"), digest_size=16).hexdigest()[:length]


def _as_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    text = str(value).strip().lower()
    if text in ("true", "1", "yes", "y", "t"):
        return True
    if text in ("false", "0", "no", "n", "f"):
        return False
    return None


def coerce_value(ctype, value):
    """Coerce a Python value to the declared column type; unparseable values become None."""
    if value is None:
        return None
    try:
        if ctype == "STRING":
            if isinstance(value, (list, tuple, dict)):
                return json.dumps(value, default=str, sort_keys=True)
            if isinstance(value, bool):
                return "true" if value else "false"
            return str(value)
        if ctype == "BOOLEAN":
            return _as_bool(value)
        if ctype in ("INT", "BIGINT"):
            if isinstance(value, (bool, int)):
                number = int(value)
            elif isinstance(value, float):
                number = int(value)
            else:
                text = str(value).strip()
                try:
                    number = int(text)
                except ValueError:
                    number = int(float(text))
            limit = 2 ** 31 if ctype == "INT" else 2 ** 63
            return number if -limit <= number < limit else None
        if ctype == "DOUBLE":
            number = float(value)
            return None if number != number else number
        if ctype == "TIMESTAMP":
            return parse_ts(value)
        if ctype == "DATE":
            moment = parse_ts(value)
            return moment.date() if moment else None
        if ctype == "ARRAY<STRING>":
            if isinstance(value, set):
                value = sorted(value, key=str)
            if isinstance(value, (list, tuple)):
                return [x if isinstance(x, str) else json.dumps(x, default=str, sort_keys=True) for x in value if x is not None]
            return [str(value)]
    except (TypeError, ValueError, OverflowError):
        return None
    raise ValueError(f"unsupported column type {ctype}")


_AUDIT = {
    "scd": (
        ("first_seen_at", "TIMESTAMP", "First run that observed the row"),
        ("last_seen_at", "TIMESTAMP", "Last run that observed the row"),
        ("is_deleted", "BOOLEAN", "Soft delete flag: row no longer returned by the source"),
        ("deleted_at", "TIMESTAMP", "When the row was flagged deleted"),
        ("run_id", "STRING", "Run that last wrote the row"),
    ),
    "run": (
        ("run_id", "STRING", "Run that wrote the row"),
        ("updated_at", "TIMESTAMP", "UTC time the row was written"),
    ),
    "none": (),
}


@dataclasses.dataclass(frozen=True)
class TableDef:
    name: str
    layer: str
    grain: str
    keys: tuple
    cols: tuple
    mode: str = "merge"
    audit: str = "none"
    partition_by: tuple = ()
    zorder_by: tuple = ()
    doc: str = ""

    def all_cols(self):
        return tuple(self.cols) + _AUDIT[self.audit]

    @property
    def col_names(self):
        return [c[0] for c in self.all_cols()]

    @property
    def soft_delete(self):
        return self.audit == "scd"


def coerce_row(table, row):
    out = {}
    for name, ctype, _ in table.all_cols():
        out[name] = coerce_value(ctype, row.get(name))
    return out


def qualified(schema, name):
    return f"{schema}.{name}" if schema else name


def _sql_comment(text):
    return text.replace("'", "`").replace("\\", "/")


def ddl_for(table, schema=""):
    lines = [f"  {n} {t} COMMENT '{_sql_comment(d)}'" for n, t, d in table.all_cols()]
    ddl = f"CREATE TABLE IF NOT EXISTS {qualified(schema, table.name)} (\n" + ",\n".join(lines) + "\n) USING DELTA"
    if table.partition_by:
        ddl += f"\nPARTITIONED BY ({', '.join(table.partition_by)})"
    ddl += f"\nCOMMENT '{_sql_comment(table.grain)}';"
    return ddl

In [ ]:
# Column tuples are (name, type, description). Descriptions avoid quotes and pipes (they feed DDL and Markdown).
TABLES = {}


def _table(name, layer, grain, keys, cols, **kw):
    TABLES[name] = TableDef(name=name, layer=layer, grain=grain, keys=tuple(keys), cols=tuple(cols), **kw)


_table("Assessment_Capacities", "silver", "One row per capacity", ["capacity_id"], [
    ("capacity_id", "STRING", "Capacity GUID"),
    ("capacity_name", "STRING", "Capacity display name"),
    ("sku", "STRING", "SKU, e.g. F64, P1, FT1, A4, EM3"),
    ("capacity_kind", "STRING", "Fabric F / Premium P / Trial / Embedded / Unknown"),
    ("is_fabric_capable", "BOOLEAN", "True when the SKU can host Fabric items such as Dataflow Gen2"),
    ("region", "STRING", "Azure region"),
    ("state", "STRING", "Capacity state"),
    ("admins", "ARRAY<STRING>", "Capacity admin UPNs"),
    ("workspace_count", "INT", "Workspaces assigned"),
    ("gen1_dataflow_count", "INT", "Gen1 dataflows hosted"),
    ("gen2_dataflow_count", "INT", "Gen2 dataflows hosted"),
    ("source", "STRING", "Where the row came from: admin, core or derived"),
], audit="scd", doc="Capacity dimension. Capacities referenced by workspaces but not returned by the APIs are kept as derived rows with capacity_name NULL.")

_table("Assessment_Workspaces", "silver", "One row per workspace", ["workspace_id"], [
    ("workspace_key", "BIGINT", "Deterministic surrogate key"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("workspace_name", "STRING", "Workspace name"),
    ("workspace_type", "STRING", "Workspace / Group / PersonalGroup / AdminWorkspace"),
    ("state", "STRING", "Active / Deleted / Removing ..."),
    ("is_personal", "BOOLEAN", "Personal workspace flag"),
    ("description", "STRING", "Workspace description"),
    ("capacity_id", "STRING", "Assigned capacity GUID"),
    ("capacity_name", "STRING", "Capacity display name when resolvable"),
    ("capacity_sku", "STRING", "Capacity SKU when resolvable"),
    ("capacity_resolution", "STRING", "Resolved / Unresolved / Shared"),
    ("is_on_dedicated_capacity", "BOOLEAN", "Workspace on Premium or Fabric capacity"),
    ("is_fabric_capable", "BOOLEAN", "Capacity can host Fabric items"),
    ("domain_id", "STRING", "Fabric domain GUID"),
    ("domain_name", "STRING", "Fabric domain name"),
    ("default_dataset_storage_format", "STRING", "Default semantic model storage format"),
    ("has_workspace_level_settings", "BOOLEAN", "Workspace level settings present"),
    ("dataflow_storage_id", "STRING", "Attached ADLS Gen2 storage (BYOSA) id when set"),
    ("pipeline_id", "STRING", "Deployment pipeline GUID when assigned"),
    ("pipeline_stage_order", "INT", "Deployment pipeline stage order (0 = first stage)"),
    ("environment", "STRING", "Dev / Test / Prod / Unknown"),
    ("environment_source", "STRING", "Override / DeploymentPipeline / NamingConvention / Unknown"),
    ("workspace_classification", "STRING", "Override, else domain name, else Unclassified"),
    ("support_team", "STRING", "Support team from governance overrides"),
    ("business_criticality", "STRING", "Workspace level criticality override"),
    ("admin_upns", "ARRAY<STRING>", "Workspace admins"),
    ("admin_count", "INT", "Number of admins"),
    ("tags", "ARRAY<STRING>", "Workspace tags"),
    ("gen1_dataflow_count", "INT", "Gen1 dataflows in the workspace"),
    ("gen2_dataflow_count", "INT", "Gen2 dataflows in the workspace"),
    ("dataset_count", "INT", "Semantic models"),
    ("report_count", "INT", "Reports"),
    ("dashboard_count", "INT", "Dashboards"),
    ("other_item_count", "INT", "Other Fabric items seen by the Scanner"),
    ("scanned_at", "TIMESTAMP", "Last successful Scanner pass"),
    ("scan_state", "STRING", "Scanned = current as of the scan watermark; Failed = retried by every run; NotScanned = not scanned yet, outside the scope, or not returned by the Scanner (inactive workspace)"),
], audit="scd", zorder_by=("workspace_id",), doc="Workspace dimension with capacity, domain, environment and governance attributes.")

_table("Assessment_Dataflows", "silver", "One row per dataflow (Gen1 and Gen2)", ["dataflow_id"], [
    ("dataflow_key", "BIGINT", "Deterministic surrogate key"),
    ("dataflow_id", "STRING", "Dataflow GUID (objectId); unchanged by the in-place Upgrade Wizard"),
    ("workspace_id", "STRING", "Owning workspace GUID"),
    ("dataflow_name", "STRING", "Dataflow name"),
    ("dataflow_type", "STRING", "Gen1 / Gen2 / Unknown"),
    ("generation", "INT", "1 or 2 when known"),
    ("is_cicd", "BOOLEAN", "Gen2 CI/CD flavour when known"),
    ("description", "STRING", "Description"),
    ("tags", "ARRAY<STRING>", "Tags when available"),
    ("owner_upn", "STRING", "Owner: configuredBy (Gen1) or creator (Gen2)"),
    ("owner_source", "STRING", "configuredBy / creatorPrincipal / override"),
    ("owner_display_name", "STRING", "Owner display name (Microsoft Graph enrichment)"),
    ("owner_enabled", "BOOLEAN", "Owner account enabled (Microsoft Graph enrichment)"),
    ("modified_by", "STRING", "Last modifier"),
    ("created_at", "TIMESTAMP", "Creation time when exposed"),
    ("modified_at", "TIMESTAMP", "Last modification time"),
    ("model_url", "STRING", "Definition location; dfs.core.windows.net indicates attached storage (BYOSA)"),
    ("storage_type", "STRING", "PowerBIManaged / CustomADLS / Unknown"),
    ("compute_engine_behavior", "STRING", "Enhanced compute engine setting when known"),
    ("endorsement", "STRING", "None / Promoted / Certified / Master data"),
    ("certified_by", "STRING", "Certifier"),
    ("sensitivity_label_id", "STRING", "Sensitivity label GUID"),
    ("schedule_enabled", "BOOLEAN", "Refresh schedule enabled (NULL = unknown)"),
    ("schedule_days", "ARRAY<STRING>", "Scheduled days"),
    ("schedule_times", "ARRAY<STRING>", "Scheduled times"),
    ("schedule_runs_per_day", "INT", "Runs per scheduled day"),
    ("schedule_timezone", "STRING", "Schedule time zone"),
    ("schedule_notify_option", "STRING", "Failure notification option"),
    ("last_refresh_at", "TIMESTAMP", "Start of the latest observed refresh"),
    ("last_refresh_status", "STRING", "Success / Failed / Cancelled / InProgress / Skipped / Unknown"),
    ("last_refresh_duration_sec", "DOUBLE", "Duration of the latest refresh"),
    ("last_success_at", "TIMESTAMP", "End of the latest successful refresh"),
    ("refresh_runs_observed", "INT", "Refresh runs in the observed window"),
    ("refresh_success_rate", "DOUBLE", "Share of successful runs in the window (0-1)"),
    ("refresh_avg_duration_sec", "DOUBLE", "Average duration of finished runs"),
    ("refresh_p95_duration_sec", "DOUBLE", "95th percentile duration of finished runs"),
    ("consecutive_failures", "INT", "Failed runs since the latest success"),
    ("last_error_message", "STRING", "Latest refresh error"),
    ("external_trigger_seen", "BOOLEAN", "Refreshes triggered through API / on demand were observed"),
    ("refresh_history_status", "STRING", "Collected / AccessDenied / NotAttempted / Failed"),
    ("refresh_history_collected_at", "TIMESTAMP", "When refresh history was last collected"),
    ("source_count", "INT", "Distinct data sources"),
    ("source_type_count", "INT", "Distinct data source types"),
    ("gateway_count", "INT", "Distinct gateways"),
    ("uses_gateway", "BOOLEAN", "At least one source goes through a gateway"),
    ("uses_on_prem_gateway", "BOOLEAN", "At least one source uses an on-premises gateway"),
    ("query_count", "INT", "Queries (excluding parameters and functions)"),
    ("entity_count", "INT", "Entities / loaded queries"),
    ("custom_function_count", "INT", "Custom M functions"),
    ("parameter_count", "INT", "Parameters"),
    ("incremental_refresh_count", "INT", "Entities with incremental refresh"),
    ("computed_entity_count", "INT", "Entities computed from other entities"),
    ("linked_entity_count", "INT", "Entities linked from other dataflows"),
    ("directquery_entity_count", "INT", "Entities in DirectQuery mode"),
    ("native_query_count", "INT", "Native queries (Value.NativeQuery / Query option)"),
    ("legacy_connector_used", "BOOLEAN", "M text uses the legacy PowerBI.Dataflows connector"),
    ("ai_function_count", "INT", "AI Insights / Azure ML / Cognitive Services functions used"),
    ("definition_findings", "STRING", "JSON evidence from the definition: incremental / computed / linked / native / DirectQuery entities, AI functions"),
    ("m_script_chars", "INT", "Size of the M document"),
    ("definition_status", "STRING", "Collected / AccessDenied / NotAttempted / Failed / NotApplicable"),
    ("definition_collected_at", "TIMESTAMP", "When the definition was inspected"),
    ("definition_hash", "STRING", "Hash of the inspected definition"),
    ("first_seen_dataflow_type", "STRING", "Type when first observed (Gen1 -> Gen2 flips mean migrated)"),
    ("type_changed_at", "TIMESTAMP", "When the dataflow type changed"),
    ("discovery_sources", "ARRAY<STRING>", "Scanner / AdminItems / AdminDataflows / WorkspaceApi"),
], audit="scd", zorder_by=("workspace_id", "dataflow_id"), doc="Observed facts per dataflow. Derived judgements live in Assessment_MigrationReadiness.")

_table("Assessment_DataflowSources", "silver", "One row per dataflow and data source", ["dataflow_id", "source_key"], [
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("source_key", "STRING", "Hash of type plus normalized connection details"),
    ("datasource_id", "STRING", "Power BI datasource id when known"),
    ("datasource_type", "STRING", "Connector type, e.g. Sql, Web, SharePointList"),
    ("source_category", "STRING", "Database / File / Web / Cloud storage / SaaS / Dataflow / Fabric / Other"),
    ("server", "STRING", "Server from connection details"),
    ("database", "STRING", "Database from connection details"),
    ("url", "STRING", "URL from connection details"),
    ("path", "STRING", "Path from connection details"),
    ("host", "STRING", "Normalized host used for grouping"),
    ("connection_details", "STRING", "Raw connection details JSON"),
    ("gateway_id", "STRING", "Gateway GUID when the source goes through a gateway"),
    ("gateway_name", "STRING", "Gateway name when resolvable"),
    ("gateway_type", "STRING", "OnPremises / VirtualNetwork / Unknown"),
    ("uses_gateway", "BOOLEAN", "Source uses a gateway"),
    ("is_on_premises", "BOOLEAN", "Source is on premises (gateway or file/folder based)"),
    ("credential_type", "STRING", "Credential type when accessible"),
    ("connection_id", "STRING", "Fabric connection id (Gen2)"),
    ("is_misconfigured", "BOOLEAN", "Reported as misconfigured by the Scanner"),
    ("discovered_via", "STRING", "Scanner / AdminDatasources / Definition / ItemConnections"),
], audit="scd", doc="Connectivity inventory with gateway usage, source categories and credential types where accessible.")

_table("Assessment_Lineage", "silver", "One row per lineage edge", ["edge_id"], [
    ("edge_id", "STRING", "Hash of source, target and relationship"),
    ("src_type", "STRING", "Upstream asset type: Datasource / Dataflow / SemanticModel / Report / Dashboard / DataPipeline / ..."),
    ("src_id", "STRING", "Upstream asset id"),
    ("src_workspace_id", "STRING", "Upstream workspace id"),
    ("src_name", "STRING", "Upstream asset name"),
    ("dst_type", "STRING", "Downstream asset type"),
    ("dst_id", "STRING", "Downstream asset id"),
    ("dst_workspace_id", "STRING", "Downstream workspace id"),
    ("dst_name", "STRING", "Downstream asset name"),
    ("relationship", "STRING", "LinkedEntity / DataflowConnector / DatasetSource / ReportOnDataset / DashboardTile / PipelineActivity / DatasourceUsage"),
    ("edge_kind", "STRING", "DATA (data moves src to dst) or CONTROL (dst orchestrates or depends on src)"),
    ("evidence", "STRING", "Scanner / AdminUpstream / Definition / MParse / PipelineDefinition"),
    ("attributes", "STRING", "JSON edge facts, e.g. storage_mode (semantic model) or legacy_connector (consumer M text)"),
], audit="scd", doc="Generic impact graph. An edge src -> dst means a change to src impacts dst.")

_table("Assessment_RefreshHistory", "silver", "One row per refresh execution", ["dataflow_id", "refresh_id"], [
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("refresh_id", "STRING", "Transaction id (Gen1) or job instance id (Gen2)"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("dataflow_type", "STRING", "Gen1 / Gen2"),
    ("source_api", "STRING", "PBI.transactions or Fabric.jobInstances"),
    ("refresh_type", "STRING", "Scheduled / OnDemand / ViaApi / Manual ..."),
    ("status", "STRING", "Raw status"),
    ("status_normalized", "STRING", "Success / Failed / Cancelled / InProgress / Skipped / Unknown"),
    ("start_time", "TIMESTAMP", "Start (UTC)"),
    ("end_time", "TIMESTAMP", "End (UTC)"),
    ("duration_sec", "DOUBLE", "End minus start"),
    ("error_code", "STRING", "Error code when failed"),
    ("error_message", "STRING", "Error message when failed"),
    ("start_date", "DATE", "Start date"),
    ("start_month", "STRING", "Start month yyyy-MM (partition column)"),
], audit="run", partition_by=("start_month",), doc="Append-only refresh log, MERGE by (dataflow_id, refresh_id). Partitioned by month.")

_table("Assessment_UnsupportedFeatures", "gold", "One row per dataflow and detected rule hit", ["dataflow_id", "rule_id", "entity_name"], [
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("rule_id", "STRING", "Rule id from Assessment_RuleCatalog"),
    ("feature_code", "STRING", "Feature family, e.g. IncrementalRefresh, LinkedEntities, Gateway, AttachedStorage"),
    ("feature_name", "STRING", "Human readable feature"),
    ("category", "STRING", "Storage / Refresh / Lineage / Connectivity / Capacity / Transform / Data quality / Governance"),
    ("severity", "STRING", "Blocker / Major / Minor / Info"),
    ("impact_wizard", "STRING", "Effect on the Upgrade Wizard path: Unsupported / NeedsAttention / Deferred / None"),
    ("impact_saveas", "STRING", "Effect on the Save-As path: Unsupported / NeedsAttention / Deferred / None"),
    ("entity_name", "STRING", "Query or entity concerned (empty when dataflow level)"),
    ("evidence", "STRING", "What was detected"),
    ("remediation", "STRING", "Recommended action"),
    ("effort_days", "DOUBLE", "Estimated remediation effort"),
    ("weight", "INT", "Weight in the unsupported-features score"),
    ("confidence", "STRING", "Verified / Heuristic"),
    ("detected_at", "TIMESTAMP", "Detection time"),
], audit="run", mode="snapshot", doc="Rule hits (unsupported or risky features) per dataflow.")

_table("Assessment_LineageNodes", "gold", "One row per lineage node", ["node_id"], [
    ("node_id", "STRING", "Asset id"),
    ("node_type", "STRING", "Datasource / Dataflow / SemanticModel / Report / Dashboard / DataPipeline / ..."),
    ("node_name", "STRING", "Asset name"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("workspace_name", "STRING", "Workspace name"),
    ("generation", "INT", "Dataflow generation when applicable"),
    ("readiness", "STRING", "Readiness when the node is an in-scope dataflow"),
    ("wave", "INT", "Wave when the node is a dataflow"),
    ("in_degree", "INT", "Upstream edges"),
    ("out_degree", "INT", "Downstream edges"),
], audit="run", mode="snapshot", doc="Node table derived from Assessment_Lineage. With the edge table it can be loaded into a Graph in Microsoft Fabric model or a network visual.")

_table("Assessment_DataflowDependencies", "gold", "One row per dataflow and related asset (direct and transitive)", ["dataflow_id", "direction", "related_asset_type", "related_asset_id", "hop"], [
    ("dataflow_id", "STRING", "Anchor dataflow GUID"),
    ("workspace_id", "STRING", "Anchor workspace GUID"),
    ("direction", "STRING", "UPSTREAM or DOWNSTREAM"),
    ("hop", "INT", "1 = direct"),
    ("related_asset_type", "STRING", "Dataflow / SemanticModel / Report / Dashboard / DataPipeline / Datasource / Other"),
    ("related_asset_id", "STRING", "Related asset id"),
    ("related_asset_name", "STRING", "Related asset name"),
    ("related_workspace_id", "STRING", "Related workspace id"),
    ("related_workspace_name", "STRING", "Related workspace name"),
    ("relationship", "STRING", "Relationship of the last edge on the path"),
    ("crosses_workspace", "BOOLEAN", "Related asset is in another workspace"),
    ("related_generation", "INT", "Generation when the related asset is a dataflow"),
    ("path", "STRING", "Readable path from the anchor"),
    ("is_cyclic", "BOOLEAN", "Part of a dependency cycle"),
], audit="run", mode="snapshot", doc="Transitive dependency closure (bounded depth) derived from Assessment_Lineage.")

_table("Assessment_MigrationReadiness", "gold", "One row per dataflow", ["dataflow_id"], [
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("dataflow_type", "STRING", "Gen1 / Gen2 / Unknown"),
    ("migration_scope", "STRING", "In scope (Gen1 -> Gen2) / Out of scope (already Gen2) / Unknown generation"),
    ("migration_status", "STRING", "NotStarted / Migrated (type flipped in place) / MigratedBySaveAs / SaveAsFailed / SaveAsCopy (Gen2 copy made by Save-As); NULL for native Gen2"),
    ("gen2_dataflow_id", "STRING", "Gen2 copy id when migrated with Save-As"),
    ("readiness", "STRING", "Ready / Needs Attention / Unsupported / Deferred (NULL when out of scope)"),
    ("readiness_reason", "STRING", "Decisive reason"),
    ("status_wizard", "STRING", "Evaluation of the Upgrade Wizard path: Ready / Needs Attention / Unsupported"),
    ("status_saveas", "STRING", "Evaluation of the Save-As path: Ready / Needs Attention / Unsupported"),
    ("predicted_wizard_status", "STRING", "Proxy for the wizard assessment using its names: Ready to migrate / Needs Attention / Upgrade unavailable"),
    ("recommended_path", "STRING", "UpgradeWizard / SaveAs / ManualRebuild / Retire / FixThenReassess / None"),
    ("recommendation", "STRING", "Plain-language recommendation"),
    ("top_actions", "ARRAY<STRING>", "Ordered remediation actions"),
    ("blocking_rules", "ARRAY<STRING>", "Rule ids that make the dataflow Unsupported"),
    ("attention_rules", "ARRAY<STRING>", "Rule ids that require attention"),
    ("complexity_score", "DOUBLE", "0-100 migration complexity score"),
    ("complexity_band", "STRING", "Low / Medium / High / Very High"),
    ("score_dependencies", "DOUBLE", "Component score 0-100"),
    ("score_refresh_criticality", "DOUBLE", "Component score 0-100"),
    ("score_gateway", "DOUBLE", "Component score 0-100"),
    ("score_source_complexity", "DOUBLE", "Component score 0-100"),
    ("score_unsupported", "DOUBLE", "Component score 0-100"),
    ("score_technical", "DOUBLE", "Component score 0-100"),
    ("effort_days", "DOUBLE", "Estimated effort (person-days)"),
    ("est_gen2_cu_seconds_per_run", "DOUBLE", "Indicative Gen2 standard compute per refresh (12 CU/s for the first 10 min, then 1.5 CU/s) using Gen1 durations"),
    ("est_gen2_cu_hours_per_month", "DOUBLE", "Indicative monthly Gen2 compute from schedule and average duration"),
    ("wave", "INT", "Wave number (0 pilot ... 4 remediate); 99 = backlog"),
    ("wave_label", "STRING", "Readable wave name"),
    ("wave_rationale", "STRING", "Why this wave"),
    ("batch_id", "STRING", "Wave x workspace batch: W<wave>-<workspace id> (the Upgrade Wizard works per workspace)"),
    ("migration_group_id", "STRING", "Connected component of linked dataflows to migrate together"),
    ("migration_group_size", "INT", "Dataflows in the group"),
    ("dependency_level", "INT", "0 = no upstream dataflow; higher = deeper downstream"),
    ("upstream_dataflow_count", "INT", "Direct upstream dataflows"),
    ("downstream_dataflow_count", "INT", "Direct downstream dataflows"),
    ("downstream_dataset_count", "INT", "Downstream semantic models (transitive)"),
    ("downstream_report_count", "INT", "Downstream reports (transitive)"),
    ("downstream_dashboard_count", "INT", "Downstream dashboards (transitive)"),
    ("downstream_other_count", "INT", "Other downstream assets (pipelines, ...)"),
    ("rebind_item_count", "INT", "Items to re-point if the dataflow id changes (Save-As path)"),
    ("business_criticality", "STRING", "High / Medium / Low"),
    ("criticality_source", "STRING", "Override / Inferred"),
    ("environment", "STRING", "Dev / Test / Prod / Unknown"),
    ("assessment_confidence", "STRING", "High / Medium / Low (data completeness)"),
    ("rules_version", "STRING", "Rule catalog version used"),
    ("assessed_at", "TIMESTAMP", "Assessment time"),
], audit="run", mode="snapshot", zorder_by=("dataflow_id",), doc="Readiness class, complexity score, effort, recommendation and wave per dataflow.")

_table("Assessment_DataflowSummary", "gold", "One row per dataflow, denormalized for reporting", ["dataflow_id"], [
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("workspace_name", "STRING", "Workspace name"),
    ("capacity_id", "STRING", "Capacity GUID"),
    ("capacity_name", "STRING", "Capacity name"),
    ("capacity_sku", "STRING", "Capacity SKU"),
    ("domain_name", "STRING", "Domain"),
    ("environment", "STRING", "Environment"),
    ("workspace_classification", "STRING", "Workspace classification"),
    ("support_team", "STRING", "Support team"),
    ("business_owner", "STRING", "Business owner (governance overrides)"),
    ("dataflow_name", "STRING", "Dataflow name"),
    ("dataflow_type", "STRING", "Gen1 / Gen2 / Unknown"),
    ("owner_upn", "STRING", "Technical owner"),
    ("created_at", "TIMESTAMP", "Created"),
    ("modified_at", "TIMESTAMP", "Modified"),
    ("description", "STRING", "Description"),
    ("tags", "ARRAY<STRING>", "Tags"),
    ("last_refresh_at", "TIMESTAMP", "Last refresh"),
    ("last_refresh_status", "STRING", "Last refresh status"),
    ("last_refresh_duration_sec", "DOUBLE", "Last refresh duration"),
    ("refresh_success_rate", "DOUBLE", "Success rate in the observed window"),
    ("refresh_ok", "BOOLEAN", "Latest refresh succeeded and no consecutive failures"),
    ("schedule_enabled", "BOOLEAN", "Schedule enabled"),
    ("schedule_runs_per_day", "INT", "Runs per day"),
    ("source_count", "INT", "Sources"),
    ("uses_gateway", "BOOLEAN", "Gateway used"),
    ("query_count", "INT", "Queries"),
    ("entity_count", "INT", "Entities"),
    ("custom_function_count", "INT", "Custom functions"),
    ("parameter_count", "INT", "Parameters"),
    ("incremental_refresh_count", "INT", "Entities with incremental refresh"),
    ("computed_entity_count", "INT", "Computed entities"),
    ("linked_entity_count", "INT", "Linked entities"),
    ("upstream_dataflow_count", "INT", "Upstream dataflows"),
    ("downstream_dataflow_count", "INT", "Downstream dataflows"),
    ("downstream_dataset_count", "INT", "Downstream semantic models"),
    ("downstream_report_count", "INT", "Downstream reports"),
    ("downstream_dashboard_count", "INT", "Downstream dashboards"),
    ("unsupported_feature_count", "INT", "Findings that need action on at least one path (informational and Deferred findings excluded)"),
    ("business_criticality", "STRING", "Business criticality"),
    ("migration_scope", "STRING", "Migration scope"),
    ("readiness", "STRING", "Ready / Needs Attention / Unsupported / Deferred"),
    ("predicted_wizard_status", "STRING", "Predicted Upgrade Wizard status (Ready to migrate / Needs Attention / Upgrade unavailable)"),
    ("wizard_status_actual", "STRING", "Actual Upgrade Wizard status recorded in governance overrides"),
    ("complexity_score", "DOUBLE", "Complexity score"),
    ("complexity_band", "STRING", "Complexity band"),
    ("effort_days", "DOUBLE", "Effort estimate"),
    ("recommended_path", "STRING", "Recommended migration path"),
    ("wave", "INT", "Wave"),
    ("wave_label", "STRING", "Wave label"),
    ("batch_id", "STRING", "Wave x workspace batch"),
    ("migration_status", "STRING", "Migration status"),
    ("assessment_confidence", "STRING", "Confidence"),
], audit="run", mode="snapshot", zorder_by=("workspace_id", "dataflow_id"), doc="Excel-style assessment master (mirrors the consolidated checklist columns). Power BI / Direct Lake friendly.")

_table("Assessment_WavePlan", "gold", "One row per wave and workspace batch", ["wave", "workspace_id"], [
    ("wave", "INT", "Wave number"),
    ("wave_label", "STRING", "Wave label"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("workspace_name", "STRING", "Workspace name"),
    ("capacity_name", "STRING", "Capacity name"),
    ("environment", "STRING", "Dev / Test / Prod, Mixed when the dataflows of the batch disagree, Unknown"),
    ("dataflow_count", "INT", "Dataflows in the batch"),
    ("ready_count", "INT", "Ready"),
    ("needs_attention_count", "INT", "Needs Attention"),
    ("unsupported_count", "INT", "Unsupported"),
    ("deferred_count", "INT", "Deferred"),
    ("total_effort_days", "DOUBLE", "Sum of effort"),
    ("max_complexity_score", "DOUBLE", "Highest complexity in the batch"),
    ("downstream_item_count", "INT", "Downstream semantic models, reports and dashboards"),
    ("path_mix", "STRING", "Recommended paths and counts"),
], audit="run", mode="snapshot", doc="Migration waves aggregated per workspace (the Upgrade Wizard operates per workspace).")

_table("Assessment_MigrationBacklog", "gold", "One row per in-scope dataflow, in execution order", ["dataflow_id"], [
    ("seq", "INT", "Execution order: wave, dependency level, workspace, name"),
    ("wave", "INT", "Wave number"),
    ("wave_label", "STRING", "Wave label"),
    ("batch_id", "STRING", "Wave x workspace batch"),
    ("workspace_id", "STRING", "Workspace GUID"),
    ("workspace_name", "STRING", "Workspace name"),
    ("dataflow_id", "STRING", "Dataflow GUID"),
    ("dataflow_name", "STRING", "Dataflow name"),
    ("owner_upn", "STRING", "Owner (the Upgrade Wizard must be run by the owner)"),
    ("readiness", "STRING", "Ready / Needs Attention / Unsupported / Deferred"),
    ("recommended_path", "STRING", "UpgradeWizard / SaveAs / ManualRebuild / Retire / FixThenReassess"),
    ("can_start_now", "BOOLEAN", "Ready, not yet migrated, and every upstream dataflow is already migrated"),
    ("blocked_by", "ARRAY<STRING>", "Upstream dataflow ids that must be migrated first"),
    ("pre_migration_actions", "ARRAY<STRING>", "Actions to complete before the migration"),
    ("post_migration_actions", "ARRAY<STRING>", "Actions to complete after the migration"),
    ("effort_days", "DOUBLE", "Estimated effort"),
    ("complexity_band", "STRING", "Complexity band"),
    ("business_criticality", "STRING", "Business criticality"),
    ("environment", "STRING", "Environment"),
    ("migration_status", "STRING", "NotStarted / SaveAsFailed"),
    ("gen2_dataflow_id", "STRING", "Gen2 copy id when a Save-As copy exists"),
], audit="run", mode="snapshot", zorder_by=("dataflow_id",), doc="Ordered work queue for the migration team and the Save-As accelerator: filter on can_start_now, wave or batch_id.")

_table("Assessment_RebindWorklist", "gold", "One row per dataflow and downstream item to re-point", ["dataflow_id", "consumer_id"], [
    ("dataflow_id", "STRING", "Dataflow GUID being migrated"),
    ("dataflow_name", "STRING", "Dataflow name"),
    ("workspace_id", "STRING", "Dataflow workspace GUID"),
    ("workspace_name", "STRING", "Dataflow workspace name"),
    ("wave", "INT", "Wave of the dataflow"),
    ("path", "STRING", "Recommended path that creates the work: UpgradeWizard or SaveAs"),
    ("consumer_type", "STRING", "Dataflow / SemanticModel / Datamart / DataPipeline"),
    ("consumer_id", "STRING", "Consumer GUID"),
    ("consumer_name", "STRING", "Consumer name"),
    ("consumer_workspace_id", "STRING", "Consumer workspace GUID"),
    ("consumer_workspace_name", "STRING", "Consumer workspace name"),
    ("uses_legacy_connector", "BOOLEAN", "Consumer M text uses the legacy PowerBI.Dataflows connector"),
    ("storage_mode", "STRING", "Semantic model storage mode when known"),
    ("action", "STRING", "What to do"),
    ("effort_days", "DOUBLE", "Estimated effort"),
    ("gen2_dataflow_id", "STRING", "Id of the Gen2 copy when already created with Save-As"),
], audit="run", mode="snapshot", doc="Consumers to re-point. The Upgrade Wizard keeps the id (only linked dataflows are re-saved); Save-As creates a new id (every direct consumer).")

_table("Assessment_RunSummary", "gold", "One row per run", ["run_id"], [
    ("run_id", "STRING", "Run id"),
    ("started_at", "TIMESTAMP", "Start"),
    ("ended_at", "TIMESTAMP", "End"),
    ("status", "STRING", "Succeeded / Partial / Failed"),
    ("mode", "STRING", "Run mode"),
    ("workspaces_total", "INT", "Workspaces known"),
    ("workspaces_scanned", "INT", "Workspaces scanned in this run"),
    ("dataflows_total", "INT", "Dataflows known"),
    ("gen1_count", "INT", "Gen1"),
    ("gen2_count", "INT", "Gen2"),
    ("ready_count", "INT", "Ready"),
    ("needs_attention_count", "INT", "Needs Attention"),
    ("unsupported_count", "INT", "Unsupported"),
    ("deferred_count", "INT", "Deferred"),
    ("migrated_count", "INT", "Migrated Gen1 -> Gen2 (type flips or Save-As results)"),
    ("avg_complexity_score", "DOUBLE", "Average complexity score of in-scope dataflows"),
    ("total_effort_days", "DOUBLE", "Total estimated effort"),
    ("api_calls", "INT", "API calls"),
    ("api_throttled", "INT", "HTTP 429 responses"),
    ("error_count", "INT", "Errors logged"),
    ("notebook_version", "STRING", "Notebook version"),
], audit="none", doc="KPI snapshot per run (burn-down and trend reporting). One row per run id: a resumed run updates its row.")

_table("Assessment_RunLog", "control", "One row per run phase", ["run_id", "seq"], [
    ("run_id", "STRING", "Run id"),
    ("seq", "INT", "Phase sequence number"),
    ("phase", "STRING", "Phase name"),
    ("status", "STRING", "Succeeded / Failed / Skipped / Partial"),
    ("started_at", "TIMESTAMP", "Start"),
    ("ended_at", "TIMESTAMP", "End"),
    ("duration_sec", "DOUBLE", "Duration"),
    ("rows_in", "INT", "Rows or entities processed"),
    ("rows_out", "INT", "Rows written"),
    ("api_calls", "INT", "API calls in the phase"),
    ("api_retries", "INT", "Retries in the phase"),
    ("api_throttled", "INT", "HTTP 429 responses in the phase"),
    ("errors", "INT", "Errors in the phase"),
    ("message", "STRING", "Summary or failure message"),
    ("mode", "STRING", "Run mode"),
    ("params_json", "STRING", "Effective parameters (no secrets)"),
], audit="none", mode="append", doc="Phase level execution log.")

_table("Assessment_ErrorLog", "control", "One row per error", ["run_id", "ts", "entity_id", "endpoint"], [
    ("run_id", "STRING", "Run id"),
    ("ts", "TIMESTAMP", "Error time"),
    ("phase", "STRING", "Phase"),
    ("entity_type", "STRING", "Workspace / Dataflow / Batch / ..."),
    ("entity_id", "STRING", "Entity id"),
    ("endpoint", "STRING", "Endpoint template"),
    ("http_status", "INT", "HTTP status"),
    ("error_code", "STRING", "API error code"),
    ("message", "STRING", "Message"),
    ("attempt", "INT", "Attempt number"),
    ("retryable", "BOOLEAN", "Retryable error"),
    ("body_snippet", "STRING", "Truncated response body"),
], audit="none", mode="append", doc="Item level errors with endpoint, status and truncated body.")

_table("Assessment_Checkpoint", "control", "One row per entity and phase", ["entity_type", "entity_id", "phase"], [
    ("entity_type", "STRING", "Dataflow / Workspace / ScanBatch"),
    ("entity_id", "STRING", "Entity id"),
    ("phase", "STRING", "definition / refresh / datasources / upstream / pipelines / scan"),
    ("status", "STRING", "Done / Failed / Denied / NotFound"),
    ("attempts", "INT", "Attempts so far"),
    ("last_attempt_at", "TIMESTAMP", "Last attempt"),
    ("completed_at", "TIMESTAMP", "Last successful completion"),
    ("source_version", "STRING", "Entity version when collected (modification time or hash)"),
    ("last_error", "STRING", "Last error"),
], audit="run", doc="Restart and incremental state: what was collected, for which entity version, and what failed.")

_table("Assessment_RuleCatalog", "config", "One row per assessment rule", ["rule_id"], [
    ("rule_id", "STRING", "Rule id"),
    ("name", "STRING", "Rule name"),
    ("category", "STRING", "Category"),
    ("severity", "STRING", "Blocker / Major / Minor / Info"),
    ("impact_wizard", "STRING", "Effect on the Upgrade Wizard path: Unsupported / NeedsAttention / Deferred / None"),
    ("impact_saveas", "STRING", "Effect on the Save-As path: Unsupported / NeedsAttention / Deferred / None"),
    ("weight", "INT", "Weight in the unsupported-features score"),
    ("effort_days", "DOUBLE", "Remediation effort"),
    ("detection", "STRING", "How the rule is detected"),
    ("remediation", "STRING", "Recommended action"),
    ("confidence", "STRING", "Verified (documented) / Heuristic"),
    ("enabled", "BOOLEAN", "Rule enabled"),
    ("version", "STRING", "Catalog version"),
], audit="none", mode="snapshot", doc="Snapshot of the rule catalog used by the run (auditable, tunable).")

_table("Assessment_GovernanceOverrides", "config", "One row per workspace or dataflow override", ["scope", "workspace_id", "dataflow_id"], [
    ("scope", "STRING", "Workspace or Dataflow"),
    ("workspace_id", "STRING", "Workspace GUID (required)"),
    ("dataflow_id", "STRING", "Dataflow GUID (empty for workspace scope)"),
    ("business_owner", "STRING", "Business owner"),
    ("technical_owner", "STRING", "Technical owner"),
    ("support_team", "STRING", "Support team"),
    ("business_criticality", "STRING", "High / Medium / Low"),
    ("workspace_classification", "STRING", "Classification"),
    ("environment", "STRING", "Dev / Test / Prod"),
    ("defer", "BOOLEAN", "Force the Deferred class"),
    ("defer_reason", "STRING", "Why migration is deferred"),
    ("pin_wave", "INT", "Pin the dataflow to a wave"),
    ("wizard_status_actual", "STRING", "Observed Upgrade Wizard status: Ready to migrate / Needs Attention / Upgrade unavailable"),
    ("rollback_plan_defined", "BOOLEAN", "Rollback strategy defined"),
    ("uat_window_start", "DATE", "Acceptance window start"),
    ("uat_window_end", "DATE", "Acceptance window end"),
    ("notes", "STRING", "Free text"),
    ("updated_by", "STRING", "Who maintained the row"),
    ("updated_at", "TIMESTAMP", "When"),
], audit="none", doc="Manual governance input merged from the CSV in Files/dea/config (or edited directly).")

In [ ]:
def table_names():
    return list(TABLES)


def spark_type_name(ctype):
    return {"STRING": "StringType", "BOOLEAN": "BooleanType", "INT": "IntegerType", "BIGINT": "LongType", "DOUBLE": "DoubleType",
            "TIMESTAMP": "TimestampType", "DATE": "DateType", "ARRAY<STRING>": "ArrayType"}[ctype]

## 1.2 Constants, configuration object and error model

In [ ]:
import collections
import concurrent.futures as cf
import contextlib
import csv
import datetime as dt
import gzip
import io
import itertools
import json
import logging
import math
import os
import random
import re
import statistics
import sys
import threading
import time
import traceback
import uuid
from collections import Counter, defaultdict, deque

import requests

NOTEBOOK_VERSION = "1.0.0"

PBI_SCOPE = "https://analysis.windows.net/powerbi/api/.default"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
GRAPH_SCOPE = "https://graph.microsoft.com/.default"

RUN_MODES = ("full", "incremental", "resume", "analysis_only", "refresh_only")
AUTH_MODES = ("auto", "spn", "user")
PREFERRED_PATHS = ("auto", "wizard", "saveas")
DEFINITION_SOURCES = ("admin", "workspace")
MIN_VACUUM_HOURS = 168      # Delta refuses a shorter VACUUM retention unless the safety check is disabled
LOG_LEVELS = ("DEBUG", "INFO", "WARNING", "ERROR")

_PARAM_KINDS = {
    "str": ["run_mode", "resume_run_id", "log_level", "lakehouse_schema", "files_root", "auth_mode", "key_vault_url",
            "kv_secret_tenant_id", "kv_secret_client_id", "kv_secret_client_secret", "authority_host", "pbi_api_base",
            "fabric_api_base", "graph_api_base", "source_capacity", "preferred_path", "governance_overrides_path",
            "migration_results_table", "definition_source"],
    "bool": ["dry_run", "fail_on_partial", "persist_raw", "run_maintenance", "include_personal_workspaces",
             "include_inactive_workspaces", "collect_gen1", "collect_gen2", "collect_definitions", "collect_gen2_definitions",
             "collect_compute_engine", "collect_refresh_history", "collect_admin_datasources", "verify_upstream_admin_api",
             "collect_pipeline_references", "collect_item_connections", "enrich_owners_graph", "scan_get_users",
             "scan_dataset_expressions", "wizard_available", "require_definition_for_ready", "export_reports", "render_lineage_graph"],
    "int": ["max_runtime_minutes", "raw_retention_days", "vacuum_retention_hours", "scan_batch_size", "scan_max_concurrent",
            "scan_poll_interval_sec", "scan_timeout_sec", "scan_lookback_days", "max_workers", "max_retries",
            "max_retry_after_sec", "budget_definitions_per_run", "budget_refresh_dataflows_per_run", "budget_datasources_per_run",
            "refresh_history_ttl_hours", "refresh_history_max_records", "denied_retry_days", "failed_retry_days",
            "max_attempts_per_entity", "batch_flush_size", "stale_days", "failing_consecutive", "pilot_size", "top_n",
            "lineage_graph_max_nodes"],
    "float": ["rate_limit_scale", "retry_base_delay_sec", "retry_max_delay_sec"],
    "list": ["only_workspaces", "exclude_workspaces"],
    "dict": ["rate_limit_overrides", "score_weights", "environment_patterns"],
}


class DeaError(Exception):
    """Base class for all notebook errors."""


class ConfigError(DeaError):
    pass


class GracefulStop(DeaError):
    """Stops a phase early without failing the run (deadline, budget, throttling): resume on the next run."""


class BudgetExceeded(GracefulStop):
    pass


class DeadlineExceeded(GracefulStop):
    pass


class AccessBreaker(GracefulStop):
    """Raised when every early call is denied: stops burning API quota on a permission problem."""


class ApiError(DeaError):
    def __init__(self, message, status=None, code=None, url=None, body=None, retryable=False, endpoint=None):
        super().__init__(message)
        self.status = status
        self.code = code
        self.url = url
        self.body = body
        self.retryable = retryable
        self.endpoint = endpoint


class AuthError(ApiError):
    pass


class ForbiddenError(ApiError):
    pass


class NotFoundError(ApiError):
    pass


class ThrottleExhausted(ApiError, GracefulStop):
    pass


def _as_bool_param(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "1", "yes", "y"):
        return True
    if text in ("false", "0", "no", "n", ""):
        return False
    raise ConfigError(f"not a boolean: {value!r}")


def as_list(value):
    """Accept lists, JSON arrays or comma / semicolon / newline separated text."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(v).strip() for v in value if str(v).strip()]
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("["):
        try:
            return [str(v).strip() for v in json.loads(text) if str(v).strip()]
        except ValueError:
            pass
    return [p.strip() for p in re.split(r"[,;\n]", text) if p.strip()]


def as_dict(value):
    if value is None or value == "":
        return {}
    if isinstance(value, dict):
        return dict(value)
    try:
        parsed = json.loads(str(value))
    except ValueError as exc:
        raise ConfigError(f"not valid JSON: {value!r}") from exc
    if not isinstance(parsed, dict):
        raise ConfigError(f"expected a JSON object: {value!r}")
    return parsed


def safe_files_root(value):
    """p_files_root must be a folder below the Lakehouse Files area (for example Files/dea): no drive, no absolute path, no '..'."""
    parts = [p for p in str(value or "").replace("\\", "/").strip().split("/") if p not in ("", ".")]
    if len(parts) < 2 or parts[0].lower() != "files" or any(p == ".." or ":" in p for p in parts):
        raise ConfigError(f"files_root must be a folder below 'Files', for example 'Files/dea' (got {value!r})")
    return "/".join(parts)


class Config:
    """Typed, validated view over the p_* notebook parameters."""

    def __init__(self, **values):
        self.__dict__.update(values)

    @classmethod
    def from_namespace(cls, ns):
        values, missing = {}, []
        for kind, names in _PARAM_KINDS.items():
            for name in names:
                key = f"p_{name}"
                if key not in ns:
                    missing.append(key)
                    continue
                raw = ns[key]
                try:
                    if kind == "str":
                        values[name] = "" if raw is None else str(raw).strip()
                    elif kind == "bool":
                        values[name] = _as_bool_param(raw)
                    elif kind == "int":
                        values[name] = int(float(raw))
                    elif kind == "float":
                        values[name] = float(raw)
                    elif kind == "list":
                        values[name] = as_list(raw)
                    else:
                        values[name] = as_dict(raw)
                except (TypeError, ValueError) as exc:
                    raise ConfigError(f"invalid value for {key}: {raw!r}") from exc
        if missing:
            raise ConfigError("missing parameters: " + ", ".join(sorted(missing)))
        cfg = cls(**values)
        cfg.validate()
        return cfg

    def validate(self):
        problems = []

        def need(condition, message):
            if not condition:
                problems.append(message)

        need(self.run_mode in RUN_MODES, f"run_mode must be one of {RUN_MODES}")
        need(self.auth_mode in AUTH_MODES, f"auth_mode must be one of {AUTH_MODES}")
        need(self.preferred_path in PREFERRED_PATHS, f"preferred_path must be one of {PREFERRED_PATHS}")
        need(self.definition_source in DEFINITION_SOURCES, f"definition_source must be one of {DEFINITION_SOURCES}")
        need(self.log_level.upper() in LOG_LEVELS, f"log_level must be one of {LOG_LEVELS}")
        need(1 <= self.scan_batch_size <= 100, "scan_batch_size must be 1..100")
        need(1 <= self.scan_max_concurrent <= 16, "scan_max_concurrent must be 1..16")
        need(1 <= self.max_workers <= 32, "max_workers must be 1..32")
        need(0 < self.rate_limit_scale <= 1, "rate_limit_scale must be in (0, 1]")
        need(self.max_retries >= 0, "max_retries must be >= 0")
        need(self.max_runtime_minutes >= 0, "max_runtime_minutes must be >= 0")
        need(1 <= self.scan_lookback_days <= 30, "scan_lookback_days must be 1..30 (API limit)")
        need(self.run_mode != "resume" or bool(self.resume_run_id), "resume_run_id is required for run_mode=resume")
        need(self.auth_mode != "spn" or bool(self.key_vault_url), "key_vault_url is required for auth_mode=spn")
        need(self.vacuum_retention_hours == 0 or self.vacuum_retention_hours >= MIN_VACUUM_HOURS,
             f"vacuum_retention_hours must be 0 (no VACUUM) or at least {MIN_VACUUM_HOURS}")
        try:
            safe_files_root(self.files_root)
        except ConfigError as exc:
            problems.append(str(exc))
        if problems:
            raise ConfigError("; ".join(problems))

    def public_dict(self):
        """Effective parameters for logging. Only Key Vault secret names are configured, never secrets."""
        return {k: v for k, v in sorted(self.__dict__.items())}

## 1.3 Logging, run context, statistics and phase runner
* `RunContext` carries configuration, clients, storage handles, statistics and the deadline - nothing is global.
* `run_phase` gives every phase: structured logs, timing, API statistics, `Assessment_RunLog` rows and consistent
  failure semantics. `GracefulStop` (deadline, API budget, exhausted throttling) yields a **Partial** phase, not a failure.

In [ ]:
class SystemClock:
    def monotonic(self):
        return time.monotonic()

    def sleep(self, seconds):
        time.sleep(max(0.0, seconds))

    def now(self):
        return utcnow()


class LogState:
    run_id = "-"
    phase = "-"


class _ContextFilter(logging.Filter):
    def filter(self, record):
        record.run_id = LogState.run_id
        record.phase = LogState.phase
        return True


class LogCapture(logging.Handler):
    """Keeps the most recent WARNING+ records for the end-of-run report."""

    def __init__(self, capacity=500):
        super().__init__(level=logging.WARNING)
        self.records = deque(maxlen=capacity)

    def emit(self, record):
        self.records.append((record.levelname, record.phase, record.getMessage()))


def setup_logging(level="INFO"):
    logger = logging.getLogger("dea")
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    logger.propagate = False
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
    stream = logging.StreamHandler(sys.stdout)
    stream.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(run_id)s | %(phase)-14s | %(message)s", "%H:%M:%S"))
    stream.addFilter(_ContextFilter())
    capture = LogCapture()
    capture.addFilter(_ContextFilter())
    logger.addHandler(stream)
    logger.addHandler(capture)
    return logger, capture


class ApiStats:
    def __init__(self):
        self._lock = threading.Lock()
        self.calls = Counter()
        self.retries = Counter()
        self.throttled = Counter()
        self.failures = Counter()
        self.latency = defaultdict(float)

    def record(self, bucket, status, elapsed, retried=False):
        with self._lock:
            self.calls[bucket] += 1
            self.latency[bucket] += elapsed
            if retried:
                self.retries[bucket] += 1
            if status == 429:
                self.throttled[bucket] += 1
            if status is None or status >= 400:
                self.failures[bucket] += 1

    def totals(self):
        with self._lock:
            return {"calls": sum(self.calls.values()), "retries": sum(self.retries.values()),
                    "throttled": sum(self.throttled.values()), "failures": sum(self.failures.values())}

    def by_bucket(self):
        with self._lock:
            return {b: {"calls": c, "retries": self.retries[b], "throttled": self.throttled[b], "failures": self.failures[b],
                        "avg_ms": round(1000 * self.latency[b] / c, 1) if c else 0.0} for b, c in sorted(self.calls.items())}


class Deadline:
    def __init__(self, minutes, clock):
        self._clock = clock
        self._end = clock.monotonic() + minutes * 60 if minutes else None

    def remaining(self):
        return float("inf") if self._end is None else self._end - self._clock.monotonic()

    def expired(self, reserve_sec=0):
        return self.remaining() <= reserve_sec

    def check(self, reserve_sec=0):
        if self.expired(reserve_sec):
            raise DeadlineExceeded("run deadline reached")


class RunContext:
    def __init__(self, cfg, run_id=None, clock=None, logger=None):
        self.cfg = cfg
        self.run_id = run_id or f"{utcnow():%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:6]}"
        self.clock = clock or SystemClock()
        self.logger = logger or logging.getLogger("dea")
        self.stats = ApiStats()
        self.deadline = Deadline(cfg.max_runtime_minutes, self.clock)
        self.started_at = utcnow()
        self.lake = None
        self.files = None
        self.checkpoints = None
        self.clients = {}
        self.state = {}
        self.phase_rows = []
        self.error_rows = []
        self.phase_status = {}
        self.blocked_phases = set()     # phases skipped because a phase they depend on did not complete (see Step.needs)
        self._seq = 0
        self._error_lock = threading.Lock()
        LogState.run_id = self.run_id

    def continue_sequence(self, last_seq):
        """Number the phases of a resumed run after the ones already logged under the same run id."""
        self._seq = max(self._seq, int(last_seq or 0))

    def record_error(self, phase, entity_type, entity_id, endpoint, exc=None, message=None, attempt=1):
        status = getattr(exc, "status", None)
        row = {
            "run_id": self.run_id, "ts": utcnow(), "phase": phase or LogState.phase, "entity_type": entity_type,
            "entity_id": entity_id, "endpoint": endpoint or getattr(exc, "endpoint", None), "http_status": status,
            "error_code": getattr(exc, "code", None), "message": (message or str(exc))[:2000], "attempt": attempt,
            "retryable": bool(getattr(exc, "retryable", False)),
            "body_snippet": (str(getattr(exc, "body", "") or "")[:1000]) or None,
        }
        with self._error_lock:
            self.error_rows.append(row)
        return row

    def run_phase(self, name, fn, enabled=True, skip_reason="disabled by parameters", critical=False):
        """Run fn(ctx) as a named phase. Returns the metrics dict returned by fn (or {})."""
        self._seq += 1
        started, t0, before = utcnow(), self.clock.monotonic(), self.stats.totals()
        errors_before = len(self.error_rows)
        LogState.phase = name
        row = {"run_id": self.run_id, "seq": self._seq, "phase": name, "started_at": started, "mode": self.cfg.run_mode,
               "params_json": None, "rows_in": None, "rows_out": None, "message": None}
        metrics = {}
        if not enabled:
            row.update(status="Skipped", ended_at=utcnow(), duration_sec=0.0, message=skip_reason,
                       api_calls=0, api_retries=0, api_throttled=0, errors=0)
            self.logger.info("skipped (%s)", skip_reason)
            self.phase_rows.append(row)
            self.phase_status[name] = "Skipped"
            return metrics
        self.logger.info("start")
        status = "Succeeded"
        try:
            metrics = fn(self) or {}
            status = metrics.get("status", "Succeeded")
            row["message"] = metrics.get("message")
        except GracefulStop as stop:
            status = "Partial"
            row["message"] = f"stopped early: {stop}"
            self.logger.warning("stopped early: %s - the next run resumes from the checkpoints", stop)
        except Exception as exc:  # noqa: BLE001 - a phase failure must never lose the run log
            status = "Failed"
            row["message"] = f"{type(exc).__name__}: {exc}"[:2000]
            self.record_error(name, "Phase", name, None, exc=exc, message=row["message"])
            self.logger.error("failed: %s\n%s", row["message"], traceback.format_exc(limit=6))
            if critical:
                self._close_phase(row, status, t0, before, errors_before, metrics)
                raise
        self._close_phase(row, status, t0, before, errors_before, metrics)
        return metrics

    def _close_phase(self, row, status, t0, before, errors_before, metrics):
        after = self.stats.totals()
        row.update(status=status, ended_at=utcnow(), duration_sec=round(self.clock.monotonic() - t0, 2),
                   rows_in=metrics.get("rows_in"), rows_out=metrics.get("rows_out"),
                   api_calls=after["calls"] - before["calls"], api_retries=after["retries"] - before["retries"],
                   api_throttled=after["throttled"] - before["throttled"], errors=len(self.error_rows) - errors_before)
        self.phase_rows.append(row)
        self.phase_status[row["phase"]] = status
        self.logger.info("%s in %.1fs | api calls=%s retries=%s throttled=%s errors=%s", status.lower(), row["duration_sec"],
                         row["api_calls"], row["api_retries"], row["api_throttled"], row["errors"])
        LogState.phase = "-"

    def overall_status(self):
        statuses = set(self.phase_status.values())
        if "Failed" in statuses:
            return "Failed"
        if "Partial" in statuses:
            return "Partial"
        return "Succeeded"


def chunked(items, size):
    items = list(items)
    for i in range(0, len(items), size):
        yield items[i:i + size]


def gid(value):
    """Normalize ids (GUIDs) for joins: trimmed lowercase, None when empty."""
    if value is None:
        return None
    text = str(value).strip().lower()
    return text or None


def dig(obj, *path, default=None):
    for key in path:
        if isinstance(obj, dict):
            obj = obj.get(key)
        else:
            return default
        if obj is None:
            return default
    return obj


def first(*values):
    for v in values:
        if v not in (None, "", [], {}):
            return v
    return None


def percentile(values, q):
    data = sorted(v for v in values if v is not None)
    if not data:
        return None
    if len(data) == 1:
        return float(data[0])
    pos = (len(data) - 1) * q
    low, high = math.floor(pos), math.ceil(pos)
    return float(data[low] + (data[high] - data[low]) * (pos - low))


def truncate(text, limit):
    text = "" if text is None else str(text)
    return text if len(text) <= limit else text[: limit - 1] + "…"

## 1.4 Storage layer (Delta through Spark, files through the mounted Lakehouse)
The pipeline only talks to the `Lake` contract below. `SparkLake` implements it with Delta `MERGE`;
`InMemoryLake` implements the same semantics for dry runs and offline tests.

* **Partial-column upserts** - each phase owns a set of columns of a table; `merge` only overwrites the columns the
  phase supplies, so the refresh phase can never null out what the Scanner phase wrote.
* **Soft deletes** for current-state tables (`is_deleted`, `deleted_at`) - history is never lost.
* **Schema evolution** - missing registry columns are added to existing tables (`ALTER TABLE ... ADD COLUMNS`).
* **Checkpoints** - `Assessment_Checkpoint` records what was collected for which entity version, driving incremental
  runs, TTLs and retry back-off for denied or failing entities.

In [ ]:
import copy


def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    text = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{text}'"


def resolve_update_cols(table, rows, update_cols=None):
    """Columns a merge may overwrite: explicit list, else the columns present in the supplied rows."""
    reserved = set(table.keys) | {c[0] for c in _AUDIT[table.audit]}
    known = set(table.col_names)
    if update_cols:
        chosen = [c for c in update_cols if c in known]
    else:
        present = set()
        for row in rows:
            present.update(row)
        chosen = [c for c in table.col_names if c in present]
    return [c for c in chosen if c not in reserved]


def merge_sql(table, target, source, update_cols, prune=None):
    on = " AND ".join(f"t.{k} <=> s.{k}" for k in table.keys)
    for column, values in (prune or {}).items():
        on += f" AND t.{column} IN ({', '.join(sql_literal(v) for v in sorted(values))})"
    sets = [f"t.{c} = s.{c}" for c in update_cols]
    if table.audit == "scd":
        sets += ["t.last_seen_at = s.last_seen_at", "t.is_deleted = false", "t.deleted_at = NULL", "t.run_id = s.run_id"]
    elif table.audit == "run":
        sets += ["t.run_id = s.run_id", "t.updated_at = s.updated_at"]
    cols = table.col_names
    action = f"WHEN MATCHED THEN UPDATE SET {', '.join(sets)} " if sets else ""
    return (f"MERGE INTO {target} AS t USING {source} AS s ON {on} {action}"
            f"WHEN NOT MATCHED THEN INSERT ({', '.join(cols)}) VALUES ({', '.join('s.' + c for c in cols)})")


def soft_delete_sql(table, target, source):
    on = " AND ".join(f"t.{k} <=> s.{k}" for k in table.keys)
    return (f"MERGE INTO {target} AS t USING {source} AS s ON {on} WHEN MATCHED AND (t.is_deleted = false OR t.is_deleted IS NULL) THEN "
            f"UPDATE SET t.is_deleted = true, t.deleted_at = s.deleted_at, t.run_id = s.run_id")


def dedupe_rows(table, rows):
    """Last row wins per key (MERGE rejects several source rows for one target row)."""
    latest = {}
    for row in rows:
        latest[tuple(row.get(k) for k in table.keys)] = row
    return list(latest.values())


def engine_conflict(exc):
    """The Delta / Fabric engine rejected a statement for an internal reason (an ambiguous internal column), not because of our data."""
    text = str(exc)
    return "AMBIGUOUS_REFERENCE" in text or "filePath" in text


def short_error(exc, limit=300):
    return " ".join(str(exc).split())[:limit]


def _matches(row, where):
    for column, wanted in (where or {}).items():
        value = row.get(column)
        if isinstance(wanted, (list, tuple, set, frozenset)):
            if value not in wanted:
                return False
        elif value != wanted:
            return False
    return True


class LakeBase:
    """Contract used by the pipeline. Rows are dicts keyed by registry column names."""

    def __init__(self, run_id, now=utcnow):
        self.run_id = run_id
        self.now = now

    def _stamp(self, table, row, insert):
        now = self.now()
        if table.audit == "scd":
            if insert:
                row["first_seen_at"] = now
            row.update(last_seen_at=now, is_deleted=False, deleted_at=None, run_id=self.run_id)
        elif table.audit == "run":
            row.update(run_id=self.run_id, updated_at=now)
        return row

    def ensure_tables(self):
        raise NotImplementedError

    def read(self, table, where=None, include_deleted=False):
        raise NotImplementedError

    def merge(self, table, rows, update_cols=None, prune=None):
        raise NotImplementedError

    def mark_deleted(self, table, keys):
        raise NotImplementedError

    def append(self, table, rows):
        raise NotImplementedError

    def replace(self, table, rows):
        raise NotImplementedError

    def count(self, table):
        raise NotImplementedError

    def read_external(self, name):
        """Rows of a table this notebook does not own (e.g. the Save-As accelerator results); None when it does not exist."""
        return None

    def maintain(self, table, vacuum_hours=0):
        """Compaction and cleanup. Returns the actions that really ran, or None when this lake has nothing to maintain."""
        return None


class InMemoryLake(LakeBase):
    def __init__(self, run_id="test-run", now=utcnow):
        super().__init__(run_id, now)
        self.data = {name: ({} if t.mode != "append" else []) for name, t in TABLES.items()}
        self.external = {}

    def read_external(self, name):
        rows = self.external.get(name)
        return None if rows is None else copy.deepcopy(rows)

    def ensure_tables(self):
        return None

    def _rows(self, name):
        store = self.data[name]
        return list(store.values()) if isinstance(store, dict) else list(store)

    def read(self, table, where=None, include_deleted=False):
        t = TABLES[table]
        out = []
        for row in self._rows(table):
            if t.soft_delete and row.get("is_deleted") and not include_deleted:
                continue
            if _matches(row, where):
                out.append(copy.deepcopy(row))
        return out

    def merge(self, table, rows, update_cols=None, prune=None):
        t = TABLES[table]
        rows = [dict(r) for r in rows]
        if not rows:
            return 0
        cols = resolve_update_cols(t, rows, update_cols)
        store = self.data[table]
        for coerced in dedupe_rows(t, [coerce_row(t, r) for r in rows]):
            key = tuple(coerced[k] for k in t.keys)
            existing = store.get(key)
            if existing is None:
                store[key] = self._stamp(t, dict(coerced), insert=True)
            else:
                for c in cols:
                    existing[c] = coerced.get(c)
                self._stamp(t, existing, insert=False)
        return len(rows)

    def mark_deleted(self, table, keys):
        t = TABLES[table]
        store = self.data[table]
        changed = 0
        for key_dict in keys:
            key = tuple(key_dict.get(k) for k in t.keys)
            row = store.get(key)
            if row is not None and not row.get("is_deleted"):
                row.update(is_deleted=True, deleted_at=self.now(), run_id=self.run_id)
                changed += 1
        return changed

    def append(self, table, rows):
        t = TABLES[table]
        rows = list(rows)
        for raw in rows:
            self.data[table].append(self._stamp(t, coerce_row(t, raw), insert=True))
        return len(rows)

    def replace(self, table, rows):
        t = TABLES[table]
        store = {}
        for raw in dedupe_rows(t, [coerce_row(t, r) for r in rows]):
            store[tuple(raw[k] for k in t.keys)] = self._stamp(t, raw, insert=True)
        self.data[table] = store if t.mode != "append" else list(store.values())
        return len(store)

    def count(self, table):
        return len(self._rows(table))


class SparkLake(LakeBase):
    def __init__(self, spark, schema, run_id, logger=None, now=utcnow):
        super().__init__(run_id, now)
        self.spark = spark
        self.schema = schema
        self.logger = logger or logging.getLogger("dea")
        for key, value in (("spark.sql.session.timeZone", "UTC"),
                           ("spark.microsoft.delta.optimizeWrite.enabled", "true"),
                           ("spark.sql.parquet.vorder.default", "true")):
            try:
                spark.conf.set(key, value)
            except Exception:  # noqa: BLE001 - optional Fabric tuning knobs
                self.logger.debug("could not set %s", key)

    def fqn(self, table):
        return qualified(self.schema, table)

    def _struct(self, table):
        from pyspark.sql import types as T
        mapping = {"STRING": T.StringType(), "BOOLEAN": T.BooleanType(), "INT": T.IntegerType(), "BIGINT": T.LongType(),
                   "DOUBLE": T.DoubleType(), "TIMESTAMP": T.TimestampType(), "DATE": T.DateType(),
                   "ARRAY<STRING>": T.ArrayType(T.StringType(), True)}
        return T.StructType([T.StructField(n, mapping[ctype], True) for n, ctype, _ in table.all_cols()])

    def _frame(self, table, rows):
        names = table.col_names
        return self.spark.createDataFrame([tuple(r.get(n) for n in names) for r in rows], schema=self._struct(table))

    def ensure_tables(self):
        if self.schema:
            self.spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.schema}")
        for table in TABLES.values():
            self.spark.sql(ddl_for(table, self.schema))
            existing = {c.lower() for c in self.spark.table(self.fqn(table.name)).columns}
            missing = [(n, t, d) for n, t, d in table.all_cols() if n.lower() not in existing]
            if missing:
                cols = ", ".join(f"{n} {t} COMMENT '{_sql_comment(d)}'" for n, t, d in missing)
                self.spark.sql(f"ALTER TABLE {self.fqn(table.name)} ADD COLUMNS ({cols})")
                self.logger.info("schema evolution: added %s to %s", [m[0] for m in missing], table.name)

    def _read_frame(self, table, where=None, include_deleted=False):
        t = TABLES[table]
        df = self.spark.table(self.fqn(table))
        for column, wanted in (where or {}).items():
            if isinstance(wanted, (list, tuple, set, frozenset)):
                df = df.filter(df[column].isin(list(wanted)))
            else:
                df = df.filter(df[column] == wanted)
        if t.soft_delete and not include_deleted:
            df = df.filter("is_deleted = false OR is_deleted IS NULL")
        return [r.asDict(recursive=True) for r in df.collect()]

    def read(self, table, where=None, include_deleted=False):
        try:
            return self._read_frame(table, where, include_deleted)
        except Exception as exc:  # noqa: BLE001
            if not where or not engine_conflict(exc):
                raise self._annotated(exc, "reading", table)
            self.logger.warning("reading %s with a filter failed (%s): filtering in Python instead", table, short_error(exc))
            return [r for r in self._read_frame(table, None, include_deleted) if _matches(r, where)]

    def _annotated(self, exc, action, table):
        try:
            exc.add_note(f"while {action} {table}")
        except Exception:  # noqa: BLE001 - notes need Python 3.11
            pass
        self.logger.error("%s %s failed: %s: %s", action, table, type(exc).__name__, short_error(exc))
        return exc

    def merge(self, table, rows, update_cols=None, prune=None):
        t = TABLES[table]
        rows = [dict(r) for r in rows]
        if not rows:
            return 0
        cols = resolve_update_cols(t, rows, update_cols)
        staged = [self._stamp(t, r, insert=True) for r in dedupe_rows(t, [coerce_row(t, r) for r in rows])]
        view = f"_dea_stage_{t.name}"
        self._frame(t, staged).createOrReplaceTempView(view)
        statement = merge_sql(t, self.fqn(table), view, cols, prune)
        try:
            self.spark.sql(statement)
            return len(staged)
        except Exception as exc:  # noqa: BLE001
            if not engine_conflict(exc):
                raise self._annotated(exc, "merging into", table)
            self.logger.warning("MERGE into %s hit an engine conflict (%s)", table, short_error(exc))
        try:   # first retry: Fabric's low-shuffle merge is the usual suspect
            self.spark.conf.set("spark.microsoft.delta.merge.lowShuffle.enabled", "false")
            self.spark.sql(statement)
            self.logger.warning("MERGE into %s succeeded with the low-shuffle merge disabled for this session", table)
            return len(staged)
        except Exception as exc:  # noqa: BLE001
            if not engine_conflict(exc):
                raise self._annotated(exc, "merging into", table)
        self.logger.warning("MERGE into %s still fails: replacing the rows by key instead (delete, then insert)", table)
        self._replace_by_key(t, staged, cols)
        return len(staged)

    def _replace_by_key(self, t, staged, cols, chunk=200):
        """Same result as MERGE without the MERGE engine: the matching rows are read, updated in Python, deleted by key and inserted again.
        Not atomic, but idempotent: a crash in between is repaired by the next run, which writes the same rows again."""
        target, keys = self.fqn(t.name), list(t.keys)
        wanted = {tuple(r.get(k) for k in keys): r for r in staged}
        first, existing = keys[0], {}
        values = sorted({r.get(first) for r in staged if r.get(first) is not None})
        for i in range(0, len(values), 500):
            frame = self.spark.table(target)
            for row in frame.filter(frame[first].isin(values[i:i + 500])).collect():
                data = row.asDict(recursive=True)
                key = tuple(data.get(k) for k in keys)
                if key in wanted:
                    existing[key] = data
        final = []
        for key, new in wanted.items():
            old = existing.get(key)
            if old is None:
                final.append(new)
                continue
            merged = {c: old.get(c) for c in t.col_names}
            merged.update({c: new.get(c) for c in cols})
            for audit in ("last_seen_at", "is_deleted", "deleted_at", "run_id", "updated_at"):
                if audit in new and (t.audit == "scd" or audit in ("run_id", "updated_at")):
                    merged[audit] = new[audit]
            final.append(merged)
        for i in range(0, len(final), chunk):
            conditions = []
            for row in final[i:i + chunk]:
                parts = [f"{k} IS NULL" if row.get(k) is None else f"{k} = {sql_literal(row[k])}" for k in keys]
                conditions.append("(" + " AND ".join(parts) + ")")
            self.spark.sql(f"DELETE FROM {target} WHERE " + " OR ".join(conditions))
        view = f"_dea_fb_{t.name}"
        self._frame(t, final).createOrReplaceTempView(view)
        names = ", ".join(t.col_names)
        self.spark.sql(f"INSERT INTO {target} ({names}) SELECT {names} FROM {view}")

    def mark_deleted(self, table, keys):
        t = TABLES[table]
        keys = list(keys)
        if not keys:
            return 0
        from pyspark.sql import types as T
        now = self.now()
        names = list(t.keys) + ["is_deleted", "deleted_at", "run_id"]
        base = {f.name: f for f in self._struct(t).fields}
        rows = [tuple([kd.get(k) for k in t.keys] + [True, now, self.run_id]) for kd in keys]
        view = f"_dea_del_{t.name}"
        self.spark.createDataFrame(rows, schema=T.StructType([base[n] for n in names])).createOrReplaceTempView(view)
        self.spark.sql(soft_delete_sql(t, self.fqn(table), view))
        return len(rows)

    def append(self, table, rows):
        t = TABLES[table]
        rows = [self._stamp(t, coerce_row(t, r), insert=True) for r in rows]
        if not rows:
            return 0
        view = f"_dea_app_{t.name}"
        self._frame(t, rows).createOrReplaceTempView(view)
        self.spark.sql(f"INSERT INTO {self.fqn(table)} ({', '.join(t.col_names)}) SELECT {', '.join(t.col_names)} FROM {view}")
        return len(rows)

    def replace(self, table, rows):
        t = TABLES[table]
        rows = [self._stamp(t, r, insert=True) for r in dedupe_rows(t, [coerce_row(t, r) for r in rows])]
        if not rows:
            self.spark.sql(f"DELETE FROM {self.fqn(table)}")
            return 0
        view = f"_dea_rep_{t.name}"
        self._frame(t, rows).createOrReplaceTempView(view)
        # INSERT OVERWRITE maps by position: follow the physical column order (it differs from the registry after schema evolution)
        physical = self.spark.table(self.fqn(table)).columns
        known = set(t.col_names)
        select = ", ".join(c if c in known else f"NULL AS {c}" for c in physical)
        self.spark.sql(f"INSERT OVERWRITE TABLE {self.fqn(table)} SELECT {select} FROM {view}")
        return len(rows)

    def count(self, table):
        return self.spark.table(self.fqn(table)).count()

    def read_external(self, name):
        target = self.fqn(name)
        try:
            if not self.spark.catalog.tableExists(target):
                return None
            return [r.asDict(recursive=True) for r in self.spark.table(target).collect()]
        except Exception as exc:  # noqa: BLE001 - an unreadable optional table must not fail the analysis
            self.logger.warning("could not read %s: %s", target, exc)
            return None

    def maintain(self, table, vacuum_hours=0):
        """OPTIMIZE (Z-ORDER where defined) and, with a safe retention, VACUUM. Returns the actions that really ran."""
        t = TABLES[table]
        target = self.fqn(table)
        done = []
        try:
            zorder = f" ZORDER BY ({', '.join(t.zorder_by)})" if t.zorder_by else ""
            self.spark.sql(f"OPTIMIZE {target}{zorder}")
            done.append("OPTIMIZE")
            if vacuum_hours and vacuum_hours >= MIN_VACUUM_HOURS:
                self.spark.sql(f"VACUUM {target} RETAIN {int(vacuum_hours)} HOURS")
                done.append("VACUUM")
        except Exception as exc:  # noqa: BLE001 - maintenance must never fail the run
            self.logger.warning("maintenance skipped for %s: %s", table, exc)
        return done

In [ ]:
class FileStore:
    """Files under <lakehouse>/Files/<root>. Paths are confined to the root (ids end up in file names)."""

    def __init__(self, root_dir, base_dir=None):
        self.root = os.path.abspath(root_dir)
        self.base = os.path.abspath(base_dir) if base_dir else self.root

    @classmethod
    def for_lakehouse(cls, files_root, base="/lakehouse/default", fallback=None):
        base_dir = base if os.path.isdir(base) else (fallback or os.path.join(os.getcwd(), "_lakehouse_default"))
        return cls(os.path.join(base_dir, *safe_files_root(files_root).split("/")), base_dir)

    def path(self, *parts):
        candidate = os.path.abspath(os.path.join(self.root, *[str(p) for p in parts]))
        if os.path.commonpath([self.root, candidate]) != self.root:
            raise ValueError(f"path escapes the file store root: {candidate}")
        return candidate

    def lakehouse_path(self, rel):
        """Resolve a path relative to the Lakehouse root (e.g. Files/dea/config/x.csv), confined to it."""
        candidate = os.path.abspath(os.path.join(self.base, rel.strip("/\\")))
        if os.path.commonpath([self.base, candidate]) != self.base:
            raise ValueError(f"path escapes the lakehouse root: {candidate}")
        return candidate

    def write_bytes(self, rel, data):
        full = self.path(rel)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        with open(full, "wb") as handle:
            handle.write(data)
        return full

    def write_json(self, rel, obj, compress=True):
        payload = json.dumps(obj, default=str, ensure_ascii=False).encode("utf-8")
        return self.write_bytes(rel, gzip.compress(payload) if compress else payload)

    def write_text(self, rel, text):
        return self.write_bytes(rel, text.encode("utf-8"))

    def read_bytes(self, rel):
        with open(self.path(rel), "rb") as handle:
            return handle.read()

    def read_json(self, rel):
        data = self.read_bytes(rel)
        if rel.endswith(".gz"):
            data = gzip.decompress(data)
        return json.loads(data.decode("utf-8"))

    def exists(self, rel):
        return os.path.exists(self.path(rel))

    def list(self, rel_dir=""):
        base = self.path(rel_dir) if rel_dir else self.root
        if not os.path.isdir(base):
            return []
        return sorted(os.path.join(rel_dir, name) if rel_dir else name for name in os.listdir(base))

    def purge_older_than(self, rel_dir, days, now=None):
        base = self.path(rel_dir)
        if not os.path.isdir(base):
            return 0
        cutoff = (now or time.time()) - days * 86400
        removed = 0
        for dirpath, _, files in os.walk(base):
            for name in files:
                full = os.path.join(dirpath, name)
                if os.path.getmtime(full) < cutoff:
                    os.remove(full)
                    removed += 1
        return removed

In [ ]:
class CheckpointStore:
    """Incremental / restart state kept in Assessment_Checkpoint (one row per entity and phase)."""

    def __init__(self, lake, cfg, clock):
        self.lake = lake
        self.cfg = cfg
        self.clock = clock
        self._rows = {}
        self._dirty = {}
        self._lock = threading.Lock()

    def load(self):
        for row in self.lake.read("Assessment_Checkpoint"):
            self._rows[(row["entity_type"], row["entity_id"], row["phase"])] = row
        return len(self._rows)

    def get(self, entity_type, entity_id, phase):
        return self._rows.get((entity_type, entity_id, phase))

    def decide(self, entity_type, entity_id, phase, version=None, ttl_hours=None, force=False):
        """Return (collect: bool, reason: str) for an entity."""
        row = self.get(entity_type, entity_id, phase)
        if force or row is None:
            return True, "new"
        now = self.clock.now()
        status = row.get("status")
        attempted = row.get("last_attempt_at") or dt.datetime.min
        if status == "Done":
            if version is not None and row.get("source_version") != version:
                return True, "changed"
            completed = row.get("completed_at")
            if ttl_hours is not None and (completed is None or now - completed >= dt.timedelta(hours=ttl_hours)):
                return True, "expired"
            return False, "up-to-date"
        if status in ("Denied", "NotFound"):
            return (now - attempted >= dt.timedelta(days=self.cfg.denied_retry_days)), "retry-denied"
        if (row.get("attempts") or 0) >= self.cfg.max_attempts_per_entity:
            return (now - attempted >= dt.timedelta(days=self.cfg.failed_retry_days)), "retry-failed"
        return True, "retry"

    def mark(self, entity_type, entity_id, phase, status, version=None, error=None):
        """A version is kept for Done; for another status only when it is given explicitly (a restart records what it restarts for)."""
        now = self.clock.now()
        with self._lock:
            prev = self._rows.get((entity_type, entity_id, phase)) or {}
            keep_version = status == "Done" or version is not None
            row = {
                "entity_type": entity_type, "entity_id": entity_id, "phase": phase, "status": status,
                "attempts": 1 if status == "Done" else (prev.get("attempts") or 0) + 1,
                "last_attempt_at": now,
                "completed_at": now if status == "Done" else prev.get("completed_at"),
                "source_version": version if keep_version else prev.get("source_version"),
                "last_error": None if status == "Done" else truncate(error, 500),
            }
            self._rows[(entity_type, entity_id, phase)] = row
            self._dirty[(entity_type, entity_id, phase)] = row

    def flush(self):
        with self._lock:
            rows, self._dirty = list(self._dirty.values()), {}
        if not rows:
            return 0
        try:
            self.lake.merge("Assessment_Checkpoint", rows)
        except Exception:
            with self._lock:
                for row in rows:   # a row marked again meanwhile is newer: keep it, restore the others for the next flush
                    self._dirty.setdefault((row["entity_type"], row["entity_id"], row["phase"]), row)
            raise
        return len(rows)

## 1.5 Rule catalog and domain knowledge
The assessment is **data-driven**: every finding comes from a rule in this catalog (also persisted to
`Assessment_RuleCatalog` for audit). Rules carry a separate effect for the two migration paths because the platform
treats them differently:

| Path | Mechanics | Id | Automation |
|---|---|---|---|
| **Upgrade Wizard** (preview) | in-place, per workspace, owner runs it | **kept** | UI only |
| **Save As** (`saveAsNativeArtifact`, preview) | creates a Gen2 CI/CD copy; incremental refresh, enhanced compute, DirectQuery, AI insights, attached storage are not copied | **new** (consumers must be re-pointed) | REST API (Microsoft accelerator) |

`Verified` rules come from Microsoft documentation (wizard assessment table, Gen1/Gen2 differences, Gen2 limits);
`Heuristic` rules are detections that infer a documented limitation from indirect signals - tune them with the
`wizard_status_actual` feedback column of the governance overrides.

In [ ]:
import dataclasses as _dc

RULES_VERSION = "2026.10.1"


@_dc.dataclass(frozen=True)
class Rule:
    rule_id: str
    name: str
    category: str
    severity: str          # Blocker / Major / Minor / Info
    impact_wizard: str     # Unsupported / NeedsAttention / Deferred / None
    impact_saveas: str
    weight: int
    effort_days: float
    detection: str
    remediation: str
    confidence: str        # Verified / Heuristic
    enabled: bool = True


_RULES = [
    ("CAP-001", "Workspace not on a Fabric-capable capacity", "Capacity", "Major", "NeedsAttention", "NeedsAttention", 40, 0.5,
     "Workspace has no dedicated capacity or its SKU cannot host Fabric items (Pro, PPU, A/EM SKUs)",
     "Assign the workspace to a Fabric (F or trial) or Premium capacity first: Dataflow Gen2 is not available on Pro or PPU workspaces.", "Verified"),
    ("CAP-002", "Capacity not resolvable", "Capacity", "Info", "None", "None", 0, 0.0,
     "capacity_id is set but absent from the capacity APIs (service principals only see some capacities)",
     "Resolve the capacity manually and confirm it can host Fabric items.", "Heuristic"),
    ("STO-001", "Attached ADLS Gen2 storage (bring your own lake)", "Storage", "Major", "NeedsAttention", "Unsupported", 60, 2.0,
     "modelUrl points to a customer dfs.core.windows.net account, or the workspace has a dataflow storage account",
     "Wizard: configure an ADLS Gen2 data destination on the upgraded dataflow. Save-As does not move attached-storage data (CDM folders, BYOL): keep on Gen1 or redesign.", "Verified"),
    ("ECE-001", "Enhanced compute engine enabled", "Compute", "Major", "NeedsAttention", "NeedsAttention", 35, 1.5,
     "computeEngineBehavior is On",
     "DirectQuery over dataflows does not exist in Gen2: move DirectQuery consumers to Import or Direct Lake or read the destination tables directly.", "Verified"),
    ("ECE-002", "Enhanced compute engine set to Optimized", "Compute", "Info", "None", "None", 0, 0.0,
     "computeEngineBehavior is Optimized (default)", "No action: Gen2 uses its own staging compute.", "Heuristic"),
    ("DQ-001", "Downstream semantic model uses DirectQuery", "Lineage", "Major", "NeedsAttention", "NeedsAttention", 45, 2.0,
     "A semantic model reading the dataflow has targetStorageMode DirectQuery",
     "Reconfigure the semantic model to Import or Direct Lake: DirectQuery over dataflows is not supported in Gen2.", "Verified"),
    ("INC-001", "Incremental refresh configured", "Refresh", "Major", "NeedsAttention", "NeedsAttention", 30, 1.0,
     "An entity has an incremental refresh policy (pbi:refreshPolicy)",
     "Incremental settings are not carried over: define a supporting destination (Lakehouse, Warehouse or Azure SQL Database) and recreate the policy before the first refresh. The source must fold and expose a Date/DateTime column.", "Verified"),
    ("LNK-001", "Contains linked entities", "Lineage", "Major", "NeedsAttention", "NeedsAttention", 25, 1.0,
     "ReferenceEntity / referenceModels, PowerPlatform.Dataflows navigation or Scanner upstreamDataflows",
     "Cascading refresh through linked entities is not supported in Gen2: upgrade upstream dataflows first and orchestrate refreshes with a pipeline or schedule.", "Verified"),
    ("LNK-002", "Read by other dataflows", "Lineage", "Major", "NeedsAttention", "NeedsAttention", 20, 0.5,
     "Downstream dataflow edges exist",
     "After the upgrade refresh this dataflow, then open and save each downstream dataflow to rebind it.", "Verified"),
    ("LNK-003", "Consumer uses the legacy PowerBI.Dataflows connector", "Lineage", "Major", "NeedsAttention", "NeedsAttention", 25, 0.5,
     "M text of a downstream dataflow or semantic model contains PowerBI.Dataflows(",
     "Switch consumers to PowerPlatform.Dataflows: the legacy connector cannot read Gen2 items and the wizard does not detect it.", "Verified"),
    ("LNK-004", "Upstream dataflow not inventoried", "Data quality", "Major", "NeedsAttention", "NeedsAttention", 20, 0.25,
     "Lineage references an upstream dataflow that is not in the inventory (deleted, outside the scan scope or no access)",
     "Locate the upstream dataflow or fix the broken link: unresolved references make the wizard report Upgrade unavailable.", "Heuristic"),
    ("CMP-001", "Computed entities", "Transform", "Minor", "NeedsAttention", "NeedsAttention", 15, 0.5,
     "An entity query references another loaded entity of the same dataflow",
     "Gen2 has no computed-entity engine: validate results and use staged queries or destinations.", "Heuristic"),
    ("QRY-001", "More than 50 enabled queries", "Transform", "Major", "NeedsAttention", "NeedsAttention", 50, 3.0,
     "More than 50 enabled (loadEnabled) queries",
     "Gen2 supports at most 50 queries with staging or a destination: split the dataflow BEFORE upgrading.", "Verified"),
    ("QRY-002", "Close to the 50-query limit", "Transform", "Info", "None", "None", 0, 0.0,
     "40 to 50 enabled queries (already reflected in the technical complexity component)", "Plan headroom or split the dataflow.", "Heuristic"),
    ("CON-001", "Unresolved data source connections", "Connectivity", "Blocker", "Unsupported", "NeedsAttention", 60, 1.0,
     "The Scanner reports misconfigured datasource usages",
     "Recreate or rebind the failing connections: the wizard reports Upgrade unavailable until they resolve.", "Verified"),
    ("GW-001", "Source through a data gateway", "Connectivity", "Minor", "NeedsAttention", "NeedsAttention", 15, 0.5,
     "A source carries a gateway id",
     "Verify the gateway runs a supported version (latest six releases) and validate or recreate the Fabric connection; OAuth2 credentials on gateway connections fail after one hour.", "Verified"),
    ("OWN-001", "Owner missing or disabled", "Governance", "Major", "NeedsAttention", "NeedsAttention", 20, 0.25,
     "Owner is empty or the account is disabled (Microsoft Graph enrichment)",
     "The wizard must be run by the dataflow owner: use Take over, then confirm the Gen1 dataflow refreshes successfully.", "Verified"),
    ("NAM-001", "Name contains unsupported characters", "Governance", "Minor", "NeedsAttention", "None", 5, 0.1,
     "Name has characters other than letters, digits, whitespace and ()[]{}+-=_#",
     "The wizard strips them automatically; rename first to control the result.", "Verified"),
    ("NAM-002", "Duplicate name in the workspace", "Governance", "Minor", "NeedsAttention", "None", 5, 0.1,
     "Another dataflow of the workspace has the same name", "The wizard prefixes Migrated: rename first.", "Verified"),
    ("SCH-001", "Schedule exceeds 48 runs per day", "Refresh", "Minor", "NeedsAttention", "NeedsAttention", 10, 0.25,
     "schedule_runs_per_day is greater than 48", "Gen2 supports up to 48 scheduled refreshes per day.", "Verified"),
    ("SCH-002", "Schedule is copied disabled", "Refresh", "Info", "None", "NeedsAttention", 5, 0.1,
     "The dataflow has an enabled refresh schedule",
     "Save-As copies the schedule in a disabled state: re-enable it after validation.", "Verified"),
    ("TRG-001", "External refresh triggers observed", "Refresh", "Major", "NeedsAttention", "NeedsAttention", 20, 0.5,
     "Refresh history shows on-demand or API-triggered runs",
     "Power Automate flows, scripts and APIs calling the Power BI refresh API must move to the Fabric Job Scheduler API (and to the new id after Save-As).", "Heuristic"),
    ("PIP-001", "Referenced by data pipelines", "Lineage", "Minor", "None", "NeedsAttention", 15, 0.25,
     "A pipeline RefreshDataFlow activity references the dataflow",
     "After Save-As re-point each pipeline activity to the new dataflow id.", "Verified"),
    ("REB-001", "Downstream items to re-point after Save-As", "Lineage", "Major", "None", "NeedsAttention", 30, 0.25,
     "Downstream semantic models, reports or dataflows exist",
     "Save-As creates a new id: re-point every downstream item (effort scales with their number). The wizard keeps the id.", "Verified"),
    ("AIX-001", "AI Insights or Azure ML functions", "Transform", "Minor", "NeedsAttention", "NeedsAttention", 15, 1.0,
     "M uses AI.*, CognitiveServices.* or AzureML.* functions",
     "AI insights are not copied: rebuild with the Fabric AI prompt in Dataflow Gen2.", "Verified"),
    ("REF-001", "Refresh failing", "Refresh", "Major", "Deferred", "Deferred", 0, 0.0,
     "The latest N refreshes all failed",
     "Fix the failing refresh first: a healthy Gen1 baseline is needed to validate the migration.", "Heuristic"),
    ("STA-001", "Stale and unused", "Governance", "Info", "Deferred", "Deferred", 0, 0.0,
     "No successful refresh for N days and no downstream consumers",
     "Decommission candidate: confirm with the owner before spending migration effort.", "Heuristic"),
    ("SEQ-001", "Blocked by an unsupported or deferred upstream dataflow", "Lineage", "Major", "Deferred", "Deferred", 0, 0.0,
     "An upstream dataflow is Unsupported or Deferred",
     "Resolve the upstream dataflow first: upstream dataflows are migrated before their consumers.", "Verified"),
    ("DAT-001", "Definition not inspected", "Data quality", "Minor", "NeedsAttention", "NeedsAttention", 10, 0.0,
     "The definition could not be collected (no access, not attempted or failed)",
     "Grant the service principal workspace access or enable deeper collection, then re-run; validate with the Upgrade Wizard assessment.", "Heuristic"),
]
RULES = {r[0]: Rule(*r) for r in _RULES}
IMPACTS = ("Unsupported", "NeedsAttention", "Deferred", "None")


def rule_catalog_rows():
    return [{"rule_id": r.rule_id, "name": r.name, "category": r.category, "severity": r.severity, "impact_wizard": r.impact_wizard,
             "impact_saveas": r.impact_saveas, "weight": r.weight, "effort_days": r.effort_days, "detection": r.detection,
             "remediation": r.remediation, "confidence": r.confidence, "enabled": r.enabled, "version": RULES_VERSION}
            for r in RULES.values()]

In [ ]:
# Datasource classification (Scanner datasourceType / M function prefix) -> category and relative migration weight.
SOURCE_CATEGORY_RULES = [
    ("Dataflow", ("powerplatform.dataflows", "powerbi.dataflows", "dataflow")),
    ("Fabric", ("lakehouse", "warehouse", "onelake", "fabric")),
    ("File", ("file", "folder", "excel", "csv", "text", "json", "xml", "pdf")),
    ("Cloud storage", ("azureblob", "blob", "azurestorage", "datalake", "adls", "amazons3", "s3", "googlecloudstorage")),
    ("SaaS", ("sharepoint", "salesforce", "dynamics", "dataverse", "commondataservice", "googleanalytics", "servicenow", "zendesk",
              "marketo", "hubspot", "jira", "github", "smartsheet")),
    ("Web", ("web", "odata", "rest", "http", "api")),
    ("Database", ("sql", "oracle", "postgre", "mysql", "odbc", "oledb", "snowflake", "teradata", "db2", "sap", "hana", "analysisservices",
                  "kusto", "dataexplorer", "databricks", "synapse", "redshift", "bigquery", "mongo", "cosmos", "access", "informix", "sybase")),
]
SOURCE_COMPLEXITY = {"Database": 1.0, "Fabric": 0.5, "Cloud storage": 1.0, "Dataflow": 1.0, "File": 1.5, "SaaS": 1.5, "Web": 2.0, "Other": 2.0}
FILE_BASED_CATEGORIES = {"File"}


def categorize_source(datasource_type):
    text = (datasource_type or "").lower().replace(" ", "")
    for category, needles in SOURCE_CATEGORY_RULES:
        if any(n in text for n in needles):
            return category
    return "Other"


def classify_sku(sku):
    """Return (capacity_kind, is_fabric_capable). Capability of P SKUs depends on the tenant enabling Fabric."""
    s = (sku or "").upper().strip()
    if not s:
        return "Unknown", None
    if s.startswith("FT") or "TRIAL" in s:
        return "Trial", True
    if re.fullmatch(r"F\d+", s):
        return "Fabric F", True
    if re.fullmatch(r"P\d+", s):
        return "Premium P", True
    if re.fullmatch(r"(EM|A)\d+", s):
        return "Embedded", False
    return "Unknown", None


_BOUND = r"(?:^|[\s_\-\.\[\(/])"
_TAIL = r"(?:[\s_\-\.\]\)/]|$)"
DEFAULT_ENV_PATTERNS = {
    "Prod": rf"{_BOUND}(prd|prod|production|live){_TAIL}",
    "Test": rf"{_BOUND}(tst|test|uat|qa|sit|stg|stage|staging|preprod|pre-prod|ppd){_TAIL}",
    "Dev": rf"{_BOUND}(dev|development|sandbox|poc){_TAIL}",
}


def detect_environment(name, patterns=None):
    """Environment from a workspace name; ambiguous or unmatched names are Unknown."""
    found = [env for env, pattern in (patterns or DEFAULT_ENV_PATTERNS).items() if re.search(pattern, name or "", re.IGNORECASE)]
    return found[0] if len(found) == 1 else "Unknown"


def stage_environment(order, stage_count):
    """Map a deployment pipeline stage to an environment (first = Dev, last = Prod, others = Test)."""
    if order is None or not stage_count or stage_count < 2:
        return "Unknown"
    return "Dev" if order == 0 else ("Prod" if order == stage_count - 1 else "Test")


ALLOWED_NAME_CHARS = re.compile(r"^[\w\s()\[\]{}+\-=#]+$", re.UNICODE)

## 1.6 Orchestration
The notebook is a **pipeline of named steps** grouped in sections. Every section ends with a small run cell
(`run_steps(ctx, sections={n})`), so the notebook can be executed top to bottom (*Run all*, a pipeline notebook activity,
a schedule) or section by section while troubleshooting.

| Run mode | Steps executed |
|---|---|
| `incremental` (default) | everything; the Scanner reads only workspaces modified since the last complete scan, collectors skip entities that are up to date |
| `full` | everything; full Scanner pass and every definition is read again |
| `resume` | like `incremental`, continues the run id in `p_resume_run_id` (log sequence and checkpoints carry on) |
| `refresh_only` | refresh history and schedules only, then analysis, publish and report |
| `analysis_only` | no API call: re-evaluates rules, scores and waves from the stored Silver tables (tune weights, rules or governance overrides) |

A critical failure (authentication, access check) **aborts** the remaining steps; the logs are still persisted and the notebook fails at the end.

In [ ]:
import tempfile

COLLECTION_MODES = ("full", "incremental", "resume")
LIVE_MODES = COLLECTION_MODES + ("refresh_only",)


@dataclasses.dataclass(frozen=True)
class Step:
    section: int
    name: str
    func: str       # resolved by name when the step runs: later sections are defined in later cells
    modes: tuple
    enabled: object
    critical: bool = False
    needs: tuple = ()   # phases whose output this step works on: it is skipped when one of them failed


def access_check(ctx):
    """Stop early when the identity can reach neither admin API family: nothing useful could be discovered."""
    access = ctx.state.get("access") or {}
    if access.get("pbi_admin") != "OK" and access.get("fabric_admin") != "OK":
        raise AuthError("neither the Power BI nor the Fabric admin APIs are accessible: " + (", ".join(f"{k}={v}" for k, v in access.items()) or "preflight did not complete"))
    return {"message": ", ".join(f"{k}={v}" for k, v in access.items())}


def pipeline_steps():
    always = lambda cfg: True  # noqa: E731
    return [
        Step(2, "preflight", "preflight", LIVE_MODES, always, critical=True),
        Step(2, "access_check", "access_check", COLLECTION_MODES, always, critical=True),
        Step(3, "governance", "phase_governance", RUN_MODES, always),
        Step(3, "capacities", "phase_capacities", COLLECTION_MODES, always),
        Step(3, "workspaces", "phase_workspaces", COLLECTION_MODES, always),
        Step(3, "scan", "phase_scan", COLLECTION_MODES, always),
        Step(4, "admin_dataflows", "phase_admin_dataflows", COLLECTION_MODES, always),
        Step(4, "admin_items", "phase_admin_items", COLLECTION_MODES, always),
        Step(4, "reconcile", "phase_reconcile_dataflows", COLLECTION_MODES, always),
        Step(5, "definitions", "phase_definitions", COLLECTION_MODES, lambda c: c.collect_definitions or c.collect_gen2_definitions),
        Step(5, "compute_engine", "phase_compute_engine", COLLECTION_MODES, lambda c: c.collect_compute_engine),
        Step(5, "admin_datasources", "phase_admin_datasources", COLLECTION_MODES, lambda c: c.collect_admin_datasources),
        Step(5, "upstream_verify", "phase_upstream_verify", COLLECTION_MODES, lambda c: c.verify_upstream_admin_api),
        Step(5, "pipelines", "phase_pipelines", COLLECTION_MODES, lambda c: c.collect_pipeline_references),
        Step(5, "connections", "phase_item_connections", COLLECTION_MODES, lambda c: c.collect_item_connections),
        Step(5, "owners_graph", "phase_owners_graph", COLLECTION_MODES, lambda c: c.enrich_owners_graph),
        Step(5, "source_rollup", "phase_source_rollup", COLLECTION_MODES, always),
        Step(6, "refresh", "phase_refresh_history", LIVE_MODES, lambda c: c.collect_refresh_history),
        Step(7, "analysis", "phase_analysis", RUN_MODES, always),
        Step(8, "publish", "phase_publish", RUN_MODES, always, needs=("analysis",)),
        Step(8, "maintenance", "phase_maintenance", RUN_MODES, lambda c: c.run_maintenance and not c.dry_run, needs=("publish",)),
        Step(9, "report", "phase_report", RUN_MODES, always, needs=("publish",)),
    ]


def run_steps(ctx, sections=None):
    """Run the steps of the given sections (all when None) that apply to the run mode and the parameters."""
    for step in pipeline_steps():
        if sections is not None and step.section not in sections:
            continue
        if ctx.aborted:
            ctx.logger.warning("run aborted - skipping %s", step.name)
            continue
        in_mode = ctx.cfg.run_mode in step.modes
        reason = "disabled by parameters" if in_mode else f"not part of run mode {ctx.cfg.run_mode}"
        broken = [n for n in step.needs if ctx.phase_status.get(n) == "Failed" or n in ctx.blocked_phases]
        if in_mode and broken:
            ctx.blocked_phases.add(step.name)
            reason = f"skipped because {', '.join(broken)} did not complete (the Gold tables could be inconsistent)"
        else:
            ctx.blocked_phases.discard(step.name)    # the marker describes the latest evaluation: a recovered step must not keep blocking its followers
        try:
            ctx.run_phase(step.name, globals()[step.func], enabled=in_mode and not broken and bool(step.enabled(ctx.cfg)), skip_reason=reason, critical=step.critical)
        except Exception as exc:  # noqa: BLE001 - run_phase re-raises critical failures; the run is aborted but its log is kept
            ctx.aborted, ctx.abort_reason = True, f"{step.name}: {type(exc).__name__}: {exc}"[:500]
            ctx.logger.error("critical step failed - the run is aborted: %s", ctx.abort_reason)
    return ctx.phase_status


def _try_import_notebookutils():
    try:
        import notebookutils
        return notebookutils
    except ImportError:
        return None


def _make_lake(cfg, ns, ctx, clock, logger):
    if cfg.dry_run:
        logger.info("dry run: results stay in memory, nothing is written to the Lakehouse")
        return InMemoryLake(ctx.run_id, clock.now)
    spark = ns.get("spark")
    if spark is None:
        raise ConfigError("no Spark session found: run the notebook in Fabric, or set p_dry_run = True for an in-memory run")
    return SparkLake(spark, cfg.lakehouse_schema, ctx.run_id, logger, clock.now)


def _make_files(cfg):
    mounted = "/lakehouse/default"
    if cfg.dry_run:
        return FileStore(os.path.join(tempfile.mkdtemp(prefix="dea-dry-"), "dea"), mounted if os.path.isdir(mounted) else None)
    if not os.path.isdir(mounted):
        raise ConfigError("attach a default Lakehouse to the notebook: /lakehouse/default is not mounted")
    return FileStore.for_lakehouse(cfg.files_root)


def start_run(ns, deps=None):
    """Sections 1-2: validate the parameters, then wire logging, storage and identity. `deps` lets tests inject collaborators."""
    deps = deps or {}
    cfg = deps.get("cfg") or Config.from_namespace(ns)
    logger, capture = setup_logging(cfg.log_level)
    clock = deps.get("clock") or SystemClock()
    resumed = cfg.run_mode == "resume"
    ctx = RunContext(cfg, run_id=cfg.resume_run_id if resumed else None, clock=clock, logger=logger)
    ctx.log_capture, ctx.aborted, ctx.abort_reason = capture, False, None
    logger.info("Dataflow assessment %s | rules %s | run %s | mode=%s | dry_run=%s", NOTEBOOK_VERSION, RULES_VERSION, ctx.run_id, cfg.run_mode, cfg.dry_run)
    ctx.lake = deps.get("lake") or _make_lake(cfg, ns, ctx, clock, logger)
    ctx.lake.run_id = ctx.run_id
    ctx.lake.ensure_tables()
    ctx.files = deps.get("files") or _make_files(cfg)
    logger.info("files root: %s", ctx.files.root)
    ctx.checkpoints = CheckpointStore(ctx.lake, cfg, clock)
    loaded = ctx.checkpoints.load()
    if resumed:
        ctx.continue_sequence(max((r["seq"] or 0 for r in ctx.lake.read("Assessment_RunLog", where={"run_id": ctx.run_id})), default=0))
    if cfg.run_mode != "analysis_only":
        session = deps.get("session") or ThreadSessions()
        tokens = deps.get("tokens") or resolve_identity(cfg, ns.get("notebookutils") or _try_import_notebookutils(), session, clock, logger)
        build_clients(ctx, tokens, session=session)
    logger.info("ready: %d checkpoints loaded", loaded)
    return ctx


def exit_notebook(ns, summary):
    """Hand the outcome to a calling pipeline (notebookutils.notebook.exit). A no-op outside Fabric."""
    nbutils = ns.get("notebookutils") or _try_import_notebookutils()
    if nbutils is None:
        return summary
    keys = ("run_id", "status", "mode", "dataflows_total", "ready_count", "needs_attention_count", "unsupported_count", "deferred_count", "total_effort_days")
    nbutils.notebook.exit(json.dumps({k: summary.get(k) for k in keys}, default=str))
    return summary

## 1.7 Silver row builders
Pure functions shared by the discovery sections (3 and 4) and the collectors: they turn Scanner payloads into
`Assessment_Dataflows`, `Assessment_DataflowSources` and `Assessment_Lineage` rows, and persist them with the
partial-column merge of the storage layer. They live in Section 1 because the Scanner phase (Section 3) and the later
collectors all depend on them.

In [ ]:
import urllib.parse

DF_SCAN_COLS = ["dataflow_key", "workspace_id", "dataflow_name", "dataflow_type", "generation", "description", "tags", "owner_upn",
                "owner_source", "modified_by", "modified_at", "endorsement", "certified_by", "sensitivity_label_id", "schedule_enabled",
                "schedule_days", "schedule_times", "schedule_runs_per_day", "schedule_timezone", "schedule_notify_option",
                "first_seen_dataflow_type", "type_changed_at"]
WS_SCAN_COLS = ["admin_upns", "admin_count", "gen1_dataflow_count", "gen2_dataflow_count", "dataset_count", "report_count",
                "dashboard_count", "other_item_count", "scanned_at", "scan_state"]
_KNOWN_ITEM_ARRAYS = {"reports", "dashboards", "datasets", "dataflows", "datamarts", "users", "tags"}


def merge_by_signature(lake, table, rows, **kw):
    """Merge rows grouped by the set of columns they supply, so a missing column never nulls existing data."""
    groups = defaultdict(list)
    for row in rows:
        groups[frozenset(row)].append(row)
    return sum(lake.merge(table, group, **kw) for group in groups.values())


def host_of(details):
    server = str(details.get("server") or "")
    if server:
        return server.split(",")[0].split(":")[0].split("\\")[0].strip().lower() or None
    url = str(details.get("url") or details.get("sharePointSiteUrl") or "")
    if url:
        return (urllib.parse.urlparse(url if "//" in url else "//" + url).hostname or "").lower() or None
    path = str(details.get("path") or "").replace("\\", "/").strip("/")
    return path.split("/")[0].lower() or None if path else None


def source_key_for(dtype, details):
    parts = [str(dtype or "").lower()] + [str(details.get(k) or "").lower() for k in ("server", "database", "url", "path", "sharePointSiteUrl")]
    return short_hash("|".join(parts))


def storage_type_of(model_url):
    if not model_url:
        return "Unknown"
    host = (urllib.parse.urlparse(str(model_url)).hostname or "").lower()
    return "CustomADLS" if host.endswith(".dfs.core.windows.net") else "PowerBIManaged"


def build_source_row(dataflow_id, workspace_id, datasource_id, inst, via, misconfigured=False):
    inst = inst or {}
    details = inst.get("details") or {}
    dtype = inst.get("type") or "Unknown"
    gateway = inst.get("gateway_id")
    category = categorize_source(dtype)
    text = json.dumps(details, sort_keys=True, default=str)
    local_path = category in FILE_BASED_CATEGORIES and bool(re.match(r"^([a-zA-Z]:\\|\\\\)", str(details.get("path") or "")))
    return {
        "dataflow_id": dataflow_id, "workspace_id": workspace_id, "source_key": source_key_for(dtype, details) if details else short_hash(f"id|{datasource_id}"),
        "datasource_id": datasource_id, "datasource_type": dtype, "source_category": category, "server": details.get("server"),
        "database": details.get("database"), "url": details.get("url") or details.get("sharePointSiteUrl"), "path": details.get("path"),
        "host": host_of(details), "connection_details": text if details else None, "gateway_id": gateway, "gateway_type": "Unknown" if gateway else None,
        "uses_gateway": bool(gateway), "is_on_premises": bool(gateway) or local_path, "is_misconfigured": bool(misconfigured or inst.get("misconfigured")),
        "discovered_via": via,
    }


def edge_row(src_type, src_id, src_ws, src_name, dst_type, dst_id, dst_ws, dst_name, relationship, kind="DATA", evidence="Scanner", attributes=None):
    src_id, dst_id = gid(src_id), gid(dst_id)
    return {"edge_id": short_hash(f"{src_id}|{dst_id}|{relationship}|{evidence}"), "src_type": src_type, "src_id": src_id, "src_workspace_id": gid(src_ws),
            "src_name": src_name, "dst_type": dst_type, "dst_id": dst_id, "dst_workspace_id": gid(dst_ws), "dst_name": dst_name,
            "relationship": relationship, "edge_kind": kind, "evidence": evidence,
            "attributes": json.dumps(attributes, sort_keys=True) if attributes else None}


def _schedule_columns(schedule):
    if not isinstance(schedule, dict):
        return {}
    times = [str(t) for t in (schedule.get("times") or [])]
    return {"schedule_enabled": _as_bool(schedule.get("enabled")) if schedule.get("enabled") is not None else None,
            "schedule_days": [str(d) for d in (schedule.get("days") or [])], "schedule_times": times, "schedule_runs_per_day": len(times),
            "schedule_timezone": schedule.get("localTimeZoneId"), "schedule_notify_option": schedule.get("notifyOption")}


def _endorsement(details):
    if isinstance(details, dict):
        return details.get("endorsement"), details.get("certifiedBy")
    return None, None


def _generation(d):
    return coerce_value("INT", d.get("generation"))


def parse_scan_result(ctx, result, batch_ids=()):
    """Normalize one Scanner result into Silver rows. Tolerant of undocumented / missing fields."""
    now = ctx.clock.now()
    instances = {}
    for key in ("datasourceInstances", "misconfiguredDatasourceInstances"):
        for inst in result.get(key) or []:
            did = gid(inst.get("datasourceId"))
            if did:
                instances[did] = {"type": inst.get("datasourceType"), "details": inst.get("connectionDetails") or {}, "gateway_id": gid(inst.get("gatewayId")),
                                  "misconfigured": key.startswith("misconfigured") or instances.get(did, {}).get("misconfigured", False)}
    out = {"workspaces": [], "dataflows": [], "sources": [], "edges": [], "scanned_workspace_ids": [], "missing_workspace_ids": [], "dataflow_ids_by_workspace": {}}
    for ws in result.get("workspaces") or []:
        wid = gid(ws.get("id"))
        if not wid:
            continue
        dataflows = [d for d in (ws.get("dataflows") or []) if gid(d.get("objectId") or d.get("id"))]
        admins = [str(u.get("emailAddress") or u.get("identifier") or u.get("displayName") or "") for u in (ws.get("users") or [])
                  if str(u.get("groupUserAccessRight") or "").lower() == "admin"]
        others = sum(len(v) for k, v in ws.items() if k not in _KNOWN_ITEM_ARRAYS and isinstance(v, list) and v and isinstance(v[0], dict))
        out["workspaces"].append({
            "workspace_id": wid, "admin_upns": [a for a in admins if a], "admin_count": len([a for a in admins if a]),
            "gen1_dataflow_count": sum(1 for d in dataflows if _generation(d) != 2), "gen2_dataflow_count": sum(1 for d in dataflows if _generation(d) == 2),
            "dataset_count": len(ws.get("datasets") or []), "report_count": len(ws.get("reports") or []),
            "dashboard_count": len(ws.get("dashboards") or []), "other_item_count": others, "scanned_at": now, "scan_state": "Scanned"})
        out["scanned_workspace_ids"].append(wid)
        ids_here = out["dataflow_ids_by_workspace"].setdefault(wid, set())
        for d in dataflows:
            did = gid(d.get("objectId") or d.get("id"))
            if not did:
                continue
            ids_here.add(did)
            generation = _generation(d)
            endorsement, certified_by = _endorsement(d.get("endorsementDetails"))
            out["dataflows"].append({
                "dataflow_key": stable_hash64(did), "dataflow_id": did, "workspace_id": wid, "dataflow_name": d.get("name"),
                "dataflow_type": "Gen2" if generation == 2 else "Gen1", "generation": generation or 1, "description": d.get("description"),
                "tags": [t.get("displayName") if isinstance(t, dict) else str(t) for t in (d.get("tags") or [])],
                "owner_upn": d.get("configuredBy"), "owner_source": "configuredBy" if d.get("configuredBy") else None,
                "modified_by": d.get("modifiedBy"), "modified_at": parse_ts(d.get("modifiedDateTime")), "endorsement": endorsement or "None",
                "certified_by": certified_by, "sensitivity_label_id": dig(d, "sensitivityLabel", "labelId"), **_schedule_columns(d.get("refreshSchedule"))})
            seen_sources = set()
            for usages, misconfigured in ((d.get("datasourceUsages"), False), (d.get("misconfiguredDatasourceUsages"), True)):
                for usage in usages or []:
                    sid = gid(usage.get("datasourceInstanceId"))
                    if sid and (sid, misconfigured) not in seen_sources:
                        seen_sources.add((sid, misconfigured))
                        row = build_source_row(did, wid, sid, instances.get(sid), "Scanner", misconfigured)
                        out["sources"].append(row)
                        details = (instances.get(sid) or {}).get("details") or {}
                        label = f"{(instances.get(sid) or {}).get('type') or 'Datasource'}: {details.get('server') or details.get('url') or details.get('path') or sid}"
                        out["edges"].append(edge_row("Datasource", sid, wid, label, "Dataflow", did, wid, d.get("name"), "DatasourceUsage"))
            for up in d.get("upstreamDataflows") or []:
                out["edges"].append(edge_row("Dataflow", up.get("targetDataflowId"), up.get("groupId"), None, "Dataflow", did, wid, d.get("name"), "LinkedEntity"))
        for ds in ws.get("datasets") or []:
            dsid = gid(ds.get("id"))
            if not dsid:
                continue
            text = json.dumps([ds.get("expressions"), ds.get("tables")], default=str).lower()
            attrs = {"storage_mode": ds.get("targetStorageMode"), "legacy_connector": "powerbi.dataflows(" in text}
            attrs = {k: v for k, v in attrs.items() if v}
            for up in ds.get("upstreamDataflows") or []:
                out["edges"].append(edge_row("Dataflow", up.get("targetDataflowId"), up.get("groupId"), None, "SemanticModel", dsid, wid, ds.get("name"),
                                             "DatasetSource", attributes=attrs or None))
        for rep in ws.get("reports") or []:
            if rep.get("datasetId") and rep.get("id"):
                out["edges"].append(edge_row("SemanticModel", rep["datasetId"], wid, None, "Report", rep["id"], wid, rep.get("name"), "ReportOnDataset"))
        for dash in ws.get("dashboards") or []:
            did_ = gid(dash.get("id"))
            for tile in dash.get("tiles") or []:
                for kind, key in (("SemanticModel", "datasetId"), ("Report", "reportId")):
                    if tile.get(key) and did_:
                        out["edges"].append(edge_row(kind, tile[key], wid, None, "Dashboard", did_, wid, dash.get("displayName") or dash.get("name"), "DashboardTile"))
        for mart in ws.get("datamarts") or []:
            for up in mart.get("upstreamDataflows") or mart.get("UpstreamDataflows") or []:
                out["edges"].append(edge_row("Dataflow", up.get("targetDataflowId"), up.get("groupId"), None, "Datamart", mart.get("id"), wid, mart.get("name"), "DatamartSource"))
    unique = {e["edge_id"]: e for e in out["edges"] if e["src_id"] and e["dst_id"]}
    out["edges"] = list(unique.values())
    requested = {gid(w) for w in batch_ids or ()} - {None}
    out["missing_workspace_ids"] = sorted(requested - set(out["scanned_workspace_ids"]))   # requested, but the Scanner did not return them
    return out


def persist_scan(ctx, parsed):
    lake, scanned = ctx.lake, set(parsed["scanned_workspace_ids"])
    existing = {r["dataflow_id"]: r for r in lake.read("Assessment_Dataflows", where={"workspace_id": scanned})} if scanned else {}
    for row in parsed["dataflows"]:
        prev = existing.get(row["dataflow_id"]) or {}
        row["first_seen_dataflow_type"] = prev.get("first_seen_dataflow_type") or row["dataflow_type"]
        changed = prev.get("dataflow_type") not in (None, row["dataflow_type"])
        row["type_changed_at"] = ctx.clock.now() if changed else prev.get("type_changed_at")
    merge_by_signature(lake, "Assessment_Dataflows", parsed["dataflows"], update_cols=DF_SCAN_COLS)
    lake.merge("Assessment_Workspaces", parsed["workspaces"], update_cols=WS_SCAN_COLS)

    new_sources = {(r["dataflow_id"], r["source_key"]) for r in parsed["sources"]}
    stale = [{"dataflow_id": r["dataflow_id"], "source_key": r["source_key"]} for r in lake.read("Assessment_DataflowSources", where={"workspace_id": scanned})
             if r.get("discovered_via") == "Scanner" and (r["dataflow_id"], r["source_key"]) not in new_sources] if scanned else []
    if stale:
        lake.mark_deleted("Assessment_DataflowSources", stale)
    merge_by_signature(lake, "Assessment_DataflowSources", parsed["sources"])

    new_edges = {e["edge_id"] for e in parsed["edges"]}
    stale_edges = [{"edge_id": r["edge_id"]} for r in lake.read("Assessment_Lineage", where={"dst_workspace_id": scanned})
                   if r.get("evidence") == "Scanner" and r["edge_id"] not in new_edges] if scanned else []
    if stale_edges:
        lake.mark_deleted("Assessment_Lineage", stale_edges)
    lake.merge("Assessment_Lineage", parsed["edges"])

    disc = ctx.state.setdefault("discovery", {"scanner": {}, "admin_pbi": {}, "admin_items": {}, "complete": {}})
    for row in parsed["dataflows"]:
        disc["scanner"][row["dataflow_id"]] = {"workspace_id": row["workspace_id"], "generation": row["generation"]}

# 2. Authentication and API client
**Identity.** `p_auth_mode`:
* `spn` - client-credentials flow; tenant id, client id and secret come from **Azure Key Vault** through
  `notebookutils.credentials.getSecret` (only secret *names* are parameters). Separate tokens are requested for the
  Power BI (`https://analysis.windows.net/powerbi/api/.default`) and Fabric (`https://api.fabric.microsoft.com/.default`)
  audiences.
* `user` - the identity running the notebook (`notebookutils.credentials.getToken("pbi")`), same pattern as the existing
  example notebooks. Needs the Fabric administrator role for the admin APIs.
* `auto` - SPN when Key Vault is configured and readable, otherwise the notebook identity.

**Resilience.** One `ApiClient` per API family with: per-bucket sliding-window rate limits (documented quotas x
`p_rate_limit_scale`), `Retry-After` aware retry with exponential back-off and jitter, token refresh on 401, long-running
operation polling, and pagination for every response shape used by these APIs. When a wait would exceed
`p_max_retry_after_sec` the phase stops gracefully and the next run resumes from the checkpoints.

In [ ]:
import base64
import urllib.parse


def jwt_claims(token):
    """Decode (without verifying) the JWT payload for diagnostics. Never log the token itself."""
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.urlsafe_b64decode(payload.encode()))
    except Exception:  # noqa: BLE001
        return {}
    keep = ("aud", "tid", "appid", "azp", "upn", "unique_name", "idtyp", "exp", "roles", "scp")
    return {k: claims[k] for k in keep if k in claims}


class TokenProvider:
    def token(self, audience):
        raise NotImplementedError

    def invalidate(self, audience=None):
        return None

    def describe(self):
        return {"mode": "unknown"}


class SpnTokenProvider(TokenProvider):
    SCOPES = {"pbi": PBI_SCOPE, "fabric": FABRIC_SCOPE, "graph": GRAPH_SCOPE}

    def __init__(self, tenant_id, client_id, client_secret, authority_host, session, clock, margin_sec=300):
        self._tenant, self._client, self._secret = tenant_id, client_id, client_secret
        self._authority = authority_host.rstrip("/")
        self._session, self._clock, self._margin = session, clock, margin_sec
        self._cache = {}
        self._lock = threading.Lock()

    def token(self, audience):
        with self._lock:
            cached = self._cache.get(audience)
            if cached and cached[1] - self._clock.monotonic() > self._margin:
                return cached[0]
            scope = self.SCOPES[audience]
            url = f"{self._authority}/{self._tenant}/oauth2/v2.0/token"
            form = {"grant_type": "client_credentials", "client_id": self._client, "client_secret": self._secret, "scope": scope}
            try:
                resp = self._session.request("POST", url, data=form, timeout=60)
            except requests.RequestException as exc:
                raise AuthError(f"token endpoint unreachable: {type(exc).__name__}", retryable=True) from None
            if resp.status_code != 200:
                detail = ""
                try:
                    detail = (resp.json() or {}).get("error_description", "")[:300]
                except ValueError:
                    pass
                raise AuthError(f"token request for {audience} failed with HTTP {resp.status_code}: {detail}", status=resp.status_code)
            body = resp.json()
            self._cache[audience] = (body["access_token"], self._clock.monotonic() + float(body.get("expires_in", 3600)))
            return body["access_token"]

    def invalidate(self, audience=None):
        with self._lock:
            self._cache.pop(audience, None) if audience else self._cache.clear()

    def describe(self):
        return {"mode": "spn", "client_id": (self._client or "")[:8] + "…", "tenant": (self._tenant or "")[:8] + "…"}


class NotebookTokenProvider(TokenProvider):
    """Identity running the notebook. Fabric accepts the notebook 'pbi' token (same as the Microsoft accelerator)."""

    KEYS = {"pbi": "pbi", "fabric": "pbi"}

    def __init__(self, nbutils):
        self._nbutils = nbutils

    def token(self, audience):
        if audience not in self.KEYS:
            raise AuthError(f"audience {audience} needs a service principal (set p_auth_mode = spn)")
        return self._nbutils.credentials.getToken(self.KEYS[audience])

    def describe(self):
        return {"mode": "user"}


def resolve_identity(cfg, nbutils, session, clock, logger):
    """Pick the token provider. SPN secrets are read from Key Vault and never logged."""
    want_spn = cfg.auth_mode == "spn" or (cfg.auth_mode == "auto" and bool(cfg.key_vault_url))
    if want_spn:
        try:
            secrets = [nbutils.credentials.getSecret(cfg.key_vault_url, name) for name in
                       (cfg.kv_secret_tenant_id, cfg.kv_secret_client_id, cfg.kv_secret_client_secret)]
            provider = SpnTokenProvider(*secrets, cfg.authority_host, session, clock)
            provider.token("pbi")
            logger.info("authenticated as service principal %s", provider.describe()["client_id"])
            return provider
        except Exception as exc:  # noqa: BLE001
            if cfg.auth_mode == "spn":
                raise AuthError(f"service principal authentication failed: {type(exc).__name__}: {str(exc)[:200]}") from None
            logger.warning("service principal unavailable (%s) - falling back to the notebook identity", type(exc).__name__)
    if nbutils is None:
        raise AuthError("no notebook identity available outside Fabric; configure the service principal")
    provider = NotebookTokenProvider(nbutils)
    claims = jwt_claims(provider.token("pbi"))
    logger.info("authenticated as notebook identity %s", claims.get("upn") or claims.get("unique_name") or claims.get("appid") or "?")
    return provider

In [ ]:
# Documented quotas as (calls, window_seconds). A call may be subject to several buckets; the most restrictive binds.
DEFAULT_RATE_LIMITS = {
    "pbi_scan_modified": [(30, 3600)],
    "pbi_scan_getinfo": [(500, 3600)],
    "pbi_scan_status": [(10000, 3600)],
    "pbi_scan_result": [(500, 3600)],
    "pbi_admin_groups": [(15, 60), (50, 3600)],
    "pbi_admin_capacities": [(200, 3600)],
    "pbi_admin_pipelines": [(200, 3600)],
    "pbi_admin_dataflows": [(200, 3600)],
    "pbi_admin_dataflow_export": [(200, 3600)],
    "pbi_admin_dataflow_datasources": [(200, 3600)],
    "pbi_admin_dataflow_upstream": [(200, 3600)],
    "pbi_admin_dataflow_users": [(200, 3600)],
    "pbi_admin_dataset_links": [(200, 3600)],
    "pbi_dataflow": [(3000, 3600)],
    "fabric_admin_items": [(200, 3600)],
    "fabric_admin_workspaces": [(200, 3600)],
    "fabric_admin_domains": [(25, 60)],
    "fabric_admin_tags": [(25, 60)],
    "fabric_platform": [(500, 60)],
    "fabric_jobs": [(200, 60)],
    "fabric_lro": [(500, 60)],
    "graph": [(600, 60)],
}


class RateLimiter:
    """Thread-safe sliding-window limiter with server-driven penalties (Retry-After)."""

    def __init__(self, rules, clock, scale=1.0, max_wait=None):
        self._rules = {b: [(max(1, int(n * scale)), w) for n, w in r] for b, r in rules.items()}
        self._hits = defaultdict(deque)
        self._blocked = {}
        self._clock = clock
        self._lock = threading.Lock()
        self.max_wait = max_wait

    def _wait_needed(self, bucket, now):
        wait = max(0.0, self._blocked.get(bucket, 0.0) - now)
        for calls, window in self._rules.get(bucket, []):
            hits = self._hits[(bucket, window)]
            while hits and hits[0] <= now - window:
                hits.popleft()
            if len(hits) >= calls:
                wait = max(wait, hits[0] + window - now)
        return wait

    def acquire(self, buckets):
        buckets = (buckets,) if isinstance(buckets, str) else tuple(buckets)
        while True:
            with self._lock:
                now = self._clock.monotonic()
                wait = max(self._wait_needed(b, now) for b in buckets)
                if wait <= 0:
                    for b in buckets:
                        for _, window in self._rules.get(b, []):
                            self._hits[(b, window)].append(now)
                    return
            if self.max_wait is not None and wait > self.max_wait:
                raise ThrottleExhausted(f"rate limit on {buckets[0]} requires waiting {int(wait)}s (> {int(self.max_wait)}s allowed)", status=429)
            self._clock.sleep(min(wait, 60.0) + 0.05)

    def penalize(self, buckets, seconds):
        buckets = (buckets,) if isinstance(buckets, str) else tuple(buckets)
        with self._lock:
            until = self._clock.monotonic() + seconds
            for b in buckets:
                self._blocked[b] = max(self._blocked.get(b, 0.0), until)


def build_rate_limits(cfg):
    rules = {b: list(r) for b, r in DEFAULT_RATE_LIMITS.items()}
    for bucket, windows in cfg.rate_limit_overrides.items():
        rules[bucket] = [(int(n), int(w)) for n, w in windows]
    return rules


def parse_retry_after(resp, default=None):
    value = resp.headers.get("Retry-After") if resp.headers else None
    if value:
        try:
            return max(0.0, float(value))
        except ValueError:
            try:
                from email.utils import parsedate_to_datetime
                when = parsedate_to_datetime(value)
                return max(0.0, (when - dt.datetime.now(when.tzinfo)).total_seconds())
            except (TypeError, ValueError):
                pass
    match = re.search(r"[Rr]etry (?:in|after) (\d+)", getattr(resp, "text", "") or "")
    return float(match.group(1)) if match else default


def error_details(resp):
    """Extract (code, message) from Fabric ({errorCode, message}) and Power BI ({error:{code,message}}) bodies."""
    try:
        body = resp.json()
    except ValueError:
        return None, (resp.text or "")[:300]
    if isinstance(body, dict):
        err = body.get("error")
        if isinstance(err, dict):
            return err.get("code"), err.get("message")
        return body.get("errorCode") or body.get("code"), body.get("message") or (err if isinstance(err, str) else None)
    return None, str(body)[:300]


class ThreadSessions:
    """One requests.Session per thread (a Session is not documented as thread-safe) behind the single call the notebook uses.

    The factory looks `requests.Session` up when a thread first needs it, so replacing it (tests, proxies) keeps working."""

    def __init__(self, factory=None):
        self._factory = factory or (lambda: requests.Session())
        self._local = threading.local()

    def request(self, *args, **kwargs):
        session = getattr(self._local, "session", None)
        if session is None:
            session = self._local.session = self._factory()
        return session.request(*args, **kwargs)


class ApiClient:
    RETRYABLE = (408, 500, 502, 503, 504)

    def __init__(self, name, base_url, audience, tokens, limiter, stats, clock, cfg, session=None, logger=None, rng=None):
        self.name = name
        self.base = base_url.rstrip("/")
        self.audience = audience
        self.tokens = tokens
        self.limiter = limiter
        self.stats = stats
        self.clock = clock
        self.cfg = cfg
        self.session = session or ThreadSessions()
        self.logger = logger or logging.getLogger("dea")
        self.rng = rng or random.Random()
        self.user_agent = f"DataflowEstateAssessment/{NOTEBOOK_VERSION}"

    def url(self, path):
        """Relative paths hang off the base URL. Absolute URLs (continuation links, operation locations) must stay on the base host:
        the bearer token is never sent anywhere else."""
        if not path.startswith(("http://", "https://")):
            return f"{self.base}/{path.lstrip('/')}"
        target, home = urllib.parse.urlparse(path), urllib.parse.urlparse(self.base)
        if (target.hostname or "").lower() != (home.hostname or "").lower() or (target.scheme != home.scheme and target.scheme != "https"):
            raise ApiError(f"refusing to send credentials to another host ({self._label(path) or target.hostname})", endpoint=self._label(path))
        return path

    def _backoff(self, attempt):
        delay = min(self.cfg.retry_max_delay_sec, self.cfg.retry_base_delay_sec * (2 ** (attempt - 1)))
        return delay + self.rng.uniform(0, delay * 0.25)

    def request(self, method, path, params=None, json_body=None, headers=None, buckets=("default",), ok=(200,), allow=(), timeout=120):
        """Send a request with throttling, retries and error classification. Returns the response for ok/allow statuses."""
        url = self.url(path)
        attempt, refreshed = 0, False
        while True:
            attempt += 1
            self.limiter.acquire(buckets)
            hdrs = {"Authorization": f"Bearer {self.tokens.token(self.audience)}", "User-Agent": self.user_agent,
                    "Accept": "application/json", **(headers or {})}
            started = self.clock.monotonic()
            try:
                resp = self.session.request(method, url, params=params, json=json_body, headers=hdrs, timeout=timeout)
            except requests.RequestException as exc:
                self.stats.record(buckets[0], None, self.clock.monotonic() - started, retried=attempt > 1)
                if attempt <= self.cfg.max_retries:
                    self.clock.sleep(self._backoff(attempt))
                    continue
                raise ApiError(f"{method} {self._label(path)} failed: {type(exc).__name__}", retryable=True, endpoint=self._label(path)) from None
            status = resp.status_code
            self.stats.record(buckets[0], status, self.clock.monotonic() - started, retried=attempt > 1)
            if status in ok or status in allow:
                return resp
            if status == 401 and not refreshed:
                refreshed = True
                self.tokens.invalidate(self.audience)
                continue
            if status == 429:
                wait = parse_retry_after(resp, default=self._backoff(attempt))
                self.limiter.penalize(buckets, wait)
                if wait > self.cfg.max_retry_after_sec or attempt > self.cfg.max_retries:
                    raise ThrottleExhausted(f"{self._label(path)} throttled (Retry-After {int(wait)}s)", status=429, endpoint=self._label(path), retryable=True)
                self.logger.warning("429 on %s - waiting %ds (attempt %d/%d)", buckets[0], int(wait), attempt, self.cfg.max_retries)
                self.clock.sleep(wait + self.rng.uniform(0, 1.0))
                continue
            if status in self.RETRYABLE and attempt <= self.cfg.max_retries:
                self.clock.sleep(self._backoff(attempt))
                continue
            raise self._classify(resp, method, path)

    def _label(self, path):
        text = re.sub(r"https?://[^/]+", "", path)
        text = re.sub(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", "{id}", text)
        return text.split("?")[0]

    def _classify(self, resp, method, path):
        code, message = error_details(resp)
        status = resp.status_code
        text = f"{method} {self._label(path)} -> HTTP {status} {code or ''} {message or ''}".strip()
        kwargs = dict(status=status, code=code, url=self._label(path), body=(resp.text or "")[:1000], endpoint=self._label(path),
                      retryable=status in self.RETRYABLE)
        if status == 401:
            return AuthError(text, **kwargs)
        if status == 403:
            return ForbiddenError(text, **kwargs)
        if status == 404:
            return NotFoundError(text, **kwargs)
        return ApiError(text, **kwargs)

    def get_json(self, path, params=None, buckets=("default",), **kw):
        resp = self.request("GET", path, params=params, buckets=buckets, **kw)
        return resp.json() if resp.content else {}

    def post_json(self, path, body=None, params=None, buckets=("default",), ok=(200,), **kw):
        resp = self.request("POST", path, params=params, json_body=body if body is not None else {}, buckets=buckets, ok=ok, **kw)
        return resp.json() if resp.content else {}

    def paginate(self, path, params=None, items_key=None, style="fabric", buckets=("default",), page_size=5000, max_pages=100000, **kw):
        """Yield items across pages.

        style='fabric' follows continuationUri / continuationToken; style='skip' uses $top/$skip (Power BI admin APIs).
        The items key is auto-detected (value, itemEntities, workspaces, ...) unless given.
        """
        params = dict(params or {})
        url, pages, seen = path, 0, set()
        previous_first = None
        if style == "skip":
            params.setdefault("$top", page_size)
            params.setdefault("$skip", 0)
        while pages < max_pages:
            pages += 1
            body = self.get_json(url, params=dict(params), buckets=buckets, **kw)
            items = self._items(body, items_key)
            if style == "skip":
                first_key = json.dumps(items[0], sort_keys=True, default=str) if items else None
                if not items or first_key == previous_first:
                    return
                previous_first = first_key
            for item in items:
                yield item
            if style == "skip":
                params["$skip"] = int(params["$skip"]) + len(items)
                continue
            next_uri, token = body.get("continuationUri"), body.get("continuationToken")
            cursor = next_uri or token
            if not cursor or cursor in seen:
                return
            seen.add(cursor)
            if next_uri:
                url, params = next_uri, {}
            else:
                params["continuationToken"] = token

    @staticmethod
    def _items(body, items_key):
        if isinstance(body, list):
            return body
        if items_key:
            return body.get(items_key) or []
        for key in ("value", "itemEntities", "workspaces", "domains", "tags", "capacities", "connections", "gateways",
                    "activityEventEntities", "accessDetails", "items"):
            if isinstance(body.get(key), list):
                return body[key]
        lists = [v for v in body.values() if isinstance(v, list) and (not v or isinstance(v[0], dict))]
        return lists[0] if len(lists) == 1 else []

    def lro(self, resp, timeout_sec=600, require_result=True):
        """Complete a Fabric long-running operation (202 + Location / x-ms-operation-id) and return its result body.

        `Retry-After` is honored (the next poll is never earlier than the service asked). `timeout_sec` bounds the time spent *waiting*: no sleep goes past it and
        a zero timeout never polls; the poll and result requests keep their own HTTP timeout and retry policy, so the call can end one request after the deadline.
        A succeeded operation whose result cannot be read is an error, unless the caller expects an operation without result."""
        if resp.status_code != 202:
            return resp.json() if resp.content else {}
        op_id = resp.headers.get("x-ms-operation-id")
        status_url = resp.headers.get("Location") or f"{self.base}/operations/{op_id}"
        delay = parse_retry_after(resp, default=5.0)
        deadline = self.clock.monotonic() + timeout_sec
        while True:
            remaining = deadline - self.clock.monotonic()
            if remaining <= 0:
                raise ApiError("long-running operation timed out", endpoint="operations", retryable=True)
            self.clock.sleep(min(max(delay, 1.0), remaining))
            poll = self.request("GET", status_url, buckets=("fabric_lro",))
            body = poll.json() if poll.content else {}
            state = body.get("status")
            if state == "Succeeded":
                result_url = poll.headers.get("Location") or status_url.rstrip("/") + "/result"
                result = self.request("GET", result_url, buckets=("fabric_lro",), allow=(404, 204))
                if result.status_code == 200 and result.content:
                    return result.json()
                if require_result:
                    label = f"operation {op_id}" if op_id else "operation"
                    raise ApiError(f"{label} succeeded but its result is not available (HTTP {result.status_code})", status=result.status_code,
                                   endpoint="operations", retryable=True)
                return {}
            if state in ("Failed", "Cancelled"):
                err = body.get("error") or {}
                raise ApiError(f"operation {state}: {err.get('errorCode', '')} {err.get('message', '')}".strip(), code=err.get("errorCode"), endpoint="operations")
            delay = parse_retry_after(poll, default=5.0)

In [ ]:
def build_clients(ctx, tokens, session=None):
    cfg = ctx.cfg
    limiter = RateLimiter(build_rate_limits(cfg), ctx.clock, scale=cfg.rate_limit_scale, max_wait=cfg.max_retry_after_sec)
    http = session or ThreadSessions()
    make = lambda name, base, audience: ApiClient(name, base, audience, tokens, limiter, ctx.stats, ctx.clock, cfg, http, ctx.logger)  # noqa: E731
    ctx.clients = {"pbi": make("pbi", cfg.pbi_api_base, "pbi"), "fabric": make("fabric", cfg.fabric_api_base, "fabric"),
                   "graph": make("graph", cfg.graph_api_base, "graph")}
    ctx.limiter = limiter
    return ctx.clients


def preflight(ctx):
    """Probe the API families and record what is accessible; later phases degrade gracefully on denied families."""
    probes = {
        "pbi_admin": ("pbi", "admin/capacities", ("pbi_admin_capacities",)),
        "fabric_admin": ("fabric", "admin/tags", ("fabric_admin_tags", "fabric_platform")),
        "fabric_core": ("fabric", "capacities", ("fabric_platform",)),
    }
    access = {}
    for key, (client, path, buckets) in probes.items():
        try:
            ctx.clients[client].request("GET", path, buckets=buckets)
            access[key] = "OK"
        except ThrottleExhausted:
            raise
        except ApiError as exc:
            access[key] = f"Denied ({exc.status})" if exc.status in (401, 403) else f"Error ({exc.status})"
            ctx.record_error("preflight", "Api", key, exc.endpoint, exc=exc)
    ctx.state["access"] = access
    hints = {
        "pbi_admin": "enable 'Service principals can access read-only admin APIs' for a security group containing the SPN; the app must have no admin-consent Power BI permissions",
        "fabric_admin": "enable 'Service principals can access read-only admin APIs' (Fabric admin APIs also need the Fabric administrator role for users)",
        "fabric_core": "enable 'Service principals can call Fabric public APIs'",
    }
    for key, state in access.items():
        (ctx.logger.info if state == "OK" else ctx.logger.warning)("access %-12s %s%s", key, state, "" if state == "OK" else f"  -> {hints[key]}")
    return {"rows_out": len(access), "message": ", ".join(f"{k}={v}" for k, v in access.items())}

### Run - sections 1 and 2: configuration and authentication
Validates the parameters, wires logging, storage and identity, then probes the API families (`preflight`, `access_check`).

In [ ]:
ctx = start_run(globals())
run_steps(ctx, sections={2})

# 3. Workspace discovery
| Step | API | Why |
|---|---|---|
| Governance overrides | CSV in `Files/dea/config` (+ table) | Business owner, criticality, support team, environment, defer / pin-wave, observed wizard status |
| Capacities | Power BI `admin/capacities` (fallback Fabric `capacities`) | Name, SKU, Fabric capability. The list can be incomplete - unresolved capacities get an explicit bucket |
| Workspaces | Fabric `admin/workspaces` + Power BI `admin/groups` | State, capacity, domain, tags + attached-storage (BYOSA) and deployment-pipeline ids |
| Domains / environments | Fabric `admin/domains`, Power BI `admin/pipelines` | Classification and Dev / Test / Prod detection (pipeline stage, then naming convention) |
| Scanner | `admin/workspaces/modified` -> `getInfo` -> `scanStatus` -> `scanResult` | Bulk metadata and lineage for 100 workspaces per call; incremental through `modifiedSince` |

Notes: the Scanner is limited to 500 `getInfo` calls/hour and **16 concurrent scans**; scan results expire after 24 h;
`modifiedSince` must be between 30 minutes and 30 days. The watermark only advances after a *complete* scan.

In [ ]:
_CRITICALITY = {"high": "High", "h": "High", "critical": "High", "critique": "High", "élevée": "High", "elevee": "High", "élevé": "High",
                "eleve": "High", "haute": "High", "medium": "Medium", "m": "Medium", "moyenne": "Medium", "moyen": "Medium",
                "low": "Low", "l": "Low", "faible": "Low", "basse": "Low"}
_ENVIRONMENT = {"dev": "Dev", "development": "Dev", "developpement": "Dev", "développement": "Dev", "test": "Test", "tst": "Test",
                "uat": "Test", "qa": "Test", "recette": "Test", "prod": "Prod", "prd": "Prod", "production": "Prod"}
_WIZARD = {"ready": "Ready to migrate", "ready to migrate": "Ready to migrate", "needs attention": "Needs Attention",
           "needs_attention": "Needs Attention", "upgrade unavailable": "Upgrade unavailable", "unavailable": "Upgrade unavailable",
           "not supported": "Upgrade unavailable", "unsupported": "Upgrade unavailable"}


def norm_criticality(value):
    return _CRITICALITY.get((value or "").strip().lower())


def norm_environment(value):
    return _ENVIRONMENT.get((value or "").strip().lower())


def norm_wizard_status(value):
    return _WIZARD.get((value or "").strip().lower())


def parse_governance_csv(text):
    """Parse and validate the governance CSV (English or French labels). Returns (rows, problems)."""
    rows, problems = [], []
    for line_no, raw in enumerate(csv.DictReader(io.StringIO(text.lstrip("\ufeff"))), start=2):
        rec = {(k or "").strip().lower(): (v or "").strip() for k, v in raw.items()}
        ws, df = gid(rec.get("workspace_id")), gid(rec.get("dataflow_id"))
        if not ws and not df:
            problems.append(f"line {line_no}: workspace_id or dataflow_id is required")
            continue
        crit = norm_criticality(rec.get("business_criticality"))
        env = norm_environment(rec.get("environment"))
        wizard = norm_wizard_status(rec.get("wizard_status_actual"))
        for field, supplied, parsed in (("business_criticality", rec.get("business_criticality"), crit),
                                        ("environment", rec.get("environment"), env),
                                        ("wizard_status_actual", rec.get("wizard_status_actual"), wizard)):
            if supplied and not parsed:
                problems.append(f"line {line_no}: unknown {field} {supplied!r} ignored")
        rows.append({
            "scope": "Dataflow" if df else "Workspace", "workspace_id": ws or "", "dataflow_id": df or "",
            "business_owner": rec.get("business_owner") or None, "technical_owner": rec.get("technical_owner") or None,
            "support_team": rec.get("support_team") or None, "business_criticality": crit,
            "workspace_classification": rec.get("workspace_classification") or None, "environment": env,
            "defer": _as_bool(rec.get("defer")) if rec.get("defer") else None, "defer_reason": rec.get("defer_reason") or None,
            "pin_wave": coerce_value("INT", rec.get("pin_wave")) if rec.get("pin_wave") else None, "wizard_status_actual": wizard,
            "rollback_plan_defined": _as_bool(rec.get("rollback_plan_defined")) if rec.get("rollback_plan_defined") else None,
            "uat_window_start": coerce_value("DATE", rec.get("uat_window_start")), "uat_window_end": coerce_value("DATE", rec.get("uat_window_end")),
            "notes": rec.get("notes") or None, "updated_by": rec.get("updated_by") or None,
            "updated_at": coerce_value("TIMESTAMP", rec.get("updated_at")) or utcnow(),
        })
    return rows, problems


def governance_index(rows):
    """Index overrides by dataflow and by workspace. Several rows can address the same target (the table is keyed by scope, workspace id and
    dataflow id): the most recently updated wins and exact ties are broken by content, so the order of a Delta read never changes the outcome."""
    def precedence(row):
        return (row.get("updated_at") or dt.datetime.min, json.dumps(row, sort_keys=True, default=str))

    index = {"workspace": {}, "dataflow": {}}
    for row in sorted(rows, key=precedence):
        if row.get("scope") == "Dataflow" and row.get("dataflow_id"):
            index["dataflow"][gid(row["dataflow_id"])] = row
        elif row.get("workspace_id"):
            index["workspace"][gid(row["workspace_id"])] = row
    return index


def phase_governance(ctx):
    """Merge the optional governance CSV into Assessment_GovernanceOverrides and index it for later phases."""
    cfg = ctx.cfg
    try:
        path = ctx.files.lakehouse_path(cfg.governance_overrides_path)
    except ValueError as exc:
        ctx.logger.warning("governance path ignored: %s", exc)
        path = None
    loaded = 0
    if path and os.path.isfile(path):
        with open(path, "r", encoding="utf-8-sig") as handle:
            rows, problems = parse_governance_csv(handle.read())
        for problem in problems:
            ctx.logger.warning("governance CSV: %s", problem)
        if rows:
            ctx.lake.merge("Assessment_GovernanceOverrides", rows)
            loaded = len(rows)
    else:
        ctx.logger.info("no governance CSV at %s - using the table content only", cfg.governance_overrides_path)
    table_rows = ctx.lake.read("Assessment_GovernanceOverrides")
    ctx.state["governance"] = governance_index(table_rows)
    return {"rows_in": loaded, "rows_out": len(table_rows), "message": f"{loaded} rows loaded from CSV, {len(table_rows)} active overrides"}

In [ ]:
def phase_capacities(ctx):
    cfg, pbi, fabric = ctx.cfg, ctx.clients["pbi"], ctx.clients["fabric"]
    source = "admin"
    try:
        items = list(pbi.paginate("admin/capacities", buckets=("pbi_admin_capacities",), items_key="value"))
    except (ForbiddenError, AuthError) as exc:
        ctx.logger.warning("admin/capacities denied (%s): using Fabric capacities, which only lists capacities this identity can access", exc.status)
        items, source = list(fabric.paginate("capacities", buckets=("fabric_platform",), items_key="value")), "core"
    rows = {}
    for c in items:
        cid = gid(c.get("id"))
        if not cid:
            continue
        kind, capable = classify_sku(c.get("sku"))
        rows[cid] = {"capacity_id": cid, "capacity_name": c.get("displayName"), "sku": c.get("sku"), "capacity_kind": kind,
                     "is_fabric_capable": capable, "region": c.get("region"), "state": c.get("state"), "admins": c.get("admins") or [],
                     "source": source}
    ctx.state["capacities"] = rows
    ctx.state["source_capacity_id"] = None
    if cfg.source_capacity:
        wanted = cfg.source_capacity.strip().lower()
        match = [cid for cid, r in rows.items() if cid == wanted or (r["capacity_name"] or "").lower() == wanted]
        if not match:
            raise ConfigError(f"capacity {cfg.source_capacity!r} not found among {len(rows)} visible capacities")
        ctx.state["source_capacity_id"] = match[0]
        ctx.logger.info("scope limited to capacity %s (%s)", rows[match[0]]["capacity_name"], match[0])
    ctx.lake.merge("Assessment_Capacities", list(rows.values()))
    return {"rows_in": len(items), "rows_out": len(rows), "message": f"source={source}"}

In [ ]:
def _stage_info(pipelines):
    """workspace id -> (pipeline id, stage order, stage count) from admin/pipelines?$expand=stages."""
    info = {}
    for pipe in pipelines:
        stages = pipe.get("stages") or []
        for stage in stages:
            wid = gid(stage.get("workspaceId"))
            if wid:
                info[wid] = (gid(pipe.get("id")), stage.get("order"), len(stages))
    return info


def resolve_workspace_attributes(name, pipeline_stage, override, domain_name, patterns):
    """Environment (override > pipeline stage > naming) and classification (override > domain)."""
    env, env_source = "Unknown", "Unknown"
    if override and override.get("environment"):
        env, env_source = override["environment"], "Override"
    elif pipeline_stage and stage_environment(pipeline_stage[1], pipeline_stage[2]) != "Unknown":
        env, env_source = stage_environment(pipeline_stage[1], pipeline_stage[2]), "DeploymentPipeline"
    else:
        guessed = detect_environment(name, patterns)
        if guessed != "Unknown":
            env, env_source = guessed, "NamingConvention"
    classification = (override or {}).get("workspace_classification") or domain_name or "Unclassified"
    return env, env_source, classification


def build_workspace_row(fab, grp, capacities, domains, stages, governance, patterns):
    wid = gid(fab.get("id") or grp.get("id"))
    name = fab.get("name") or grp.get("name")
    wtype = fab.get("type") or grp.get("type")
    cap_id = gid(fab.get("capacityId") or grp.get("capacityId"))
    cap = capacities.get(cap_id) if cap_id else None
    if not cap_id:
        resolution, capable = "Shared", False
    elif cap:
        resolution, capable = "Resolved", cap.get("is_fabric_capable")
    else:
        resolution, capable = "Unresolved", None
    dedicated = grp.get("isOnDedicatedCapacity")
    domain_id = gid(fab.get("domainId"))
    domain_name = domains.get(domain_id)
    override = governance["workspace"].get(wid)
    stage = stages.get(wid)
    env, env_source, classification = resolve_workspace_attributes(name, stage, override, domain_name, patterns)
    tags = [t.get("displayName") if isinstance(t, dict) else str(t) for t in (fab.get("tags") or [])]
    return {
        "workspace_key": stable_hash64(wid), "workspace_id": wid, "workspace_name": name, "workspace_type": wtype,
        "state": fab.get("state") or grp.get("state"), "is_personal": str(wtype).lower() in ("personal", "personalgroup"),
        "description": grp.get("description"), "capacity_id": cap_id, "capacity_name": (cap or {}).get("capacity_name"),
        "capacity_sku": (cap or {}).get("sku"), "capacity_resolution": resolution,
        "is_on_dedicated_capacity": bool(cap_id) if dedicated is None else bool(dedicated), "is_fabric_capable": capable,
        "domain_id": domain_id, "domain_name": domain_name, "default_dataset_storage_format": grp.get("defaultDatasetStorageFormat"),
        "has_workspace_level_settings": grp.get("hasWorkspaceLevelSettings"), "dataflow_storage_id": grp.get("dataflowStorageId"),
        "pipeline_id": gid(grp.get("pipelineId")) or (stage[0] if stage else None), "pipeline_stage_order": stage[1] if stage else None,
        "environment": env, "environment_source": env_source, "workspace_classification": classification,
        "support_team": (override or {}).get("support_team"), "business_criticality": (override or {}).get("business_criticality"),
        "tags": [t for t in tags if t],
    }


def make_scope_filter(cfg, source_capacity_id):
    only = {x.strip().lower() for x in cfg.only_workspaces}
    excluded = {x.strip().lower() for x in cfg.exclude_workspaces}

    def keep(row):
        name, wid = (row.get("workspace_name") or "").lower(), row["workspace_id"]
        if only and name not in only and wid not in only:
            return False
        if name in excluded or wid in excluded:
            return False
        if source_capacity_id and row.get("capacity_id") != source_capacity_id:
            return False
        if row.get("is_personal") and not cfg.include_personal_workspaces:
            return False
        if row.get("state") != "Active" and not cfg.include_inactive_workspaces:
            return False
        return True

    return keep


def listing_excludes(cfg, row):
    """True when the configured listing cannot contain this stored workspace (personal and non-active ones are left out unless asked
    for): its absence says nothing about the tenant, so it must not be taken for a deletion."""
    if row.get("is_personal") and not cfg.include_personal_workspaces:
        return True
    return row.get("state") not in (None, "Active") and not cfg.include_inactive_workspaces


def phase_workspaces(ctx):
    cfg, pbi, fabric = ctx.cfg, ctx.clients["pbi"], ctx.clients["fabric"]
    fabric_ws, groups, fabric_ok, groups_ok = {}, {}, True, True
    kinds = ["Workspace"] + (["Personal"] if cfg.include_personal_workspaces else [])
    states = ["Active"] + (["Deleted"] if cfg.include_inactive_workspaces else [])
    try:
        for kind in kinds:
            for state in states:
                for w in fabric.paginate("admin/workspaces", params={"type": kind, "state": state}, items_key="workspaces",
                                         buckets=("fabric_admin_workspaces", "fabric_platform")):
                    fabric_ws[gid(w.get("id"))] = w
    except (ForbiddenError, AuthError) as exc:
        fabric_ok = False
        ctx.logger.warning("Fabric admin/workspaces denied (%s): relying on Power BI admin/groups only (no domain or tags)", exc.status)
    try:
        for g in pbi.paginate("admin/groups", style="skip", page_size=5000, buckets=("pbi_admin_groups",)):
            if g.get("state") in (None, "Active") or cfg.include_inactive_workspaces:
                groups[gid(g.get("id"))] = g
    except (ForbiddenError, AuthError) as exc:
        groups_ok = False
        ctx.logger.warning("Power BI admin/groups denied (%s): attached-storage and pipeline info unavailable", exc.status)
    if not fabric_ok and not groups_ok:
        raise ApiError("no workspace inventory API is accessible: check the tenant settings and the service principal group", status=403)

    domains = {}
    if fabric_ok:
        try:
            domains = {gid(d.get("id")): d.get("displayName") for d in
                       fabric.paginate("admin/domains", params={"preview": "false"}, items_key="domains", buckets=("fabric_admin_domains", "fabric_platform"))}
        except ApiError as exc:
            ctx.logger.warning("domains unavailable (%s)", exc.status)
    stages = {}
    try:
        stages = _stage_info(pbi.paginate("admin/pipelines", params={"$expand": "stages"}, style="skip", page_size=5000, buckets=("pbi_admin_pipelines",)))
    except ApiError as exc:
        ctx.logger.warning("deployment pipelines unavailable (%s): environment falls back to naming conventions", exc.status)

    patterns = {**DEFAULT_ENV_PATTERNS, **cfg.environment_patterns}
    governance = ctx.state.get("governance") or {"workspace": {}, "dataflow": {}}
    rows = {}
    for wid in set(fabric_ws) | set(groups):
        if wid:
            rows[wid] = build_workspace_row(fabric_ws.get(wid, {}), groups.get(wid, {}), ctx.state["capacities"], domains, stages, governance, patterns)
    known = ctx.lake.read("Assessment_Workspaces", include_deleted=True)
    live = [r for r in known if not r.get("is_deleted")]
    if not rows and live:
        raise ApiError(f"both workspace listings came back empty although {len(live)} workspaces are known: not taken for a tenant-wide deletion", status=200,
                       endpoint="admin/workspaces", retryable=True)
    keep = make_scope_filter(cfg, ctx.state.get("source_capacity_id"))
    scope = {wid for wid, r in rows.items() if keep(r)}
    ctx.state.update(workspaces=rows, scope_ws_ids=scope, workspace_listing_complete=fabric_ok or groups_ok)

    # a workspace that was flagged deleted and is listed again is revived by the merge: in the same write, what was stored for it is
    # reset, so that it is scanned again (nobody needs to have modified it) and its sources and lineage come back with it
    was_deleted = {r["workspace_id"] for r in known if r.get("is_deleted")}
    ctx.lake.merge("Assessment_Workspaces", [r for w, r in rows.items() if w not in was_deleted])
    if was_deleted & set(rows):
        ctx.lake.merge("Assessment_Workspaces", [{**r, "scan_state": "NotScanned"} for w, r in rows.items() if w in was_deleted])
    active_ids = {wid for wid, r in rows.items() if r["state"] == "Active"}
    gone = [{"workspace_id": r["workspace_id"]} for r in live if r["workspace_id"] not in rows and not listing_excludes(cfg, r)]
    if gone:
        ctx.lake.mark_deleted("Assessment_Workspaces", gone)
    unresolved = sum(1 for r in rows.values() if r["capacity_resolution"] == "Unresolved")
    if unresolved:
        ctx.logger.warning("%d workspaces reference a capacity that no capacity API returned (kept as Unresolved)", unresolved)
    by_capacity = Counter(r["capacity_id"] for r in rows.values() if r["capacity_id"])
    caps = ctx.state["capacities"]
    for cid, count in by_capacity.items():
        row = caps.setdefault(cid, {"capacity_id": cid, "source": "derived", "capacity_kind": "Unknown", "admins": []})
        row["workspace_count"] = count
    ctx.lake.merge("Assessment_Capacities", [{"capacity_id": cid, "workspace_count": by_capacity.get(cid, 0), "source": r.get("source")}
                                             for cid, r in caps.items()], update_cols=["workspace_count", "source"])
    return {"rows_in": len(rows), "rows_out": len(scope), "message": f"{len(active_ids)} active, {len(scope)} in scope, {unresolved} unresolved capacity"}

In [ ]:
SCAN_STATUS_DONE, SCAN_STATUS_FAILED = "Succeeded", "Failed"
MAX_SCAN_SPLITS = 40   # a failed batch is retried as two halves so that one bad workspace cannot sink its batch-mates; the budget bounds the extra calls


class ScanBatchError(ApiError):
    """The Scanner itself gave up on a batch (status Failed, or no result in time): usually a single workspace is the cause."""


def _iso_z(moment):
    return moment.strftime("%Y-%m-%dT%H:%M:%S.000Z")


def scan_fingerprint(ctx):
    """What a watermark covers: the scope and the Scanner flags. A watermark is only reused for an identical fingerprint,
    so a narrower run can never hide changes from a later, wider one."""
    cfg = ctx.cfg
    parts = {"only": sorted(x.strip().lower() for x in cfg.only_workspaces), "exclude": sorted(x.strip().lower() for x in cfg.exclude_workspaces),
             "capacity": ctx.state.get("source_capacity_id") or "", "personal": bool(cfg.include_personal_workspaces),
             "inactive": bool(cfg.include_inactive_workspaces), "users": bool(cfg.scan_get_users), "expressions": bool(cfg.scan_dataset_expressions)}
    return short_hash(json.dumps(parts, sort_keys=True), 12)


def mark_scan_state(ctx, workspace_ids, state, strict=False):
    """Record scan_state, so that the next run picks the workspaces up again. With strict=True a storage failure stops the caller:
    a plan must never go on, or certify anything, on states it could not write."""
    if not workspace_ids:
        return
    try:
        ctx.lake.merge("Assessment_Workspaces", [{"workspace_id": w, "scan_state": state} for w in workspace_ids])
    except Exception as exc:  # noqa: BLE001
        if strict:
            raise
        ctx.logger.error("could not record scan_state=%s for %d workspaces: %s: %s", state, len(workspace_ids), type(exc).__name__, exc)


def plan_scan(ctx, scope_ids, scan_started):
    """Return (workspace ids to scan, plan label).

    Incremental runs scan modified + never-scanned/failed workspaces. The watermark is the last *complete* scan of the same
    scope and Scanner flags. A different scope or other flags restart the scan from scratch, tracked per workspace (scan_state)
    and by a `Reset` checkpoint that names what it restarts for, so that the restart itself can be resumed - and is itself
    restarted when the configuration changes again. Without a usable watermark the oldest per-workspace scan time is used, so a
    restarted run does not repeat the batches that already succeeded.

    scan_state = Scanned means "current as of the watermark": what is stored for a workspace outside the scope is not, so it is
    reset, and the workspace is scanned again when it comes back.
    """
    cfg, pbi = ctx.cfg, ctx.clients["pbi"]
    ids = sorted(scope_ids)
    stored = {r["workspace_id"]: r for r in ctx.lake.read("Assessment_Workspaces")}
    uncovered = sorted(w for w, r in stored.items() if r.get("scan_state") == "Scanned" and w not in scope_ids)
    if uncovered:
        mark_scan_state(ctx, uncovered, "NotScanned", strict=True)
        stored.update({w: {**stored[w], "scan_state": "NotScanned"} for w in uncovered})
    watermark = None
    if cfg.run_mode != "full":
        fingerprint = scan_fingerprint(ctx)
        cp = ctx.checkpoints.get("Tenant", "scanner", "scan")
        stamp, _, recorded = str((cp or {}).get("source_version") or "").partition("|")
        if cp and cp.get("status") in ("Done", "Reset") and recorded != fingerprint:
            # the states first, then the checkpoint: when the states cannot be written the old checkpoint stays and the reset is retried
            mark_scan_state(ctx, ids, "NotScanned", strict=True)
            ctx.checkpoints.mark("Tenant", "scanner", "scan", "Reset", version=f"|{fingerprint}")
            ctx.checkpoints.flush()
            return ids, "full (the scope or the Scanner flags changed since the last complete scan)"
        if cp and cp.get("status") == "Done":
            watermark = parse_ts(stamp)
        if watermark is None:
            times = [r["scanned_at"] for w, r in stored.items() if w in scope_ids and r.get("scan_state") == "Scanned" and r.get("scanned_at")]
            watermark = min(times) if times else None
    if watermark is None or scan_started - watermark > dt.timedelta(days=cfg.scan_lookback_days):
        return ids, "full"
    since = min(watermark - dt.timedelta(hours=1), scan_started - dt.timedelta(minutes=31))
    since = max(since, scan_started - dt.timedelta(days=cfg.scan_lookback_days))
    params = {"modifiedSince": _iso_z(since), "excludePersonalWorkspaces": str(not cfg.include_personal_workspaces).lower()}
    if cfg.include_inactive_workspaces:
        params["excludeInActiveWorkspaces"] = "false"   # otherwise the service default applies and the scope filter drops non-active workspaces
    modified = pbi.get_json("admin/workspaces/modified", params=params, buckets=("pbi_scan_modified",))
    modified_ids = {gid(w.get("id")) for w in (modified if isinstance(modified, list) else modified.get("value", []))}
    pending = {wid for wid in scope_ids if (stored.get(wid) or {}).get("scan_state") != "Scanned"}
    chosen = sorted((modified_ids & set(scope_ids)) | pending)
    return chosen, f"incremental ({len(modified_ids & set(scope_ids))} modified, {len(pending)} unscanned or failed)"


def scan_one_batch(ctx, batch):
    """Worker: getInfo -> poll -> result. Pure API work, safe to run in threads."""
    cfg, pbi = ctx.cfg, ctx.clients["pbi"]
    flags = {"lineage": "true", "datasourceDetails": "true", "getArtifactUsers": str(cfg.scan_get_users).lower(),
             "datasetSchema": "false", "datasetExpressions": str(cfg.scan_dataset_expressions).lower()}
    started = pbi.post_json("admin/workspaces/getInfo", {"workspaces": batch}, params=flags, buckets=("pbi_scan_getinfo",), ok=(200, 202))
    scan_id = started["id"]
    give_up = ctx.clock.monotonic() + cfg.scan_timeout_sec
    while True:
        status = pbi.get_json(f"admin/workspaces/scanStatus/{scan_id}", buckets=("pbi_scan_status",)).get("status")
        if status == SCAN_STATUS_DONE:
            break
        if status == SCAN_STATUS_FAILED:
            raise ScanBatchError(f"scan {scan_id} failed", endpoint="admin/workspaces/scanStatus")
        if ctx.clock.monotonic() > give_up:
            raise ScanBatchError(f"scan {scan_id} did not finish within {cfg.scan_timeout_sec}s", endpoint="admin/workspaces/scanStatus", retryable=True)
        ctx.clock.sleep(cfg.scan_poll_interval_sec)
    return pbi.get_json(f"admin/workspaces/scanResult/{scan_id}", buckets=("pbi_scan_result",))


def phase_scan(ctx):
    cfg = ctx.cfg
    scope = ctx.state["scope_ws_ids"]
    scan_started = ctx.clock.now()
    ids, plan = plan_scan(ctx, scope, scan_started)
    ctx.logger.info("scanner plan: %d of %d workspaces (%s)", len(ids), len(scope), plan)
    batches = list(chunked(ids, cfg.scan_batch_size))
    ctx.state.setdefault("scanned_ws_ids", set())
    states = {wid: r.get("state") for wid, r in ctx.state["workspaces"].items()}
    failed_batches = missing_active = splits = results = 0
    stop, issued = None, len(batches)
    pending = deque(enumerate(batches, start=1))
    with cf.ThreadPoolExecutor(max_workers=cfg.scan_max_concurrent) as pool:
        inflight = {}
        while pending or inflight:
            while pending and len(inflight) < cfg.scan_max_concurrent and stop is None and not ctx.deadline.expired(reserve_sec=120):
                number, batch = pending.popleft()
                inflight[pool.submit(scan_one_batch, ctx, batch)] = (number, batch)
            if not inflight:
                break
            done, _ = cf.wait(list(inflight), return_when=cf.FIRST_COMPLETED)
            for future in done:
                number, batch = inflight.pop(future)
                try:
                    result = future.result()
                except GracefulStop as exc:
                    stop = exc
                    pending.appendleft((number, batch))
                    continue
                except Exception as exc:  # noqa: BLE001 - one bad batch must not lose the others
                    ctx.record_error("scan", "ScanBatch", f"batch-{number}", getattr(exc, "endpoint", None), exc=exc,
                                     message=str(exc) if len(batch) > 1 else f"{exc} (workspace {batch[0]})")
                    if isinstance(exc, ScanBatchError) and len(batch) > 1 and splits < MAX_SCAN_SPLITS:
                        splits += 1
                        half = (len(batch) + 1) // 2
                        pending.appendleft((issued + 2, batch[half:]))
                        pending.appendleft((issued + 1, batch[:half]))
                        issued += 2
                        ctx.logger.warning("scan batch %d failed (%s): retrying its %d workspaces as two smaller batches", number, exc, len(batch))
                    else:
                        failed_batches += 1
                        mark_scan_state(ctx, batch, "Failed")
                    continue
                try:
                    if cfg.persist_raw:
                        ctx.files.write_json(f"raw/{ctx.run_id}/scanner/batch-{number:04d}.json.gz", result)
                    parsed = parse_scan_result(ctx, result, batch)
                    persist_scan(ctx, parsed)
                except Exception as exc:  # noqa: BLE001
                    failed_batches += 1
                    ctx.record_error("scan", "ScanBatch", f"batch-{number}", "scanResult", exc=exc)
                    ctx.logger.error("could not ingest scan batch %d: %s: %s\n%s", number, type(exc).__name__, exc, "".join(traceback.format_exception(exc, limit=-4)))
                    mark_scan_state(ctx, batch, "Failed")
                    continue
                results += len(parsed["dataflows"])
                ctx.state["scanned_ws_ids"].update(parsed["scanned_workspace_ids"])
                # a workspace the Scanner silently left out is only normal when it is no longer active
                absent = [w for w in parsed["missing_workspace_ids"] if states.get(w, "Active") == "Active"]
                mark_scan_state(ctx, absent, "Failed")
                mark_scan_state(ctx, [w for w in parsed["missing_workspace_ids"] if w not in absent], "NotScanned")
                missing_active += len(absent)
                for wid in absent:
                    ctx.record_error("scan", "Workspace", wid, "scanResult", message="the workspace was requested but is missing from the scan result")
    if stop is None and pending:
        stop = DeadlineExceeded(f"deadline reached with {len(pending)} of {issued} scan batches left")
    complete = not pending and failed_batches == 0 and missing_active == 0 and stop is None
    if complete:
        ctx.checkpoints.mark("Tenant", "scanner", "scan", "Done", version=f"{_iso_z(scan_started)}|{scan_fingerprint(ctx)}")
        ctx.checkpoints.flush()
    if stop is not None:
        raise stop
    if cfg.persist_raw and cfg.raw_retention_days:
        ctx.files.purge_older_than("raw", cfg.raw_retention_days)
    healthy = failed_batches == 0 and missing_active == 0
    return {"rows_in": len(ids), "rows_out": results, "status": "Succeeded" if healthy else "Partial",
            "message": f"{issued} batches ({splits} split after a failure), {failed_batches} failed, {missing_active} workspaces missing from the results, plan={plan}"}

### Run - section 3: workspace discovery

In [ ]:
run_steps(ctx, sections={3})

# 4. Dataflow discovery
**Sources, in order of authority**
1. Scanner (`dataflows[]`): identity, owner, modified, endorsement, sensitivity label, schedule, tags, data source usages,
   `upstreamDataflows`, plus the semantic models / reports / dashboards / datamarts that give the **downstream** lineage.
2. Power BI `admin/dataflows` (tenant wide, 200 req/h): `modelUrl` (attached storage = BYOSA indicator) and a complete Gen1 id list.
3. Fabric `admin/items?type=Dataflow` (tenant wide, 10 000 per page): Gen2 items, creator, last update, tags.

**Generation resolution.** An explicit `generation` wins; otherwise an id known to the Power BI APIs is Gen1; an id seen
only by the Fabric items API is Gen2. A Gen1 -> Gen2 flip of the same id (in-place Upgrade Wizard) is recorded in
`type_changed_at` and feeds the migration burn-down. Items of deleted workspaces are ignored (the items API still reports them as active).
**Deletions** are only recorded when the tenant-wide lists were complete.

In [ ]:
def phase_admin_dataflows(ctx):
    """Tenant-wide Gen1 list: modelUrl (attached storage) and an authoritative id set independent of the Scanner scope."""
    pbi = ctx.clients["pbi"]
    disc = ctx.state.setdefault("discovery", {"scanner": {}, "admin_pbi": {}, "admin_items": {}, "complete": {}})
    rows, found = [], {}
    for d in pbi.paginate("admin/dataflows", style="skip", page_size=1000, buckets=("pbi_admin_dataflows",)):
        did, wid = gid(d.get("objectId")), gid(d.get("workspaceId"))
        if not did:
            continue
        generation = coerce_value("INT", d.get("generation"))
        found[did] = {"workspace_id": wid, "generation": generation, "name": d.get("name"), "description": d.get("description"),
                      "configured_by": d.get("configuredBy"), "model_url": d.get("modelUrl"), "storage_type": storage_type_of(d.get("modelUrl"))}
        row = {"dataflow_id": did, "workspace_id": wid, "model_url": d.get("modelUrl"), "storage_type": storage_type_of(d.get("modelUrl"))}
        if generation:
            row.update(generation=generation, dataflow_type="Gen2" if generation == 2 else "Gen1")
        rows.append(row)
    disc["admin_pbi"] = found
    disc["complete"]["admin_pbi"] = True
    known = {r["dataflow_id"] for r in ctx.lake.read("Assessment_Dataflows")}
    merge_by_signature(ctx.lake, "Assessment_Dataflows", [r for r in rows if r["dataflow_id"] in known], update_cols=["model_url", "storage_type"])
    return {"rows_in": len(rows), "rows_out": len(found), "message": f"{sum(1 for r in rows if r['storage_type'] == 'CustomADLS')} with attached ADLS storage"}


def phase_admin_items(ctx):
    """Tenant-wide Fabric items of type Dataflow (Gen2). Ghost items of deleted workspaces are dropped."""
    fabric = ctx.clients["fabric"]
    disc = ctx.state.setdefault("discovery", {"scanner": {}, "admin_pbi": {}, "admin_items": {}, "complete": {}})
    buckets = ("fabric_admin_items", "fabric_platform")
    try:
        items = list(fabric.paginate("admin/items", params={"type": "Dataflow"}, items_key="itemEntities", buckets=buckets))
    except ApiError as exc:
        if exc.status != 400:
            raise
        ctx.logger.warning("type filter rejected (%s): listing all items and filtering locally", exc.code)
        items = [i for i in fabric.paginate("admin/items", items_key="itemEntities", buckets=buckets) if i.get("type") == "Dataflow"]
    active = {wid for wid, r in ctx.state["workspaces"].items() if r.get("state") == "Active"} if ctx.state.get("workspaces") else None
    kept, ghosts = {}, 0
    for item in items:
        wid, did = gid(item.get("workspaceId")), gid(item.get("id"))
        if not did:
            continue
        if active is not None and wid not in active:
            ghosts += 1
            continue
        kept[did] = item
    disc["admin_items"] = kept
    disc["complete"]["admin_items"] = True
    return {"rows_in": len(items), "rows_out": len(kept), "message": f"{ghosts} items of deleted or unknown workspaces ignored"}


def resolve_generation(did, scanner, admin_pbi, items):
    explicit = (scanner.get(did) or {}).get("generation") or (admin_pbi.get(did) or {}).get("generation")
    if explicit == 2:
        return "Gen2", 2
    if explicit == 1 or did in scanner or did in admin_pbi:
        return "Gen1", 1
    if did in items:
        return "Gen2", 2
    return "Unknown", None


def cascade_workspace_deletions(ctx):
    """Dataflows, sources and lineage edges of workspaces that left the tenant listing are flagged as deleted with them.

    Derived from the Lakehouse (workspace rows already flagged is_deleted), so an interrupted run is repaired by the next one.
    """
    lake = ctx.lake
    dead = {r["workspace_id"] for r in lake.read("Assessment_Workspaces", include_deleted=True) if r.get("is_deleted")}
    if not dead:
        return {"dataflows": 0, "sources": 0, "edges": 0}
    flows = [{"dataflow_id": r["dataflow_id"]} for r in lake.read("Assessment_Dataflows", where={"workspace_id": dead})]
    sources = [{"dataflow_id": r["dataflow_id"], "source_key": r["source_key"]} for r in lake.read("Assessment_DataflowSources", where={"workspace_id": dead})]
    edge_ids = {r["edge_id"] for column in ("src_workspace_id", "dst_workspace_id") for r in lake.read("Assessment_Lineage", where={column: dead})}
    for table, keys in (("Assessment_Dataflows", flows), ("Assessment_DataflowSources", sources), ("Assessment_Lineage", [{"edge_id": e} for e in sorted(edge_ids)])):
        if keys:
            lake.mark_deleted(table, keys)
    if flows or sources or edge_ids:
        ctx.logger.info("%d deleted workspaces: %d dataflows, %d sources and %d lineage edges flagged as deleted with them", len(dead), len(flows), len(sources), len(edge_ids))
    return {"dataflows": len(flows), "sources": len(sources), "edges": len(edge_ids)}


def phase_reconcile_dataflows(ctx):
    cfg, lake, disc = ctx.cfg, ctx.lake, ctx.state.get("discovery") or {}
    scanner, admin_pbi, items = disc.get("scanner", {}), disc.get("admin_pbi", {}), disc.get("admin_items", {})
    existing = {r["dataflow_id"]: r for r in lake.read("Assessment_Dataflows")}
    in_scope = ctx.state.get("scope_ws_ids")
    rows = []
    for did in set(scanner) | set(admin_pbi) | set(items):
        wid = (scanner.get(did) or admin_pbi.get(did) or {}).get("workspace_id") or gid((items.get(did) or {}).get("workspaceId"))
        if in_scope is not None and wid not in in_scope:
            continue
        dtype, generation = resolve_generation(did, scanner, admin_pbi, items)
        prev = existing.get(did) or {}
        sources = [s for s, present in (("Scanner", did in scanner), ("AdminDataflows", did in admin_pbi), ("AdminItems", did in items)) if present]
        row = {"dataflow_id": did, "workspace_id": wid, "dataflow_type": dtype, "generation": generation, "dataflow_key": stable_hash64(did),
               "first_seen_dataflow_type": prev.get("first_seen_dataflow_type") or dtype, "discovery_sources": sources,
               "type_changed_at": ctx.clock.now() if prev.get("dataflow_type") not in (None, dtype) and dtype != "Unknown" else prev.get("type_changed_at")}
        if did in admin_pbi and did not in scanner:
            adm = admin_pbi[did]
            row.update(dataflow_name=adm.get("name"), description=adm.get("description"), owner_upn=adm.get("configured_by"),
                       owner_source="configuredBy" if adm.get("configured_by") else None, model_url=adm.get("model_url"),
                       storage_type=adm.get("storage_type"))
        elif did in items and did not in scanner:
            item = items[did]
            creator = dig(item, "creatorPrincipal", "userDetails", "userPrincipalName")
            row.update(dataflow_name=item.get("name"), description=item.get("description"), modified_at=parse_ts(item.get("lastUpdatedDate")),
                       owner_upn=creator, owner_source="creatorPrincipal" if creator else None,
                       tags=[t.get("displayName") for t in (item.get("tags") or []) if isinstance(t, dict)])
        rows.append(row)
    merge_by_signature(lake, "Assessment_Dataflows", rows)

    complete = disc.get("complete", {})
    can_delete = (complete.get("admin_items") or not cfg.collect_gen2) and (complete.get("admin_pbi") or not cfg.collect_gen1)
    present = set(scanner) | set(admin_pbi) | set(items)
    gone = [{"dataflow_id": did} for did, r in existing.items() if did not in present and (in_scope is None or r.get("workspace_id") in in_scope)]
    if gone and can_delete:
        lake.mark_deleted("Assessment_Dataflows", gone)
    elif gone:
        ctx.logger.warning("%d dataflows missing from the scan but the tenant-wide lists were incomplete: not flagged as deleted", len(gone))
    cascaded = cascade_workspace_deletions(ctx)
    current = {r["dataflow_id"]: r for r in lake.read("Assessment_Dataflows")}
    counts = defaultdict(Counter)
    for r in current.values():
        counts[r["workspace_id"]][r["dataflow_type"]] += 1
    if counts:
        lake.merge("Assessment_Workspaces", [{"workspace_id": w, "gen1_dataflow_count": c["Gen1"], "gen2_dataflow_count": c["Gen2"]} for w, c in counts.items()],
                   update_cols=["gen1_dataflow_count", "gen2_dataflow_count"])
    caps = ctx.state.get("capacities", {})
    by_cap = defaultdict(Counter)
    for w, c in counts.items():
        cap = (ctx.state.get("workspaces", {}).get(w) or {}).get("capacity_id")
        if cap:
            by_cap[cap].update(c)
    if by_cap:
        lake.merge("Assessment_Capacities", [{"capacity_id": cid, "gen1_dataflow_count": c["Gen1"], "gen2_dataflow_count": c["Gen2"]} for cid, c in by_cap.items() if cid in caps],
                   update_cols=["gen1_dataflow_count", "gen2_dataflow_count"])
    flips = sum(1 for r in current.values() if r.get("type_changed_at") and r.get("first_seen_dataflow_type") == "Gen1" and r["dataflow_type"] == "Gen2")
    types = Counter(r["dataflow_type"] for r in current.values())
    return {"rows_in": len(present), "rows_out": len(current),
            "message": f"Gen1={types['Gen1']} Gen2={types['Gen2']} Unknown={types['Unknown']} migrated-in-place={flips} removed-with-their-workspace={cascaded['dataflows']}"}

### Run - section 4: dataflow discovery

In [ ]:
run_steps(ctx, sections={4})

# 5. Dependency collection
## 5.1 Definition analysis (Gen1 `model.json`, Gen2 `queryMetadata.json` + `mashup.pq`)
Definitions are the only source for the *technical complexity* indicators and several migration rules. They are
expensive (admin export: 200 req/h; Gen2 `getDefinition` needs read+write on the item), so collection is **budgeted per
run, prioritized (new, then changed, then retries) and checkpointed by dataflow modification time**.

The M analyzer is tokenizer-aware (strings, quoted identifiers and comments are masked before structure is derived),
so a `;` or a keyword inside a string never splits a query. It extracts: queries, parameters, custom functions, linked
entities (`PowerPlatform.Dataflows` / legacy `PowerBI.Dataflows` navigation with workspace and dataflow ids), computed
entities (a loaded query referencing another loaded query), native queries and AI-insight functions.
Incremental refresh is detected through the real CDM trait `pbi:refreshPolicy` (with `ppiConfig` as a tolerated alias).

In [ ]:
import hashlib

_MASK_RE = re.compile(r'"(?:[^"]|"")*"|//[^\n]*|/\*.*?\*/', re.S)
_BRACKETS = re.compile(r"[()\[\]{};]")
_MEMBER_RE = re.compile(r'^\s*shared\s+(#"(?:[^"]|"")*"|[^\W\d][\w\.]*)\s*=', re.S)   # [^\W\d]: a letter (any script) or underscore
_PARAM_RE = re.compile(r"IsParameterQuery\s*=\s*true", re.I)
_NAV_RE = re.compile(r'\b(workspaceId|dataflowId|entity)\s*=\s*"([^"]+)"', re.I)
_NATIVE_RE = re.compile(r"Value\.NativeQuery\s*\(|\[\s*Query\s*=", re.I)
_AI_RE = re.compile(r"\b(?:AI|AzureML|CognitiveServices|AzureOpenAI|TextAnalytics|VisionService)\.[A-Za-z]\w*")
_SOURCE_CALL_RE = re.compile(r"\b([A-Z][A-Za-z0-9]*)\.([A-Z][A-Za-z0-9]*)\s*\(")
_DF_URL_RE = re.compile(r"groups/([0-9a-fA-F-]{36})/dataflows/([0-9a-fA-F-]{36})")
SOURCE_FUNCTION_ROOTS = {"Sql", "Oracle", "Web", "OData", "Excel", "Csv", "Json", "Xml", "SharePoint", "AzureStorage", "Odbc", "OleDb", "Folder",
                         "File", "Salesforce", "PostgreSQL", "MySQL", "Snowflake", "Lakehouse", "Warehouse", "AnalysisServices", "Teradata", "DB2",
                         "SapHana", "SapBusinessWarehouse", "GoogleAnalytics", "Dynamics365", "CommonDataService", "Dataverse", "Databricks",
                         "AzureDataExplorer", "Access", "Informix", "Sybase", "Redshift", "GoogleBigQuery", "Cds", "PowerPlatform", "PowerBI",
                         "AzureSQL", "AzureCosmos", "Cosmos", "MongoDB", "Parquet", "Pdf", "Exchange", "ActiveDirectory", "Kusto", "Fabric"}


def mask_m(text):
    """Replace string contents and comments with filler of equal length so structure can be derived safely."""
    def repl(match):
        piece = match.group(0)
        if piece.startswith('"'):
            return '"' + re.sub(r"[^\r\n]", "_", piece[1:-1]) + '"'
        return re.sub(r"[^\r\n]", " ", piece)
    return _MASK_RE.sub(repl, text)


def _top_level_segments(masked):
    depth, start, segments = 0, 0, []
    for match in _BRACKETS.finditer(masked):
        ch = match.group()
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth = max(0, depth - 1)
        elif depth == 0:
            segments.append((start, match.start()))
            start = match.end()
    if masked[start:].strip():
        segments.append((start, len(masked)))
    return segments


def _skip_attribute(masked, begin):
    """Skip leading [attribute] groups (e.g. [DataDestinations = ...]) of a segment."""
    pos = begin
    while True:
        while pos < len(masked) and masked[pos].isspace():
            pos += 1
        if pos >= len(masked) or masked[pos] != "[":
            return pos
        depth = 0
        for idx in range(pos, len(masked)):
            depth += masked[idx] == "["
            depth -= masked[idx] == "]"
            if depth == 0:
                pos = idx + 1
                break
        else:
            return pos


def is_function_expr(masked_expr):
    s = masked_expr.lstrip()
    if not s.startswith("("):
        return False
    depth = 0
    for idx, ch in enumerate(s):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            if depth == 0:
                rest = s[idx + 1:].lstrip()
                return rest.startswith("=>") or bool(re.match(r"as\s+[\w\.\s\?]+?\s*=>", rest))
    return False


def parse_m_members(text, masked=None):
    masked = masked if masked is not None else mask_m(text)
    members = []
    for begin, end in _top_level_segments(masked):
        start = _skip_attribute(masked, begin)
        head = _MEMBER_RE.match(masked[start:end])
        if not head:
            continue
        name_span = head.span(1)
        raw_name = text[start + name_span[0]: start + name_span[1]]
        name = raw_name[2:-1].replace('""', '"') if raw_name.startswith('#"') else raw_name
        expr_start = start + head.end()
        expr, mexpr = text[expr_start:end], masked[expr_start:end]
        kind = "parameter" if _PARAM_RE.search(expr) else ("function" if is_function_expr(mexpr) else "query")
        members.append({"name": name, "kind": kind, "expr": expr, "masked": mexpr})
    return members


# Tokens of a masked M expression (string contents and comments are already blanked): quoted identifier, #keyword, string, number,
# dotted identifier (Table.SelectRows and dim.Customer are ONE identifier in M), =>, brackets, comma and equals; anything else is one character.
_TOKEN_RE = re.compile(r'(?P<quoted>#"[_\r\n]*")|(?P<hash>#[^\W\d]\w*)|(?P<string>"[_\r\n]*")|(?P<num>\d[\w\.]*)'
                       r'|(?P<name>[^\W\d]\w*(?:\.[^\W\d]\w*)*)|(?P<op>=>|[()\[\]{},=])|\S')
_CLOSER = {")": "paren", "}": "brace", "]": "bracket"}


class _LetScope:
    """A `let` being scanned: what it binds, what its initializers and body read, and the binding being initialised right now."""
    __slots__ = ("bound", "uses", "current", "body")

    def __init__(self):
        self.bound, self.uses, self.current, self.body = set(), set(), None, False


def _scan_m(expr, masked, interesting=None):
    """The names an M expression reads (a set), restricted to `interesting` when given. One pass, independent of the number of queries.

    * Field selectors ([Country], x[Country]?) and record keys ([Country = 1]) name fields, not queries, so they are not reads.
      Whether `Name =` is a binding or a comparison depends on the enclosing bracket, not on the preceding comma: it binds only
      right after a `let` or `[` or after a comma whose innermost enclosing scope is that `let` or `[`; inside (...) and {...}
      it is an equality test and the name is read.
    * `let` scopes are lexical. A step is visible in every initializer and in the body of its own `let`, whatever the order, but
      not in its own initializer (there the name reads the enclosing scope, unless written @name), and not outside the let: the
      body ends at the comma or closing bracket of whatever encloses the let. Names the let does not bind are passed outwards.
      Function parameters and the fields of a record literal are not modelled: they can only add a read that is not one."""
    tokens = [(m.lastgroup, m.start(), m.end()) for m in _TOKEN_RE.finditer(masked)]
    used, stack, lets, expect = set(), [], [], False

    def emit(name, innermost, inclusive=False):
        i = innermost
        while not inclusive and i >= 0 and lets[i].current == name:    # inside its own initializer: the name belongs to the enclosing scope
            i -= 1
        (lets[i].uses if i >= 0 else used).add(name)

    def pop_to(size):
        while len(stack) > size:
            frame = stack.pop()
            if isinstance(frame, _LetScope):
                lets.pop()
                for name in frame.uses - frame.bound:
                    emit(name, len(lets) - 1)

    for i, (kind, start, end) in enumerate(tokens):
        text = masked[start:end]
        if kind in ("name", "quoted"):
            if kind == "name" and text == "let":
                scope = _LetScope()
                stack.append(scope)
                lets.append(scope)
                expect = True
                continue
            if kind == "name" and text == "in":
                opened = next((j for j in range(len(stack) - 1, -1, -1) if isinstance(stack[j], _LetScope) and not stack[j].body), None)
                if opened is not None:
                    pop_to(opened + 1)                                  # brackets opened since the let and never closed are dropped
                    stack[opened].body, stack[opened].current = True, None
                    expect = False
                    continue
            following = masked[tokens[i + 1][1]:tokens[i + 1][2]] if i + 1 < len(tokens) else ""
            top = stack[-1] if stack else None
            binds, selects = expect and following == "=", expect and top == "bracket" and following == "]"
            expect = False
            name = text if kind == "name" else expr[start + 2:end - 1].replace('""', '"')
            if binds:
                if isinstance(top, _LetScope) and not top.body:
                    top.current = name
                    if interesting is None or name in interesting:
                        top.bound.add(name)
            elif not selects and (interesting is None or name in interesting):
                emit(name, len(lets) - 1, inclusive=i > 0 and masked[tokens[i - 1][1]:tokens[i - 1][2]] == "@")
        elif text == "[":
            stack.append("bracket")
            expect = True
        elif text == "(":
            stack.append("paren")
            expect = False
        elif text == "{":
            stack.append("brace")
            expect = False
        elif text in _CLOSER:
            for j in range(len(stack) - 1, -1, -1):
                if stack[j] == _CLOSER[text]:
                    pop_to(j)
                    break
            expect = False
        elif text == ",":
            while stack and isinstance(stack[-1], _LetScope) and stack[-1].body:    # the comma of what encloses the let ends its body
                pop_to(len(stack) - 1)
            top = stack[-1] if stack else None
            if isinstance(top, _LetScope):
                top.current = None
            expect = isinstance(top, _LetScope) or top == "bracket"
        else:
            expect = False
    pop_to(0)
    return used


def analyze_m_document(text):
    """Structure and risk signals of an M section document. Safe on empty or malformed input."""
    text = text or ""
    members = parse_m_members(text)
    queries = [m for m in members if m["kind"] == "query"]
    names = [m["name"] for m in queries]
    refs, linked, natives, ai, connectors = {}, [], [], set(), Counter()
    legacy = False
    name_set, order = set(names), {n: i for i, n in enumerate(names)}
    for member in queries:
        expr, mexpr = member["expr"], member["masked"]
        refs[member["name"]] = sorted(_scan_m(expr, mexpr, name_set) - {member["name"]}, key=order.get)
        if re.search(r"(?:PowerPlatform|PowerBI)\.Dataflows\s*\(", mexpr):
            is_legacy = bool(re.search(r"PowerBI\.Dataflows\s*\(", mexpr))
            legacy = legacy or is_legacy
            workspace, mine = None, []
            for key, value in _NAV_RE.findall(expr):
                key = key.lower()
                if key == "workspaceid":
                    workspace = value
                elif key == "dataflowid":
                    mine.append({"query": member["name"], "workspace_id": gid(workspace), "dataflow_id": gid(value), "entity": None, "legacy": is_legacy})
                elif mine:
                    mine[-1]["entity"] = value
            linked.extend(mine or [{"query": member["name"], "workspace_id": gid(workspace), "dataflow_id": None, "entity": None, "legacy": is_legacy}])
        if _NATIVE_RE.search(mexpr):
            natives.append(member["name"])
        ai.update(_AI_RE.findall(mexpr))
        for root, fn in _SOURCE_CALL_RE.findall(mexpr):
            if root in SOURCE_FUNCTION_ROOTS:
                connectors[f"{root}.{fn}"] += 1
    return {"query_names": names, "parameter_names": [m["name"] for m in members if m["kind"] == "parameter"],
            "function_names": [m["name"] for m in members if m["kind"] == "function"], "refs": refs, "linked": linked,
            "native_queries": natives, "ai_functions": sorted(ai), "connectors": dict(connectors), "legacy_connector": legacy, "chars": len(text)}

In [ ]:
def _find_keys(obj, predicate, depth=0):
    """Yield (key, value) pairs anywhere in a JSON structure whose key satisfies predicate."""
    if depth > 8:
        return
    if isinstance(obj, dict):
        for key, value in obj.items():
            if predicate(str(key)):
                yield key, value
            yield from _find_keys(value, predicate, depth + 1)
    elif isinstance(obj, list):
        for item in obj:
            yield from _find_keys(item, predicate, depth + 1)


def entity_has_incremental_refresh(entity):
    """pbi:refreshPolicy of type IncrementalRefreshPolicy (CDM extension); ppiConfig tolerated as an alias."""
    for key, value in _find_keys(entity, lambda k: k.lower() in ("pbi:refreshpolicy", "refreshpolicy", "ppiconfig")):
        if key.lower() == "ppiconfig" and value:
            return True
        if isinstance(value, dict):
            if "incremental" in str(value.get("$type", "")).lower():
                return True
            if any(k in value for k in ("incrementalGranularity", "rollingWindowGranularity", "incrementalPeriods", "rollingWindowPeriods")):
                return True
    return False


def entity_is_directquery(entity):
    for key, value in _find_keys(entity, lambda k: k.lower() in ("mode", "storagemode")):
        if str(value).lower() == "directquery":
            return True
    return False


def _model_mashup(model):
    mashup = model.get("pbi:mashup")
    if not isinstance(mashup, dict):
        for annotation in model.get("annotations") or []:
            if isinstance(annotation, dict) and annotation.get("name") == "pbi:mashup":
                value = annotation.get("value")
                if isinstance(value, str):
                    try:
                        value = json.loads(value)
                    except ValueError:
                        value = None
                mashup = value if isinstance(value, dict) else None
    return mashup if isinstance(mashup, dict) else {}


def _definition_result(analysis, entity_names, enabled, incremental, directquery, extra_linked):
    """Shared shaping of facts (Silver columns) and evidence for Gen1 and Gen2 definitions."""
    loaded = set(enabled)
    computed = sorted(q for q in loaded if any(r in loaded for r in analysis["refs"].get(q, [])))
    linked = {l["query"] for l in analysis["linked"]} | set(extra_linked)
    findings = {"incremental_entities": sorted(incremental), "computed_entities": computed, "linked_entities": sorted(linked),
                "native_query_entities": sorted(analysis["native_queries"]), "directquery_entities": sorted(directquery),
                "ai_functions": analysis["ai_functions"]}
    facts = {
        "query_count": len(analysis["query_names"]) or len(entity_names), "entity_count": len(loaded) if loaded else len(entity_names),
        "custom_function_count": len(analysis["function_names"]), "parameter_count": len(analysis["parameter_names"]),
        "incremental_refresh_count": len(incremental), "computed_entity_count": len(computed), "linked_entity_count": len(linked),
        "directquery_entity_count": len(directquery), "native_query_count": len(analysis["native_queries"]), "m_script_chars": analysis["chars"],
        "legacy_connector_used": analysis["legacy_connector"], "ai_function_count": len(analysis["ai_functions"]),
        "definition_findings": json.dumps(findings, sort_keys=True),
    }
    return facts


def analyze_model_json(model):
    """Gen1 definition. Returns {'facts', 'linked_refs', 'hash'}."""
    model = model if isinstance(model, dict) else {}
    mashup = _model_mashup(model)
    document = mashup.get("document") or ""
    analysis = analyze_m_document(document)
    entities = [e for e in (model.get("entities") or []) if isinstance(e, dict)]
    entity_names = [str(e.get("name")) for e in entities if e.get("name")]
    qmeta = mashup.get("queriesMetadata") if isinstance(mashup.get("queriesMetadata"), dict) else {}
    if qmeta:
        known = set(analysis["query_names"])
        enabled = [n for n, meta in qmeta.items() if isinstance(meta, dict) and meta.get("loadEnabled") is not False]
        enabled = [n for n in enabled if n in known] or ([] if known else enabled)
    else:
        enabled = entity_names
    reference_models = {gid(r.get("id")): r.get("location") for r in (model.get("referenceModels") or []) if isinstance(r, dict)}
    ref_entities, linked_refs = [], [dict(l) for l in analysis["linked"] if l["dataflow_id"]]
    for entity in entities:
        if entity.get("$type") == "ReferenceEntity" or entity.get("linkedEntityRef"):
            ref_entities.append(str(entity.get("name")))
            location = reference_models.get(gid(entity.get("modelId"))) or ""
            match = _DF_URL_RE.search(str(location))
            if match:
                linked_refs.append({"query": entity.get("name"), "workspace_id": gid(match.group(1)), "dataflow_id": gid(match.group(2)),
                                    "entity": entity.get("source"), "legacy": False})
    incremental = [str(e.get("name")) for e in entities if entity_has_incremental_refresh(e)]
    directquery = [str(e.get("name")) for e in entities if entity_is_directquery(e)]
    facts = _definition_result(analysis, entity_names, enabled, incremental, directquery, ref_entities)
    seen, unique = set(), []
    for ref in linked_refs:
        key = (ref["dataflow_id"], ref["query"])
        if key not in seen:
            seen.add(key)
            unique.append(ref)
    digest = short_hash(document + "|" + ",".join(sorted(entity_names)) + "|" + json.dumps(model.get("referenceModels") or [], sort_keys=True, default=str), 20)
    return {"facts": facts, "linked_refs": unique, "hash": digest, "connections": [], "gateway_object_id": None}


def decode_parts(definition):
    parts = {}
    for part in (definition or {}).get("parts") or []:
        payload = part.get("payload")
        if payload is None or part.get("payloadType", "InlineBase64") != "InlineBase64":
            continue
        try:
            parts[part.get("path")] = base64.b64decode(payload).decode("utf-8", "replace")
        except (ValueError, TypeError):
            continue
    return parts


def analyze_gen2_definition(definition):
    """Gen2 definition: queryMetadata.json + mashup.pq, or the legacy dataflow-content.json (editingSessionMashup)."""
    parts = decode_parts(definition)
    meta, document = {}, parts.get("mashup.pq") or ""
    try:
        if "queryMetadata.json" in parts:
            meta = json.loads(parts["queryMetadata.json"])
        elif "dataflow-content.json" in parts:
            content = json.loads(parts["dataflow-content.json"])
            session = content.get("editingSessionMashup") or content
            document = document or session.get("mashupDocument") or ""
            meta = {"queriesMetadata": session.get("queriesMetadata") or {}, "connections": session.get("connectionOverrides") or [],
                    "gatewayObjectId": session.get("gatewayObjectId")}
    except ValueError:
        meta = {}
    analysis = analyze_m_document(document)
    qmeta = meta.get("queriesMetadata") if isinstance(meta.get("queriesMetadata"), dict) else {}
    enabled = [n for n, m in qmeta.items() if isinstance(m, dict) and m.get("loadEnabled", True) is not False]
    if not qmeta:
        enabled = analysis["query_names"]
    facts = _definition_result(analysis, list(qmeta) or analysis["query_names"], enabled, [], [], [])
    if meta.get("parametric") and not facts["parameter_count"]:
        facts["parameter_count"] = 1
    linked_refs = [dict(l) for l in analysis["linked"] if l["dataflow_id"]]
    connections = [c for c in (meta.get("connections") or []) if isinstance(c, dict)]
    return {"facts": facts, "linked_refs": linked_refs, "hash": short_hash(document + json.dumps(meta, sort_keys=True, default=str), 20),
            "connections": connections, "gateway_object_id": gid(meta.get("gatewayObjectId"))}

In [ ]:
DF_DEF_COLS = ["query_count", "entity_count", "custom_function_count", "parameter_count", "incremental_refresh_count", "computed_entity_count",
               "linked_entity_count", "directquery_entity_count", "native_query_count", "legacy_connector_used", "ai_function_count",
               "definition_findings", "m_script_chars", "definition_status", "definition_collected_at", "definition_hash"]
_RANK = {"new": 0, "changed": 1, "retry": 2, "retry-failed": 2, "retry-denied": 3, "expired": 4}


def _version(row):
    moment = row.get("modified_at")
    return moment.isoformat() if moment else "none"


def select_candidates(ctx, phase, wanted, budget, ttl_hours=None, force=False):
    """Prioritized work list: never collected, then changed, then retries; denied / up-to-date entities are skipped."""
    scope = ctx.state.get("scope_ws_ids")
    picked = []
    for row in ctx.lake.read("Assessment_Dataflows", where={"workspace_id": scope} if scope else None):
        if not wanted(row):
            continue
        collect, reason = ctx.checkpoints.decide("Dataflow", row["dataflow_id"], phase, version=_version(row) if phase == "definition" else None,
                                                  ttl_hours=ttl_hours, force=force)
        if not collect:
            continue
        cp = ctx.checkpoints.get("Dataflow", row["dataflow_id"], phase)
        rank = 4 if (force and cp is not None and reason == "new") else _RANK.get(reason, 5)
        picked.append((rank, (cp or {}).get("completed_at") or dt.datetime.min, row["dataflow_id"], row))
    picked.sort(key=lambda p: p[:3])
    return [p[3] for p in picked[:budget]], len(picked)


GEN2_DEFINITION_PARTS = ("queryMetadata.json", "mashup.pq", "dataflow-content.json")
GEN1_MODEL_KEYS = ("entities", "pbi:mashup", "annotations")


class EmptyDefinition(ApiError):
    """The API answered, but the body holds no dataflow definition (empty or unexpected): not a result to store as collected."""


def fetch_definition(ctx, row):
    wid, did = row["workspace_id"], row["dataflow_id"]
    if row["dataflow_type"] == "Gen2":
        fabric = ctx.clients["fabric"]
        resp = fabric.request("POST", f"workspaces/{wid}/dataflows/{did}/getDefinition", json_body={}, ok=(200, 202), buckets=("fabric_platform",))
        body = fabric.lro(resp)
        definition = body.get("definition") or body
        if not any(part in decode_parts(definition) for part in GEN2_DEFINITION_PARTS):
            raise EmptyDefinition("the Gen2 definition holds none of " + ", ".join(GEN2_DEFINITION_PARTS), endpoint="dataflows/getDefinition", retryable=True)
        return analyze_gen2_definition(definition)
    pbi = ctx.clients["pbi"]
    order = ("workspace", "admin") if ctx.cfg.definition_source == "workspace" else ("admin", "workspace")
    denied = ctx.state.setdefault("definition_denied_workspaces", set())
    failure = None
    for source in order:
        if source == "workspace" and wid in denied:
            continue
        try:
            if source == "admin":
                model = pbi.get_json(f"admin/dataflows/{did}/export", buckets=("pbi_admin_dataflow_export",))
            else:
                model = pbi.get_json(f"groups/{wid}/dataflows/{did}", buckets=("pbi_dataflow",))
            if not isinstance(model, dict) or not any(key in model for key in GEN1_MODEL_KEYS):
                raise EmptyDefinition(f"the {source} API returned no dataflow model", endpoint="dataflows/definition", retryable=True)
            return analyze_model_json(model)
        except (ForbiddenError, NotFoundError, AuthError, EmptyDefinition) as exc:
            failure = exc
            if source == "workspace" and isinstance(exc, (ForbiddenError, AuthError)):
                denied.add(wid)
    raise failure


def _definition_edges(row, result):
    edges = []
    for ref in result["linked_refs"]:
        if ref["dataflow_id"] and ref["dataflow_id"] != row["dataflow_id"]:
            attrs = {"legacy_connector": True} if ref.get("legacy") else None
            edges.append(edge_row("Dataflow", ref["dataflow_id"], ref["workspace_id"], None, "Dataflow", row["dataflow_id"], row["workspace_id"],
                                  row.get("dataflow_name"), "LinkedEntity", evidence="Definition", attributes=attrs))
    return edges


def _connection_sources(row, result):
    sources = []
    for conn in result["connections"]:
        kind, path = conn.get("kind") or "Unknown", str(conn.get("path") or "")
        details = {"path": path}
        if kind.lower() in ("sql", "azuresqldatabase", "azuresql", "synapse", "postgresql", "mysql", "oracle", "snowflake"):
            server, _, database = path.partition(";")
            details = {"server": server, "database": database or None}
        elif path.lower().startswith("http"):
            details = {"url": path}
        gateway = result.get("gateway_object_id")
        inst = {"type": kind, "details": {k: v for k, v in details.items() if v}, "gateway_id": gateway}
        src = build_source_row(row["dataflow_id"], row["workspace_id"], gid(conn.get("connectionId")), inst, "Definition")
        src["connection_id"] = gid(conn.get("connectionId"))
        sources.append(src)
    return sources


def persist_definitions(ctx, results):
    """results: [(row, parsed result or None, status)] -> Silver rows, lineage edges and connection sources."""
    now, rows, edges, sources, ids = ctx.clock.now(), [], [], [], []
    for row, result, status in results:
        ids.append(row["dataflow_id"])
        update = {"dataflow_id": row["dataflow_id"], "definition_status": status, "definition_collected_at": now if status == "Collected" else None}
        if result:
            update.update(result["facts"])
            update["definition_hash"] = result["hash"]
            edges += _definition_edges(row, result)
            sources += _connection_sources(row, result)
        rows.append(update)
    collected = [r for r, res, st in results if st == "Collected"]
    if collected:
        ctx.lake.merge("Assessment_Dataflows", [r for r in rows if r.get("definition_status") == "Collected"], update_cols=DF_DEF_COLS)
    others = [r for r in rows if r.get("definition_status") != "Collected"]
    if others:
        ctx.lake.merge("Assessment_Dataflows", others, update_cols=["definition_status"])
    collected_ids = {c["dataflow_id"] for c in collected}
    new_edges = {e["edge_id"] for e in edges}
    stale = [{"edge_id": r["edge_id"]} for r in ctx.lake.read("Assessment_Lineage", where={"dst_id": collected_ids})
             if r.get("evidence") == "Definition" and r["edge_id"] not in new_edges] if collected_ids else []
    if stale:
        ctx.lake.mark_deleted("Assessment_Lineage", stale)
    if edges:
        ctx.lake.merge("Assessment_Lineage", edges)
    if sources:
        merge_by_signature(ctx.lake, "Assessment_DataflowSources", sources)
    return len(rows)


BREAKER_THRESHOLD = 25


def run_budgeted(ctx, phase, candidates, work, handle_ok, persist, entity_type="Dataflow", key_of=None, breaker=BREAKER_THRESHOLD):
    """Execute work(row) for candidates in a thread pool; checkpoint outcomes; stop gracefully on throttling / deadline.

    persist(outcomes) receives [(row, result | None, status)] in micro-batches of p_batch_flush_size.
    Circuit breaker: when the first `breaker` calls are all denied (401/403) the phase stops instead of burning the quota.
    """
    cfg, outcomes, stop, done = ctx.cfg, [], None, 0
    key_of = key_of or (lambda r: r["dataflow_id"])
    denied = succeeded = 0
    pending = deque(candidates)
    with cf.ThreadPoolExecutor(max_workers=cfg.max_workers) as pool:
        inflight = {}
        while pending or inflight:
            while pending and len(inflight) < cfg.max_workers * 2 and stop is None and not ctx.deadline.expired(reserve_sec=60):
                row = pending.popleft()
                inflight[pool.submit(work, row)] = row
            if not inflight:
                break
            finished, _ = cf.wait(list(inflight), return_when=cf.FIRST_COMPLETED)
            for future in finished:
                row = inflight.pop(future)
                key = key_of(row)
                try:
                    outcome = future.result()
                except GracefulStop as exc:
                    stop = exc
                    continue
                except (ForbiddenError, AuthError) as exc:
                    denied += 1
                    ctx.checkpoints.mark(entity_type, key, phase, "Denied", error=str(exc))
                    ctx.record_error(phase, entity_type, key, exc.endpoint, exc=exc)
                    outcomes.append((row, None, "AccessDenied"))
                except NotFoundError as exc:
                    ctx.checkpoints.mark(entity_type, key, phase, "NotFound", error=str(exc))
                    outcomes.append((row, None, "Failed"))
                except Exception as exc:  # noqa: BLE001 - per-entity isolation
                    ctx.checkpoints.mark(entity_type, key, phase, "Failed", error=f"{type(exc).__name__}: {exc}")
                    ctx.record_error(phase, entity_type, key, getattr(exc, "endpoint", None), exc=exc)
                    outcomes.append((row, None, "Failed"))
                else:
                    succeeded += 1
                    handle_ok(row, outcome)
                    outcomes.append((row, outcome, "Collected"))
                done += 1
                if breaker and succeeded == 0 and denied >= breaker and stop is None:
                    stop = AccessBreaker(f"the first {denied} calls were all denied (HTTP 401/403): grant the service principal access to the workspaces "
                                         f"(or use p_auth_mode = user as a Fabric admin) and re-run")
                if len(outcomes) >= cfg.batch_flush_size:
                    persist(outcomes)
                    outcomes = []
                    ctx.checkpoints.flush()
    if outcomes:
        persist(outcomes)
    ctx.checkpoints.flush()
    if stop is not None:
        raise stop
    return done


def phase_definitions(ctx):
    cfg = ctx.cfg
    wanted = lambda r: ((r["dataflow_type"] == "Gen1" and cfg.collect_gen1 and cfg.collect_definitions)  # noqa: E731
                        or (r["dataflow_type"] == "Gen2" and cfg.collect_gen2_definitions))
    candidates, eligible = select_candidates(ctx, "definition", wanted, cfg.budget_definitions_per_run, force=cfg.run_mode == "full")
    ctx.logger.info("definitions: %d eligible, %d selected (budget %d)", eligible, len(candidates), cfg.budget_definitions_per_run)

    def ok(row, result):
        ctx.checkpoints.mark("Dataflow", row["dataflow_id"], "definition", "Done", version=_version(row))

    done = run_budgeted(ctx, "definition", candidates, lambda row: fetch_definition(ctx, row), ok, lambda outcomes: persist_definitions(ctx, outcomes))
    denied = sum(1 for r in ctx.error_rows if r["phase"] == "definition" and r["http_status"] in (401, 403))
    if denied:
        ctx.logger.warning("%d definitions could not be read (HTTP 401/403): add the service principal to those workspaces (Contributor for Gen2) "
                           "or accept lower assessment confidence for them", denied)
    left = max(0, eligible - len(candidates))
    if left:
        ctx.logger.info("%d eligible dataflows remain for the next run", left)
    return {"rows_in": eligible, "rows_out": done, "status": "Partial" if left and done else "Succeeded",
            "message": f"{done} collected, {denied} denied, {left} left for the next run"}

## 5.2 Complementary collectors (all budgeted, checkpointed and individually switchable)
| Phase | API | Default | Purpose |
|---|---|---|---|
| `compute_engine` | `groups/{ws}/dataflows` (workspace member) | on | Enhanced compute engine setting (rule ECE-001) |
| `admin_datasources` | `admin/dataflows/{id}/datasources` | off | Sources for dataflows the Scanner did not describe |
| `upstream_verify` | `admin/groups/{ws}/dataflows/{id}/upstreamDataflows` (200 req/h) | off | Cross-check of the Scanner lineage |
| `pipelines` | `admin/items?type=DataPipeline` + `getDefinition` | off | Pipelines whose `RefreshDataFlow` activity runs a dataflow (control edges) |
| `item_connections` | `workspaces/{ws}/items/{id}/connections` (read+write) | off | Gen2 connection details, credential type, gateway type |
| `owners_graph` | Microsoft Graph `users/{upn}` (`User.Read.All`) | off | Owner display name and disabled / deleted accounts (rule OWN-001) |
| `source_rollup` | gateways (security-trimmed) | on | Per-dataflow source and gateway counts |

A **circuit breaker** stops a phase after 25 consecutive denied calls with no success, so a missing workspace permission
costs 25 calls instead of the whole quota. Denied entities are not retried for `p_denied_retry_days`.

In [ ]:
def phase_compute_engine(ctx):
    cfg, pbi, lake = ctx.cfg, ctx.clients["pbi"], ctx.lake
    scope = ctx.state.get("scope_ws_ids")
    picked = []
    for ws in lake.read("Assessment_Workspaces", where={"workspace_id": scope} if scope else None):
        if not ws.get("gen1_dataflow_count"):
            continue
        collect, reason = ctx.checkpoints.decide("Workspace", ws["workspace_id"], "compute_engine", ttl_hours=168)
        if collect:
            picked.append((_RANK.get(reason, 5), ws["workspace_id"], ws))
    picked.sort(key=lambda p: p[:2])
    candidates = [p[2] for p in picked[:cfg.budget_definitions_per_run]]

    def work(row):
        body = pbi.get_json(f"groups/{row['workspace_id']}/dataflows", buckets=("pbi_dataflow",))
        return {gid(d.get("objectId")): d.get("computeEngineBehavior") or "NotConfigured" for d in body.get("value", [])}

    def persist(outcomes):
        rows = [{"dataflow_id": did, "compute_engine_behavior": value} for _, result, status in outcomes if result for did, value in result.items()]
        known = {r["dataflow_id"] for r in lake.read("Assessment_Dataflows")}
        rows = [r for r in rows if r["dataflow_id"] in known]
        if rows:
            lake.merge("Assessment_Dataflows", rows, update_cols=["compute_engine_behavior"])

    done = run_budgeted(ctx, "compute_engine", candidates, work,
                        lambda row, result: ctx.checkpoints.mark("Workspace", row["workspace_id"], "compute_engine", "Done"), persist,
                        entity_type="Workspace", key_of=lambda r: r["workspace_id"])
    return {"rows_in": len(picked), "rows_out": done, "message": f"{done} workspaces read, {max(0, len(picked) - len(candidates))} left"}

In [ ]:
def phase_admin_datasources(ctx):
    cfg, pbi, lake = ctx.cfg, ctx.clients["pbi"], ctx.lake
    covered = {s["dataflow_id"] for s in lake.read("Assessment_DataflowSources")}
    candidates, eligible = select_candidates(ctx, "datasources", lambda r: r["dataflow_type"] == "Gen1" and r["dataflow_id"] not in covered,
                                             cfg.budget_datasources_per_run, ttl_hours=168)

    def work(row):
        body = pbi.get_json(f"admin/dataflows/{row['dataflow_id']}/datasources", buckets=("pbi_admin_dataflow_datasources",))
        return [build_source_row(row["dataflow_id"], row["workspace_id"], gid(d.get("datasourceId")),
                                 {"type": d.get("datasourceType"), "details": d.get("connectionDetails") or {}, "gateway_id": gid(d.get("gatewayId"))}, "AdminDatasources")
                for d in body.get("value", [])]

    def persist(outcomes):
        rows = [s for _, result, _s in outcomes if result for s in result]
        if rows:
            merge_by_signature(lake, "Assessment_DataflowSources", rows)

    done = run_budgeted(ctx, "datasources", candidates, work, lambda row, result: ctx.checkpoints.mark("Dataflow", row["dataflow_id"], "datasources", "Done"), persist)
    return {"rows_in": eligible, "rows_out": done, "message": f"{done} dataflows read, {max(0, eligible - len(candidates))} left"}


def phase_upstream_verify(ctx):
    cfg, pbi, lake = ctx.cfg, ctx.clients["pbi"], ctx.lake
    candidates, eligible = select_candidates(ctx, "upstream", lambda r: r["dataflow_type"] == "Gen1", cfg.budget_datasources_per_run, ttl_hours=168)

    def work(row):
        body = pbi.get_json(f"admin/groups/{row['workspace_id']}/dataflows/{row['dataflow_id']}/upstreamDataflows", buckets=("pbi_admin_dataflow_upstream",))
        return [edge_row("Dataflow", u.get("targetDataflowId"), u.get("groupId"), None, "Dataflow", row["dataflow_id"], row["workspace_id"], row.get("dataflow_name"),
                         "LinkedEntity", evidence="AdminUpstream") for u in body.get("value", [])]

    def persist(outcomes):
        done_ids = {r["dataflow_id"] for r, result, _ in outcomes if result is not None}
        edges = [e for _, result, _s in outcomes if result for e in result]
        fresh = {e["edge_id"] for e in edges}
        stale = [{"edge_id": e["edge_id"]} for e in lake.read("Assessment_Lineage", where={"dst_id": done_ids})
                 if e.get("evidence") == "AdminUpstream" and e["edge_id"] not in fresh] if done_ids else []
        if stale:
            lake.mark_deleted("Assessment_Lineage", stale)
        if edges:
            lake.merge("Assessment_Lineage", edges)

    done = run_budgeted(ctx, "upstream", candidates, work, lambda row, result: ctx.checkpoints.mark("Dataflow", row["dataflow_id"], "upstream", "Done"), persist)
    return {"rows_in": eligible, "rows_out": done, "message": f"{done} dataflows verified, {max(0, eligible - len(candidates))} left"}

In [ ]:
def find_dataflow_activities(node, found=None):
    """All RefreshDataFlow activities anywhere in a pipeline definition (containers nest activities arbitrarily)."""
    found = [] if found is None else found
    if isinstance(node, dict):
        props = node.get("typeProperties")
        if str(node.get("type", "")).lower() == "refreshdataflow" and isinstance(props, dict) and props.get("dataflowId"):
            found.append({"dataflow_id": gid(props["dataflowId"]), "workspace_id": gid(props.get("workspaceId")), "activity": node.get("name")})
        for value in node.values():
            find_dataflow_activities(value, found)
    elif isinstance(node, list):
        for value in node:
            find_dataflow_activities(value, found)
    return found


def phase_pipelines(ctx):
    cfg, fabric, lake = ctx.cfg, ctx.clients["fabric"], ctx.lake
    buckets = ("fabric_admin_items", "fabric_platform")
    try:
        items = list(fabric.paginate("admin/items", params={"type": "DataPipeline"}, items_key="itemEntities", buckets=buckets))
    except ApiError as exc:
        if exc.status != 400:
            raise
        items = [i for i in fabric.paginate("admin/items", items_key="itemEntities", buckets=buckets) if i.get("type") == "DataPipeline"]
    scope = ctx.state.get("scope_ws_ids")
    rows = [{"id": gid(i["id"]), "workspace_id": gid(i.get("workspaceId")), "name": i.get("name"), "modified": str(i.get("lastUpdatedDate") or "none")}
            for i in items if i.get("id") and (not scope or gid(i.get("workspaceId")) in scope)]
    picked = []
    for row in rows:
        collect, reason = ctx.checkpoints.decide("Pipeline", row["id"], "pipelines", version=row["modified"])
        if collect:
            picked.append((_RANK.get(reason, 5), row["id"], row))
    picked.sort(key=lambda p: p[:2])
    candidates = [p[2] for p in picked[:cfg.budget_datasources_per_run]]

    def work(row):
        resp = fabric.request("POST", f"workspaces/{row['workspace_id']}/dataPipelines/{row['id']}/getDefinition", json_body={}, ok=(200, 202), buckets=("fabric_platform",))
        body = fabric.lro(resp)
        parts = decode_parts(body.get("definition") or body)
        if "pipeline-content.json" not in parts:
            raise EmptyDefinition("the pipeline definition has no pipeline-content.json part", endpoint="dataPipelines/getDefinition", retryable=True)
        content = json.loads(parts["pipeline-content.json"] or "{}")
        return find_dataflow_activities(content)

    def persist(outcomes):
        done_ids = {r["id"] for r, result, _ in outcomes if result is not None}
        edges = []
        for row, result, _ in outcomes:
            for ref in result or []:
                edges.append(edge_row("Dataflow", ref["dataflow_id"], ref["workspace_id"], None, "DataPipeline", row["id"], row["workspace_id"], row["name"],
                                      "PipelineActivity", kind="CONTROL", evidence="PipelineDefinition", attributes={"activity": ref["activity"]}))
        fresh = {e["edge_id"] for e in edges}
        stale = [{"edge_id": e["edge_id"]} for e in lake.read("Assessment_Lineage", where={"dst_id": done_ids})
                 if e.get("evidence") == "PipelineDefinition" and e["edge_id"] not in fresh] if done_ids else []
        if stale:
            lake.mark_deleted("Assessment_Lineage", stale)
        if edges:
            lake.merge("Assessment_Lineage", edges)

    done = run_budgeted(ctx, "pipelines", candidates, work,
                        lambda row, result: ctx.checkpoints.mark("Pipeline", row["id"], "pipelines", "Done", version=row["modified"]), persist,
                        entity_type="Pipeline", key_of=lambda r: r["id"])
    return {"rows_in": len(rows), "rows_out": done, "message": f"{done} pipelines inspected of {len(rows)}"}

In [ ]:
_CONNECTIVITY_GATEWAY_TYPE = {"OnPremisesGateway": "OnPremises", "OnPremisesGatewayPersonal": "OnPremises", "VirtualNetworkGateway": "VirtualNetwork",
                              "StreamingVirtualNetworkGateway": "VirtualNetwork"}


def connection_to_source(row, conn):
    details = conn.get("connectionDetails") or {}
    ctype, path = details.get("type") or "Unknown", str(details.get("path") or "")
    parsed = {"path": path}
    if ctype.lower() in ("sql", "azuresqldatabase", "azuresql", "synapse", "postgresql", "mysql", "oracle", "snowflake"):
        server, _, database = path.partition(";")
        parsed = {"server": server, "database": database or None}
    elif path.lower().startswith("http"):
        parsed = {"url": path}
    gateway_type = _CONNECTIVITY_GATEWAY_TYPE.get(conn.get("connectivityType"))
    gateway_id = gid(conn.get("gatewayId"))
    inst = {"type": ctype, "details": {k: v for k, v in parsed.items() if v}, "gateway_id": gateway_id}
    src = build_source_row(row["dataflow_id"], row["workspace_id"], gid(conn.get("id")), inst, "ItemConnections")
    src.update(connection_id=gid(conn.get("id")), credential_type=dig(conn, "credentialDetails", "credentialType"), gateway_type=gateway_type or src["gateway_type"],
               uses_gateway=bool(gateway_id or gateway_type), is_on_premises=gateway_type == "OnPremises" or src["is_on_premises"])
    return src


def phase_item_connections(ctx):
    cfg, fabric, lake = ctx.cfg, ctx.clients["fabric"], ctx.lake
    candidates, eligible = select_candidates(ctx, "connections", lambda r: r["dataflow_type"] == "Gen2", cfg.budget_definitions_per_run, ttl_hours=168)

    def work(row):
        conns = list(fabric.paginate(f"workspaces/{row['workspace_id']}/items/{row['dataflow_id']}/connections", items_key="value", buckets=("fabric_platform",)))
        return [connection_to_source(row, c) for c in conns]

    def persist(outcomes):
        rows = [s for _, result, _s in outcomes if result for s in result]
        if rows:
            merge_by_signature(lake, "Assessment_DataflowSources", rows)

    done = run_budgeted(ctx, "connections", candidates, work, lambda row, result: ctx.checkpoints.mark("Dataflow", row["dataflow_id"], "connections", "Done"), persist)
    return {"rows_in": eligible, "rows_out": done, "message": f"{done} Gen2 dataflows read, {max(0, eligible - len(candidates))} left"}

In [ ]:
def phase_owners_graph(ctx):
    """Owner display name and account state through Microsoft Graph (needs an SPN with User.Read.All)."""
    graph, lake = ctx.clients["graph"], ctx.lake
    rows = lake.read("Assessment_Dataflows")
    owners = sorted({(r.get("owner_upn") or "").lower() for r in rows if r.get("owner_upn") and "@" in r["owner_upn"]})
    picked = [o for o in owners if ctx.checkpoints.decide("Owner", o, "graph", ttl_hours=168)[0]][:ctx.cfg.budget_datasources_per_run]
    resolved = {}
    for upn in picked:
        try:
            body = graph.get_json(f"users/{urllib.parse.quote(upn)}", params={"$select": "displayName,accountEnabled,department"}, buckets=("graph",))
            resolved[upn] = {"owner_display_name": body.get("displayName"), "owner_enabled": body.get("accountEnabled")}
            ctx.checkpoints.mark("Owner", upn, "graph", "Done")
        except NotFoundError:
            resolved[upn] = {"owner_display_name": None, "owner_enabled": False}
            ctx.checkpoints.mark("Owner", upn, "graph", "Done")
        except (ForbiddenError, AuthError) as exc:
            ctx.logger.warning("Microsoft Graph denied (%s): grant User.Read.All to the service principal or leave p_enrich_owners_graph off", exc.status)
            ctx.record_error("owners_graph", "Owner", upn, exc.endpoint, exc=exc)
            break
    updates = [{"dataflow_id": r["dataflow_id"], **resolved[r["owner_upn"].lower()]} for r in rows if r.get("owner_upn") and r["owner_upn"].lower() in resolved]
    if updates:
        lake.merge("Assessment_Dataflows", updates, update_cols=["owner_display_name", "owner_enabled"])
    ctx.checkpoints.flush()
    return {"rows_in": len(owners), "rows_out": len(resolved), "message": f"{len(resolved)} owners resolved, {sum(1 for v in resolved.values() if v['owner_enabled'] is False)} disabled or missing"}

In [ ]:
def phase_source_rollup(ctx):
    """Per-dataflow connectivity indicators from the source inventory; gateway names where the identity can list gateways."""
    lake, fabric = ctx.lake, ctx.clients["fabric"]
    gateways = {}
    try:
        gateways = {gid(g.get("id")): g for g in fabric.paginate("gateways", items_key="value", buckets=("fabric_platform",))}
    except ApiError as exc:
        ctx.logger.info("gateway names unavailable (%s): the gateway list is security-trimmed, ids are kept", exc.status)
    sources = lake.read("Assessment_DataflowSources")
    named = []
    for s in sources:
        g = gateways.get(s.get("gateway_id"))
        if g and not s.get("gateway_name"):
            named.append({"dataflow_id": s["dataflow_id"], "source_key": s["source_key"], "gateway_name": g.get("displayName") or g.get("name"),
                          "gateway_type": g.get("type") or s.get("gateway_type")})
    if named:
        lake.merge("Assessment_DataflowSources", named, update_cols=["gateway_name", "gateway_type"])
        sources = lake.read("Assessment_DataflowSources")
    by_df = defaultdict(list)
    for s in sources:
        by_df[s["dataflow_id"]].append(s)
    workspaces = {w["workspace_id"]: w for w in lake.read("Assessment_Workspaces")}
    updates = []
    for df in lake.read("Assessment_Dataflows"):
        srcs = by_df.get(df["dataflow_id"], [])
        scanned = df["dataflow_type"] == "Gen1" and (workspaces.get(df["workspace_id"]) or {}).get("scan_state") == "Scanned"
        if not srcs and not scanned:
            continue
        gateway_ids = {s["gateway_id"] for s in srcs if s.get("gateway_id")}
        updates.append({"dataflow_id": df["dataflow_id"], "source_count": len({s["source_key"] for s in srcs}),
                        "source_type_count": len({(s.get("datasource_type") or "").lower() for s in srcs}), "gateway_count": len(gateway_ids),
                        "uses_gateway": any(s.get("uses_gateway") for s in srcs), "uses_on_prem_gateway": any(s.get("is_on_premises") and s.get("uses_gateway") for s in srcs)})
    if updates:
        lake.merge("Assessment_Dataflows", updates)
    return {"rows_in": len(sources), "rows_out": len(updates), "message": f"{len(gateways)} gateways visible"}

### Run - section 5: definitions, dependencies and connectivity

In [ ]:
run_steps(ctx, sections={5})

# 6. Refresh history collection
| Generation | API | Notes |
|---|---|---|
| Gen1 | `groups/{ws}/dataflows/{id}/transactions` | Needs workspace access (an SPN must be a member); no admin variant exists |
| Gen2 | `workspaces/{ws}/items/{id}/jobs/instances` | Fabric keeps about the 100 most recent completed runs; `failureReason` carries the error |
| Gen2 schedules | `items/{id}/jobs/{jobType}/schedules` (`Refresh`, then `Execute`) | Gen1 schedules come from the Scanner (there is no read API) |

Every record is normalized (status, refresh type, duration) and merged into `Assessment_RefreshHistory`
(partitioned by month). Per-dataflow KPIs - success rate, average and p95 duration, **consecutive failures**, last error,
API-triggered runs - are computed from the observed window and drive the *Deferred* class (failing refresh) and the
refresh-criticality score. Work is budgeted per run and refreshed after `p_refresh_history_ttl_hours`.

In [ ]:
_STATUS = {
    "Success": ("success", "succeeded", "completed", "complete", "succeededwithwarnings"),
    "Failed": ("failed", "failure", "error", "faulted", "timedout"),
    "Cancelled": ("cancelled", "canceled", "cancelling"),
    "InProgress": ("inprogress", "in progress", "running", "notstarted", "not started", "queued", "accepted", "pending", "executing"),
    "Skipped": ("deduped", "skipped"),
}
_EXTERNAL_TYPES = {"viaapi", "external", "api"}
REFRESH_COLS = ["last_refresh_at", "last_refresh_status", "last_refresh_duration_sec", "last_success_at", "refresh_runs_observed", "refresh_success_rate",
                "refresh_avg_duration_sec", "refresh_p95_duration_sec", "consecutive_failures", "last_error_message", "external_trigger_seen",
                "refresh_history_status", "refresh_history_collected_at"]
SCHEDULE_COLS = ["schedule_enabled", "schedule_days", "schedule_times", "schedule_runs_per_day", "schedule_timezone", "schedule_notify_option"]
WEEK = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def normalize_status(raw):
    text = str(raw or "").strip().lower().replace("_", "")
    for status, names in _STATUS.items():
        if text in names:
            return status
    return "Unknown"


def build_history_row(df, record, source_api):
    """Normalize one Gen1 transaction or Gen2 job instance. Returns None for records that never started."""
    if source_api == "PBI.transactions":
        rid, rtype, raw = record.get("id"), record.get("refreshType"), record.get("status")
        start, end = parse_ts(record.get("startTime")), parse_ts(record.get("endTime"))
        code, message = None, record.get("errorMessage") or record.get("error")
    else:
        rid, rtype, raw = record.get("id"), record.get("invokeType"), record.get("status")
        start, end = parse_ts(record.get("startTimeUtc")), parse_ts(record.get("endTimeUtc"))
        failure = record.get("failureReason") or {}
        code, message = failure.get("errorCode"), failure.get("message")
    if not rid or start is None:
        return None
    status = normalize_status(raw)
    duration = (end - start).total_seconds() if end and end >= start else None
    return {"dataflow_id": df["dataflow_id"], "refresh_id": str(rid), "workspace_id": df["workspace_id"], "dataflow_type": df["dataflow_type"],
            "source_api": source_api, "refresh_type": rtype, "status": raw, "status_normalized": status, "start_time": start, "end_time": end,
            "duration_sec": duration, "error_code": code, "error_message": truncate(message, 2000) if message else None, "start_date": start.date(),
            "start_month": start.strftime("%Y-%m")}


def compute_refresh_metrics(rows, now):
    """KPIs from the observed window (rows are Assessment_RefreshHistory dicts)."""
    ordered = sorted(rows, key=lambda r: r["start_time"], reverse=True)
    finished = [r for r in ordered if r["status_normalized"] in ("Success", "Failed")]
    successes = [r for r in finished if r["status_normalized"] == "Success"]
    streak = 0
    for r in ordered:
        if r["status_normalized"] == "Failed":
            streak += 1
        elif r["status_normalized"] == "Success":
            break
    last = ordered[0] if ordered else None
    last_failed = next((r for r in ordered if r["status_normalized"] == "Failed"), None)
    last_done = next((r for r in ordered if r["status_normalized"] in ("Success", "Failed", "Cancelled") and r["duration_sec"] is not None), None)
    durations = [r["duration_sec"] for r in finished if r["duration_sec"] is not None]
    success_durations = [r["duration_sec"] for r in successes if r["duration_sec"] is not None]
    return {
        "last_refresh_at": last["start_time"] if last else None, "last_refresh_status": last["status_normalized"] if last else None,
        "last_refresh_duration_sec": last_done["duration_sec"] if last_done else None,
        "last_success_at": max((r["end_time"] or r["start_time"] for r in successes), default=None),
        "refresh_runs_observed": len(ordered), "refresh_success_rate": (len(successes) / len(finished)) if finished else None,
        "refresh_avg_duration_sec": (sum(durations) / len(durations)) if durations else None,
        "refresh_p95_duration_sec": percentile(success_durations or durations, 0.95), "consecutive_failures": streak,
        "last_error_message": last_failed["error_message"] if last_failed else None,
        "external_trigger_seen": any(str(r.get("refresh_type") or "").lower() in _EXTERNAL_TYPES for r in ordered),
        "refresh_history_status": "Collected", "refresh_history_collected_at": now,
    }


def parse_gen2_schedules(schedules):
    """Combine the enabled Fabric schedules of a dataflow into the Silver schedule columns."""
    items = [s for s in schedules or [] if isinstance(s, dict)]
    if not items:
        return {"schedule_enabled": False, "schedule_days": [], "schedule_times": [], "schedule_runs_per_day": 0}
    enabled = [s for s in items if s.get("enabled")]
    runs, times, days, tz = 0, [], set(), None
    for sched in enabled:
        cfg = sched.get("configuration") or {}
        kind = str(cfg.get("type") or "").lower()
        tz = tz or cfg.get("localTimeZoneId")
        if kind == "cron":
            interval = max(1, int(cfg.get("interval") or 1440))
            runs += max(1, round(1440 / interval))
            days.update(WEEK)
        else:
            slots = [str(t) for t in cfg.get("times") or []]
            runs += len(slots)
            times += slots
            days.update(cfg.get("weekdays") or WEEK)
    return {"schedule_enabled": bool(enabled), "schedule_days": [d for d in WEEK if d in days], "schedule_times": sorted(set(times)),
            "schedule_runs_per_day": runs, "schedule_timezone": tz}

In [ ]:
def fetch_refresh(ctx, row):
    """Worker: read the refresh window (and the Gen2 schedule) of one dataflow."""
    cfg, wid, did = ctx.cfg, row["workspace_id"], row["dataflow_id"]
    schedule, schedule_denied = None, False
    if row["dataflow_type"] == "Gen2":
        fabric = ctx.clients["fabric"]
        records = list(fabric.paginate(f"workspaces/{wid}/items/{did}/jobs/instances", items_key="value", buckets=("fabric_jobs", "fabric_platform")))
        records = [r for r in records if str(r.get("jobType") or "refresh").lower() in ("refresh", "execute")]
        source = "Fabric.jobInstances"
        for job_type in ("Refresh", "Execute"):
            try:
                schedule = parse_gen2_schedules(list(fabric.paginate(f"workspaces/{wid}/items/{did}/jobs/{job_type}/schedules", items_key="value",
                                                                     buckets=("fabric_jobs", "fabric_platform"))))
                break
            except NotFoundError:
                continue
            except (ForbiddenError, AuthError) as exc:
                # the job history is still valid: keep it, but make the missing schedule visible (schedule_enabled stays NULL = unknown)
                schedule_denied = True
                ctx.record_error("refresh", "DataflowSchedule", did, exc.endpoint, exc=exc)
                break
    else:
        body = ctx.clients["pbi"].get_json(f"groups/{wid}/dataflows/{did}/transactions", buckets=("pbi_dataflow",))
        records = body.get("value", []) if isinstance(body, dict) else body
        source = "PBI.transactions"
    rows = [r for r in (build_history_row(row, rec, source) for rec in records) if r]
    rows.sort(key=lambda r: r["start_time"], reverse=True)
    rows = rows[:cfg.refresh_history_max_records]
    return {"rows": rows, "metrics": compute_refresh_metrics(rows, ctx.clock.now()), "schedule": schedule, "schedule_denied": schedule_denied}


def persist_refresh(ctx, outcomes):
    """Returns (history rows written, Gen2 schedules that could not be read)."""
    history, updates, schedules_denied = [], [], 0
    for row, result, status in outcomes:
        if result is None:
            updates.append({"dataflow_id": row["dataflow_id"], "refresh_history_status": "AccessDenied" if status == "AccessDenied" else "Failed"})
            continue
        history += result["rows"]
        schedules_denied += bool(result.get("schedule_denied"))
        update = {"dataflow_id": row["dataflow_id"], **result["metrics"]}
        if result["schedule"]:
            update.update({k: result["schedule"].get(k) for k in SCHEDULE_COLS})
        updates.append(update)
    if history:
        months = {r["start_month"] for r in history}
        ctx.lake.merge("Assessment_RefreshHistory", history, prune={"start_month": months})
    merge_by_signature(ctx.lake, "Assessment_Dataflows", updates)
    return len(history), schedules_denied


def phase_refresh_history(ctx):
    cfg = ctx.cfg
    wanted = lambda r: ((r["dataflow_type"] == "Gen1" and cfg.collect_gen1) or (r["dataflow_type"] == "Gen2" and cfg.collect_gen2))  # noqa: E731
    candidates, eligible = select_candidates(ctx, "refresh", wanted, cfg.budget_refresh_dataflows_per_run, ttl_hours=cfg.refresh_history_ttl_hours)
    ctx.logger.info("refresh history: %d due, %d selected (budget %d dataflows)", eligible, len(candidates), cfg.budget_refresh_dataflows_per_run)
    written = []

    def ok(row, result):
        ctx.checkpoints.mark("Dataflow", row["dataflow_id"], "refresh", "Done")

    def persist(outcomes):
        written.append(persist_refresh(ctx, outcomes))

    done = run_budgeted(ctx, "refresh", candidates, lambda row: fetch_refresh(ctx, row), ok, persist)
    denied = sum(1 for r in ctx.error_rows if r["phase"] == "refresh" and r["entity_type"] == "Dataflow" and r["http_status"] in (401, 403))
    runs, schedules_denied = sum(w[0] for w in written), sum(w[1] for w in written)
    if denied:
        ctx.logger.warning("%d refresh histories could not be read (HTTP 401/403): the service principal must be a member of those workspaces "
                           "(Gen1 transactions have no admin API); the Deferred and criticality rules treat them as unknown", denied)
    if schedules_denied:
        ctx.logger.warning("%d Gen2 schedules could not be read (HTTP 401/403): their schedule_enabled stays unknown (see Assessment_ErrorLog)", schedules_denied)
    left = max(0, eligible - len(candidates))
    return {"rows_in": eligible, "rows_out": runs, "status": "Partial" if left else "Succeeded",
            "message": f"{done} dataflows, {runs} runs written, {denied} denied, {schedules_denied} schedules denied, {left} left for the next run"}

### Run - section 6: refresh history and schedules

In [ ]:
run_steps(ctx, sections={6})

# 7. Migration readiness analysis
The analysis is a **pure function of the Silver tables** (`run_analysis`): it never calls an API, so it can be re-run
alone (`p_run_mode = "analysis_only"`) after tuning rules, weights or governance overrides.

```
Silver rows -> lineage graph -> rule evaluation -> per-path status (Wizard / Save-As) -> readiness class
            -> complexity score (6 components) -> effort -> recommended path -> waves (topological) -> Gold rows
```

## 7.1 Readiness precedence
| # | Condition | Class |
|---|---|---|
| 1 | Governance override `defer` | **Deferred** (override) |
| 2 | Stale (no successful refresh for `p_stale_days`) **and** no downstream consumers | **Deferred** - decommission candidate |
| 3 | A rule is *Unsupported* on every available path | **Unsupported** |
| 4 | `p_failing_consecutive` consecutive refresh failures | **Deferred** - fix first |
| 5 | An upstream dataflow is Unsupported or Deferred | **Deferred** - blocked by upstream (SEQ-001, propagated transitively) |
| 6 | A rule needs attention on the best available path (or the definition could not be inspected) | **Needs Attention** |
| 7 | otherwise | **Ready** |

Dataflows that are already Gen2 are out of scope (`readiness = NULL`).

In [ ]:
MAX_DEPTH = 6
MAX_DEPENDENCY_ROWS = 2000
DEFAULT_WEIGHTS = {"dependencies": 20, "refresh_criticality": 15, "gateway": 15, "source_complexity": 15, "unsupported": 25, "technical": 10}
BAND_THRESHOLDS = [(75, "Very High"), (50, "High"), (25, "Medium"), (0, "Low")]
BASE_EFFORT_DAYS = {"Low": 0.5, "Medium": 1.5, "High": 4.0, "Very High": 8.0}
WAVE_LABELS = {0: "Wave 0 - Pilot", 1: "Wave 1 - Quick wins", 2: "Wave 2 - Standard", 3: "Wave 3 - Complex or critical",
               4: "Wave 4 - Remediate or rebuild", 99: "Backlog - Deferred"}
STATUS_ORDER = {"Ready": 0, "NeedsAttention": 1, "Unsupported": 2}
STATUS_LABEL = {"Ready": "Ready", "NeedsAttention": "Needs Attention", "Unsupported": "Unsupported"}
WIZARD_NAME = {"Ready": "Ready to migrate", "NeedsAttention": "Needs Attention", "Unsupported": "Upgrade unavailable"}
SEVERITY_ORDER = {"Blocker": 0, "Major": 1, "Minor": 2, "Info": 3}
REBIND_TYPES = ("Dataflow", "SemanticModel", "Datamart", "DataPipeline")


def compute_engine_state(value):
    """Normalize the undocumented computeEngineBehavior setting to On / Optimized / Off / Unknown."""
    text = re.sub(r"[^a-z]", "", str(value or "").lower())
    if not text or text == "notconfigured":
        return "Unknown"
    if "optimiz" in text:
        return "Optimized"
    if "disabl" in text or text.endswith("off"):
        return "Off"
    if text.endswith("on") or "enabl" in text or "ondemand" in text:
        return "On"
    return "Unknown"


def json_object(value):
    """A dict from a JSON text (or a dict). null, lists, scalars and invalid text all become an empty dict."""
    if isinstance(value, dict):
        return value
    if not value:
        return {}
    try:
        decoded = json.loads(value)
    except (TypeError, ValueError):
        return {}
    return decoded if isinstance(decoded, dict) else {}


def edge_sort_key(e):
    return (e["src_id"], e["dst_id"], str(e.get("relationship")), str(e.get("evidence")))


def dedupe_edges(edges):
    """Collapse edges seen by several sources into one per (src, dst, relationship); keep evidences and merged attributes.
    Edges without a source or a target id cannot be placed in the graph and are skipped. Input order never changes the result."""
    merged = {}
    for e in sorted((e for e in edges if e.get("src_id") and e.get("dst_id")), key=edge_sort_key):
        key = (e["src_id"], e["dst_id"], e.get("relationship"))
        attrs = json_object(e.get("attributes"))
        cur = merged.get(key)
        if cur is None:
            merged[key] = {**e, "evidences": [e.get("evidence")], "attrs": dict(attrs)}
        else:
            cur["evidences"].append(e.get("evidence"))
            cur["attrs"].update(attrs)
            for field in ("src_name", "dst_name", "src_workspace_id", "dst_workspace_id"):
                cur[field] = cur.get(field) or e.get(field)
    return list(merged.values())


class LineageGraph:
    """Impact graph (edge src -> dst: a change to src impacts dst) with bounded closure, SCCs and dependency levels."""

    def __init__(self, edges):
        self.edges = sorted(edges, key=edge_sort_key)
        self.down, self.up = defaultdict(list), defaultdict(list)
        self.nodes = {}
        for e in self.edges:
            self.down[e["src_id"]].append(e)
            self.up[e["dst_id"]].append(e)
            self.nodes.setdefault(e["src_id"], {"type": e.get("src_type"), "name": e.get("src_name"), "workspace_id": e.get("src_workspace_id")})
            self.nodes.setdefault(e["dst_id"], {"type": e.get("dst_type"), "name": e.get("dst_name"), "workspace_id": e.get("dst_workspace_id")})
            for end, nid in (("src", e["src_id"]), ("dst", e["dst_id"])):
                node = self.nodes[nid]
                node["name"] = node.get("name") or e.get(f"{end}_name")
                node["workspace_id"] = node.get("workspace_id") or e.get(f"{end}_workspace_id")

    def closure_info(self, anchor, direction, max_depth=MAX_DEPTH, limit=None):
        """BFS from anchor: ([(node_id, hop, edge, path_ids), ...] nearest first, truncated).
        With a limit, `truncated` is True only when more nodes were reachable than returned. Datasource nodes are leaves."""
        adjacency = self.down if direction == "DOWNSTREAM" else self.up
        seen, queue, out = {anchor}, deque([(anchor, 0, (anchor,))]), []
        while queue:
            node, hop, path = queue.popleft()
            if hop >= max_depth:
                continue
            for edge in adjacency.get(node, ()):
                nxt = edge["dst_id"] if direction == "DOWNSTREAM" else edge["src_id"]
                if nxt in seen:
                    continue
                if limit is not None and len(out) >= limit:
                    return out, True
                seen.add(nxt)
                out.append((nxt, hop + 1, edge, path + (nxt,)))
                if self.nodes.get(nxt, {}).get("type") != "Datasource":
                    queue.append((nxt, hop + 1, path + (nxt,)))
        return out, False

    def closure(self, anchor, direction, max_depth=MAX_DEPTH, limit=None):
        return self.closure_info(anchor, direction, max_depth, limit)[0]


def strongly_connected_components(nodes, successors):
    """Iterative Tarjan. Returns components in topological order of the condensation (upstream components first)."""
    index, low, on_stack, stack, comps, counter = {}, {}, set(), [], [], [0]
    for root in nodes:
        if root in index:
            continue
        work = [(root, iter(sorted(successors.get(root, ()))))]
        index[root] = low[root] = counter[0]
        counter[0] += 1
        stack.append(root)
        on_stack.add(root)
        while work:
            node, it = work[-1]
            advanced = False
            for nxt in it:
                if nxt not in index:
                    index[nxt] = low[nxt] = counter[0]
                    counter[0] += 1
                    stack.append(nxt)
                    on_stack.add(nxt)
                    work.append((nxt, iter(sorted(successors.get(nxt, ())))))
                    advanced = True
                    break
                if nxt in on_stack:
                    low[node] = min(low[node], index[nxt])
            if advanced:
                continue
            work.pop()
            if work:
                parent = work[-1][0]
                low[parent] = min(low[parent], low[node])
            if low[node] == index[node]:
                comp = []
                while True:
                    member = stack.pop()
                    on_stack.discard(member)
                    comp.append(member)
                    if member == node:
                        break
                comps.append(sorted(comp))
    comps.reverse()
    return comps


def weak_components(nodes, adjacency):
    parent = {n: n for n in nodes}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, targets in adjacency.items():
        for b in targets:
            if a in parent and b in parent:
                parent[find(a)] = find(b)
    groups = defaultdict(list)
    for n in nodes:
        groups[find(n)].append(n)
    return list(groups.values())

In [ ]:
def bucket(value, steps):
    """Step function: the score of the highest threshold reached. steps = [(threshold, score), ...] ascending."""
    score = 0
    for threshold, points in steps:
        if value is not None and value >= threshold:
            score = points
    return score


def _label(node_id, graph, dfs):
    row = dfs.get(node_id)
    return (row or {}).get("dataflow_name") or graph.nodes.get(node_id, {}).get("name") or node_id


def _unique_by(edges, key):
    """One edge per related node (edges can arrive through several evidences); attributes are merged."""
    unique = {}
    for e in edges:
        cur = unique.get(e[key])
        if cur is None:
            unique[e[key]] = {**e, "attrs": dict(e["attrs"])}
        else:
            cur["attrs"].update(e["attrs"])
    return list(unique.values())


def direct_facts(df, graph, dfs):
    """Direct and transitive lineage facts used by rules, scores and criticality."""
    did = df["dataflow_id"]
    up_direct = _unique_by([e for e in graph.up.get(did, []) if e["src_type"] == "Dataflow"], "src_id")
    down_direct = _unique_by(graph.down.get(did, []), "dst_id")
    closure = graph.closure(did, "DOWNSTREAM")
    by_type = defaultdict(set)
    for nid, hop, edge, path in closure:
        by_type[graph.nodes.get(nid, {}).get("type", "Other")].add(nid)
    direct_types = defaultdict(list)
    for e in down_direct:
        direct_types[e["dst_type"]].append(e)
    wid = df.get("workspace_id")
    crosses = (any(e.get("src_workspace_id") and e["src_workspace_id"] != wid for e in up_direct)
               or any(e.get("dst_workspace_id") and e["dst_workspace_id"] != wid for e in down_direct))
    return {
        "upstream_dataflows": up_direct, "downstream_direct": down_direct, "direct_by_type": direct_types, "crosses_workspace": crosses,
        "downstream_dataflows": direct_types.get("Dataflow", []), "datasets_all": by_type.get("SemanticModel", set()),
        "reports_all": by_type.get("Report", set()), "dashboards_all": by_type.get("Dashboard", set()),
        "other_all": {n for t, s in by_type.items() if t not in ("SemanticModel", "Report", "Dashboard", "Dataflow") for n in s},
        "total_downstream": len(closure),
        "rebind_items": [e for t in REBIND_TYPES for e in direct_types.get(t, [])],
    }


def infer_criticality(df, facts, env):
    points = 0
    points += 2 if env == "Prod" else 0
    points += {"Certified": 2, "Promoted": 1, "Master data": 2}.get(df.get("endorsement"), 0)
    datasets = len(facts["datasets_all"])
    points += 2 if datasets >= 5 else (1 if datasets >= 1 else 0)
    points += 1 if facts["dashboards_all"] else 0
    points += 1 if len(facts["reports_all"]) >= 10 else 0
    points += 1 if (df.get("schedule_runs_per_day") or 0) >= 4 else 0
    return "High" if points >= 4 else ("Medium" if points >= 2 else "Low")


def make_hit(rule_id, entity="", evidence="", count=1):
    return {"rule_id": rule_id, "entity_name": entity or "", "evidence": evidence, "count": count}


def finding_names(findings, key):
    value = findings.get(key)
    return [str(v) for v in value] if isinstance(value, list) else []


def source_label(s):
    where = s.get("server") or s.get("url") or s.get("path") or s.get("datasource_id") or "?"
    return f"{s.get('datasource_type') or 'Source'}: {where}" + (f" / {s['database']}" if s.get("database") else "")


def evaluate_rules(cfg, now, df, ws, sources, facts, graph, dfs, same_name_count):
    """Rule hits for one Gen1 dataflow. Deferred-type rules (REF-001, STA-001) are included; SEQ-001 is added after propagation."""
    hits = []
    ws = ws or {}
    findings = json_object(df.get("definition_findings"))
    if ws.get("is_fabric_capable") is False or ws.get("is_on_dedicated_capacity") is False or ws.get("capacity_resolution") == "Shared":
        hits.append(make_hit("CAP-001", evidence=f"Workspace capacity: {ws.get('capacity_sku') or 'shared (Pro/PPU)'}"))
    if ws.get("capacity_resolution") == "Unresolved":
        hits.append(make_hit("CAP-002", evidence=f"capacity {ws.get('capacity_id')} not returned by the capacity APIs"))
    if df.get("storage_type") == "CustomADLS" or ws.get("dataflow_storage_id"):
        hits.append(make_hit("STO-001", evidence=f"modelUrl={df.get('model_url')}" if df.get("storage_type") == "CustomADLS" else f"workspace storage {ws.get('dataflow_storage_id')}"))
    ece = compute_engine_state(df.get("compute_engine_behavior"))
    if ece == "On":
        hits.append(make_hit("ECE-001", evidence=f"computeEngineBehavior={df.get('compute_engine_behavior')}"))
    elif ece == "Optimized":
        hits.append(make_hit("ECE-002", evidence="computeEngineBehavior=Optimized"))
    for e in facts["direct_by_type"].get("SemanticModel", []):
        if str(e["attrs"].get("storage_mode", "")).lower() == "directquery":
            hits.append(make_hit("DQ-001", e.get("dst_name") or e["dst_id"], "semantic model storage mode DirectQuery"))
    incremental = finding_names(findings, "incremental_entities")
    for name in incremental[:25]:
        hits.append(make_hit("INC-001", name, "incremental refresh policy on the entity"))
    if df.get("incremental_refresh_count") and not incremental:
        hits.append(make_hit("INC-001", "", f"{df['incremental_refresh_count']} entities with incremental refresh"))
    upstream_names = sorted({_label(e["src_id"], graph, dfs) for e in facts["upstream_dataflows"]})
    if (df.get("linked_entity_count") or 0) > 0 or upstream_names:
        hits.append(make_hit("LNK-001", evidence="reads: " + ", ".join(upstream_names[:8]) if upstream_names else f"{df.get('linked_entity_count')} linked entities"))
    if facts["downstream_dataflows"]:
        names = sorted({e.get("dst_name") or e["dst_id"] for e in facts["downstream_dataflows"]})
        hits.append(make_hit("LNK-002", evidence="read by: " + ", ".join(names[:8]), count=len(names)))
    for e in facts["downstream_direct"]:
        if e["attrs"].get("legacy_connector"):
            hits.append(make_hit("LNK-003", e.get("dst_name") or e["dst_id"], f"{e['dst_type']} uses PowerBI.Dataflows"))
    missing = sorted({e["src_id"] for e in facts["upstream_dataflows"] if e["src_id"] not in dfs})
    if missing:
        hits.append(make_hit("LNK-004", evidence="not in the inventory: " + ", ".join(missing[:5]), count=len(missing)))
    for name in finding_names(findings, "computed_entities")[:25]:
        hits.append(make_hit("CMP-001", name, "entity reads another loaded entity"))
    entities = df.get("entity_count") or 0
    if entities > 50:
        hits.append(make_hit("QRY-001", evidence=f"{entities} enabled queries (limit 50)"))
    elif entities >= 40:
        hits.append(make_hit("QRY-002", evidence=f"{entities} enabled queries"))
    for s in sources:
        if s.get("is_misconfigured"):
            hits.append(make_hit("CON-001", source_label(s), "reported as misconfigured"))
    gateways = [s for s in sources if s.get("uses_gateway")]
    if gateways:
        hits.append(make_hit("GW-001", evidence="gateways: " + ", ".join(sorted({s.get("gateway_name") or s.get("gateway_id") or "?" for s in gateways}))))
    if df.get("owner_enabled") is False or not df.get("owner_upn"):
        hits.append(make_hit("OWN-001", evidence=f"owner {df.get('owner_upn') or '(none)'} is " + ("disabled or missing" if df.get("owner_upn") else "not set")))
    name = df.get("dataflow_name") or ""
    if name and not ALLOWED_NAME_CHARS.match(name):
        hits.append(make_hit("NAM-001", evidence=f"name: {name}"))
    if same_name_count > 1:
        hits.append(make_hit("NAM-002", evidence=f"{same_name_count} dataflows named {name!r} in the workspace"))
    if (df.get("schedule_runs_per_day") or 0) > 48:
        hits.append(make_hit("SCH-001", evidence=f"{df['schedule_runs_per_day']} runs per day"))
    if df.get("schedule_enabled") is True:
        hits.append(make_hit("SCH-002", evidence=f"{df.get('schedule_runs_per_day') or '?'} runs per day"))
    if df.get("external_trigger_seen"):
        hits.append(make_hit("TRG-001", evidence="API-triggered refreshes in the observed window"))
    for e in facts["direct_by_type"].get("DataPipeline", []):
        hits.append(make_hit("PIP-001", e.get("dst_name") or e["dst_id"], "RefreshDataFlow activity"))
    if facts["rebind_items"]:
        hits.append(make_hit("REB-001", evidence=f"{len(facts['rebind_items'])} direct downstream items", count=len(facts["rebind_items"])))
    if (df.get("ai_function_count") or 0) > 0:
        hits.append(make_hit("AIX-001", evidence=", ".join(finding_names(findings, "ai_functions")[:5]) or f"{df['ai_function_count']} AI functions"))
    if cfg.require_definition_for_ready and df.get("definition_status") != "Collected":
        hits.append(make_hit("DAT-001", evidence=f"definition status: {df.get('definition_status') or 'not collected'}"))
    if (df.get("consecutive_failures") or 0) >= cfg.failing_consecutive:
        message = truncate(df.get("last_error_message"), 160)
        hits.append(make_hit("REF-001", evidence=f"{df['consecutive_failures']} consecutive failures" + (f": {message}" if message else "")))
    if df.get("refresh_history_status") == "Collected" and facts["total_downstream"] == 0:
        last = df.get("last_success_at") or df.get("modified_at")
        if last is not None and (now - last).days >= cfg.stale_days:
            basis = "last successful refresh" if df.get("last_success_at") else "no successful refresh observed, last modified"
            hits.append(make_hit("STA-001", evidence=f"{basis} {last:%Y-%m-%d}, no downstream consumers"))
    return [h for h in hits if RULES[h["rule_id"]].enabled]


def path_status(hits, path):
    """Worst non-deferred impact for a migration path: Ready / NeedsAttention / Unsupported."""
    key = "impact_wizard" if path == "wizard" else "impact_saveas"
    status = "Ready"
    for h in hits:
        impact = getattr(RULES[h["rule_id"]], key)
        if impact in STATUS_ORDER and STATUS_ORDER[impact] > STATUS_ORDER[status]:
            status = impact
    return status

## 7.2 Complexity score, effort, readiness class and recommendation
**Complexity score** (0-100) = weighted average of six component scores (each 0-100). Weights are configurable through
`p_score_weights` (JSON, e.g. `{"gateway": 25, "bands": {"high": 45}}`).

| Component | Default weight | Signals (points are capped at 100) |
|---|---|---|
| `dependencies` | 20 | 12 per upstream dataflow, 12 per downstream dataflow, 6 per downstream semantic model, 2 per report, 4 per dashboard, 4 per other asset, +15 crosses workspaces, +25 in a dependency cycle |
| `refresh_criticality` | 15 | schedule frequency (0-50), p95 duration (0-25), external triggers (+15), business criticality (High +10, Medium +5) |
| `gateway` | 15 | uses a gateway (50), on-premises source (+25), several gateways (+10 each, max 25) |
| `source_complexity` | 15 | number of sources (0-55), number of source types (0-30), heaviest category (Fabric 0, database / storage 5, file / SaaS 10, web / other 15) |
| `unsupported` | 25 | sum of the rule weights hit on the recommended path (x1.25 per additional entity, max x2) |
| `technical` | 10 | enabled queries (0-50), custom functions (0-15), parameters (0-10), native queries (0-10), script size (0-10), computed + linked entities (0-5) |

Bands (configurable): **Low** < 20, **Medium** < 40, **High** < 60, **Very High** >= 60.
**Effort** (person-days) = band base (0.5 / 1.5 / 4 / 8) + remediation effort of every rule that needs action on the recommended
path (1 + 0.5 per additional entity, REB-001 is 0.25 per downstream item to re-point).

In [ ]:
DEFAULT_BANDS = {"very_high": 60.0, "high": 40.0, "medium": 20.0}
PER_ITEM_RULES = {"REB-001"}
POST_MIGRATION_RULES = {"SCH-002", "REB-001", "PIP-001", "LNK-002", "LNK-003", "TRG-001", "INC-001", "AIX-001"}
FIX_FIRST_CATEGORIES = {"Connectivity", "Governance", "Capacity", "Data quality"}
ACTIONABLE = ("Unsupported", "NeedsAttention")
PATH_CODE = {"wizard": "UpgradeWizard", "saveas": "SaveAs"}
PATH_LABEL = {"wizard": "the Upgrade Wizard", "saveas": "Save-As"}
PATH_TEXT = {
    "UpgradeWizard": "upgrade in place with the Upgrade Wizard (keeps the dataflow id and name; the owner runs it for the whole workspace batch)",
    "SaveAs": "create the Gen2 copy with Save-As (new id: re-point the downstream items and re-enable the schedule)",
}


def resolve_scoring(cfg):
    """Component weights and band thresholds from the defaults and p_score_weights."""
    weights, bands = dict(DEFAULT_WEIGHTS), dict(DEFAULT_BANDS)
    for key, value in (cfg.score_weights or {}).items():
        if key == "bands":
            if not isinstance(value, dict):
                raise ConfigError("score_weights.bands must be an object such as {\"high\": 45}")
            for name, limit in value.items():
                if name not in bands:
                    raise ConfigError(f"unknown band {name!r}; expected one of {sorted(bands)}")
                bands[name] = float(limit)
        elif key in weights:
            weights[key] = float(value)
            if weights[key] < 0:
                raise ConfigError(f"score weight {key} must be >= 0")
        else:
            raise ConfigError(f"unknown score weight {key!r}; expected one of {sorted(weights)} or 'bands'")
    if sum(weights.values()) <= 0:
        raise ConfigError("score weights must not all be zero")
    if not bands["medium"] < bands["high"] < bands["very_high"]:
        raise ConfigError("band thresholds must increase: medium < high < very_high")
    return weights, bands


def band_for(score, bands):
    if score >= bands["very_high"]:
        return "Very High"
    if score >= bands["high"]:
        return "High"
    if score >= bands["medium"]:
        return "Medium"
    return "Low"


def path_impact(rule, path):
    return rule.impact_wizard if path == "wizard" else rule.impact_saveas


def hit_groups(hits):
    """Hits grouped per rule: total count, distinct entities and a few evidences."""
    groups = {}
    for h in hits:
        g = groups.setdefault(h["rule_id"], {"rule": RULES[h["rule_id"]], "count": 0, "entities": [], "evidence": []})
        g["count"] += h.get("count", 1)
        if h["entity_name"] and h["entity_name"] not in g["entities"]:
            g["entities"].append(h["entity_name"])
        if h["evidence"] and len(g["evidence"]) < 3:
            g["evidence"].append(h["evidence"])
    return groups


def rule_effort(rule, count):
    if rule.rule_id in PER_ITEM_RULES:
        return rule.effort_days * min(max(count, 1), 40)
    return rule.effort_days * (1 + 0.5 * (min(max(count, 1), 10) - 1))


def remediation_effort(hits, path):
    return sum(rule_effort(g["rule"], g["count"]) for g in hit_groups(hits).values() if path_impact(g["rule"], path) in ACTIONABLE)

In [ ]:
def score_dependencies(facts, in_cycle):
    points = (12 * len(facts["upstream_dataflows"]) + 12 * len(facts["downstream_dataflows"]) + 6 * len(facts["datasets_all"])
              + 2 * len(facts["reports_all"]) + 4 * len(facts["dashboards_all"]) + 4 * len(facts["other_all"])
              + (15 if facts.get("crosses_workspace") else 0) + (25 if in_cycle else 0))
    return float(min(100, points))


def score_refresh(df, criticality):
    runs = df.get("schedule_runs_per_day") or 0
    if df.get("schedule_enabled") is False:
        runs = 0
    duration = df.get("refresh_p95_duration_sec") or df.get("refresh_avg_duration_sec") or df.get("last_refresh_duration_sec")
    points = (bucket(runs, [(1, 15), (4, 30), (12, 40), (24, 50)]) + bucket(duration, [(300, 5), (1800, 15), (3600, 25)])
              + (15 if df.get("external_trigger_seen") else 0) + {"High": 10, "Medium": 5}.get(criticality, 0))
    return float(min(100, points))


def score_gateway(df, sources):
    gateways = {s.get("gateway_id") for s in sources if s.get("gateway_id")}
    points = 50 if (df.get("uses_gateway") or gateways) else 0
    points += 25 if (df.get("uses_on_prem_gateway") or any(s.get("is_on_premises") for s in sources)) else 0
    points += min(25, 10 * max(0, len(gateways) - 1))
    return float(min(100, points))


def score_sources(df, sources):
    count = df.get("source_count") if df.get("source_count") is not None else len(sources)
    types = df.get("source_type_count") if df.get("source_type_count") is not None else len({s.get("datasource_type") for s in sources})
    worst = max((SOURCE_COMPLEXITY.get(s.get("source_category") or "Other", 2.0) for s in sources), default=0.0)
    category = round(15 * (worst - 0.5) / 1.5) if worst else 0
    return float(min(100, bucket(count, [(1, 10), (3, 25), (6, 40), (11, 55)]) + bucket(types, [(2, 10), (4, 20), (7, 30)]) + max(0, category)))


def score_technical(df):
    entities = df.get("entity_count") or df.get("query_count") or 0
    points = (bucket(entities, [(10, 10), (25, 25), (40, 40), (50, 50)]) + bucket(df.get("custom_function_count"), [(1, 5), (4, 10), (10, 15)])
              + bucket(df.get("parameter_count"), [(1, 3), (5, 7), (10, 10)]) + bucket(df.get("native_query_count"), [(1, 5), (5, 10)])
              + bucket(df.get("m_script_chars"), [(20000, 5), (100000, 10)])
              + min(5, (df.get("computed_entity_count") or 0) + (df.get("linked_entity_count") or 0)))
    return float(min(100, points))


def score_unsupported(hits, path):
    """Rule weights of the hits that need action on the path; extra entities of one rule add 25% each (max x2)."""
    counts = Counter()
    for h in hits:
        if path_impact(RULES[h["rule_id"]], path) in ACTIONABLE:
            counts[h["rule_id"]] += h.get("count", 1)
    return float(min(100, sum(RULES[r].weight * (1 + 0.25 * min(n - 1, 4)) for r, n in counts.items())))


def complexity(weights, components, bands):
    score = sum(weights[k] * components[k] for k in weights) / sum(weights.values())
    return round(score, 1), band_for(score, bands)


def estimate_effort(band, hits, path):
    return round(BASE_EFFORT_DAYS[band] + remediation_effort(hits, path), 2)


def runs_per_month(df):
    if df.get("schedule_enabled") is None:
        return None
    if not df.get("schedule_enabled"):
        return 0.0
    days = len(df.get("schedule_days") or []) or 7
    return (df.get("schedule_runs_per_day") or 0) * days * 52 / 12


def estimate_cu(df):
    """Indicative Gen2 standard compute: 12 CU/s for the first 10 minutes of a refresh, 1.5 CU/s afterwards, from Gen1 durations."""
    duration = df.get("refresh_avg_duration_sec") or df.get("last_refresh_duration_sec")
    if not duration:
        return None, None
    per_run = 12.0 * min(duration, 600) + 1.5 * max(0.0, duration - 600)
    runs = runs_per_month(df)
    return round(per_run, 1), (None if runs is None else round(per_run * runs / 3600, 2))

In [ ]:
def choose_path(cfg, statuses, hits):
    """Recommended migration path: the configured preference, else the least severe status, then the least remediation effort."""
    options = ["wizard", "saveas"] if cfg.wizard_available else ["saveas"]
    if cfg.preferred_path in options:
        return cfg.preferred_path
    return min(options, key=lambda p: (STATUS_ORDER[statuses[p]], remediation_effort(hits, p), 0 if p == "wizard" else 1))


def classify_own(cfg, hits, statuses, path, override):
    """(class, reason, recommended path) from the dataflow's own findings. SEQ-001 is applied afterwards by propagation."""
    ids = {h["rule_id"] for h in hits}
    by_id = {h["rule_id"]: h for h in hits}
    if override and override.get("defer"):
        return "Deferred", f"Deferred by governance: {override.get('defer_reason') or 'no reason recorded'}", "None"
    if "STA-001" in ids:
        return "Deferred", f"Stale and unused: {by_id['STA-001']['evidence']}", "Retire"
    status = statuses[path]
    if status == "Unsupported":
        blocking = sorted({h["rule_id"] for h in hits if path_impact(RULES[h["rule_id"]], path) == "Unsupported"})
        fixable = all(RULES[r].category in FIX_FIRST_CATEGORIES for r in blocking)
        return ("Unsupported", f"Blocked on {PATH_LABEL[path]}: " + "; ".join(f"{r} {RULES[r].name}" for r in blocking),
                "FixThenReassess" if fixable else "ManualRebuild")
    if "REF-001" in ids:
        return "Deferred", f"Refresh failing: {by_id['REF-001']['evidence']}", "FixThenReassess"
    if status == "NeedsAttention":
        groups = hit_groups(hits)
        attention = sorted((g for g in groups.values() if path_impact(g["rule"], path) == "NeedsAttention"),
                           key=lambda g: (SEVERITY_ORDER[g["rule"].severity], -g["rule"].weight, g["rule"].rule_id))
        return ("NeedsAttention", f"{len(attention)} finding(s) to resolve on {PATH_LABEL[path]}: " + "; ".join(g["rule"].name for g in attention[:4])
                + (" ..." if len(attention) > 4 else ""), PATH_CODE[path])
    return "Ready", f"No blocking or attention finding on {PATH_LABEL[path]}", PATH_CODE[path]


def build_actions(hits, path, klass):
    """Remediation actions split into before / after the migration, most severe first."""
    wanted = ACTIONABLE + (("Deferred",) if klass == "Deferred" else ())
    groups = [g for g in hit_groups(hits).values() if path_impact(g["rule"], path) in wanted]
    groups.sort(key=lambda g: (0 if path_impact(g["rule"], path) == "Deferred" else 1, SEVERITY_ORDER[g["rule"].severity], -g["rule"].weight, g["rule"].rule_id))
    before, after = [], []
    for g in groups:
        rule = g["rule"]
        text = f"{rule.rule_id}: {rule.remediation}"
        if g["entities"]:
            text += f" [{', '.join(g['entities'][:5])}{' ...' if len(g['entities']) > 5 else ''}]"
        (after if rule.rule_id in POST_MIGRATION_RULES else before).append(text)
    return before, after


def build_recommendation(klass, rec_path, reason, wave_label):
    if klass == "Ready":
        return f"Ready: {PATH_TEXT[rec_path]}. Planned in {wave_label}."
    if klass == "NeedsAttention":
        return f"Needs attention before migrating - {reason}. Then {PATH_TEXT[rec_path]}. Planned in {wave_label}."
    if klass == "Unsupported":
        tail = "Fix the blockers and re-run the assessment." if rec_path == "FixThenReassess" else "Rebuild it as a native Dataflow Gen2 or keep it on Gen1."
        return f"Not migratable as is - {reason}. {tail}"
    if rec_path == "Retire":
        return f"Candidate for retirement - {reason}. Confirm with the owner and decommission instead of migrating."
    if rec_path == "FixThenReassess":
        return f"Deferred - {reason}. Resolve it, then re-run the assessment."
    return f"Deferred - {reason}."

## 7.3 Governance enrichment, dependency order and migration waves
**Wave logic** (evaluated after the readiness class, score and criticality are known):

| Wave | Rule |
|---|---|
| **0 - Pilot** | up to `p_pilot_size` dataflows chosen among the quick wins: isolated (no linked dataflow), **never Prod**, no failing refresh, a verified successful refresh preferred, few consumers, at most a third of the pilot per workspace |
| **1 - Quick wins** | Ready, Low complexity, business criticality not High |
| **2 - Standard** | Ready (Medium complexity or High criticality) or Needs Attention (Low / Medium complexity, criticality not High) |
| **3 - Complex or critical** | Ready with High / Very High complexity, Needs Attention with High / Very High complexity or High criticality |
| **4 - Remediate or rebuild** | Unsupported |
| **99 - Backlog** | Deferred (stale, failing refresh, blocked by an upstream, or deferred by governance) |

A **topological constraint** is applied afterwards: among the planned waves (0-4) a dataflow is never planned before the
dataflows it reads (`wave(d) >= wave(u)`); dependency cycles are condensed and migrate together. The backlog (99) is outside
the order: a dataflow blocked by an Unsupported or Deferred upstream (rule `SEQ-001`) waits there until the upstream is fixed
and the assessment re-run. `pin_wave` (0-4) in the governance overrides wins for Ready / Needs Attention dataflows.
Batches are `wave x workspace`, because the Upgrade Wizard works per workspace.

In [ ]:
SCOPE_IN, SCOPE_OUT, SCOPE_UNKNOWN = "In scope (Gen1 -> Gen2)", "Out of scope (already Gen2)", "Unknown generation"
READINESS_LABEL = {"Ready": "Ready", "NeedsAttention": "Needs Attention", "Unsupported": "Unsupported", "Deferred": "Deferred"}
ENV_RANK = {"Dev": 0, "Test": 1, "Unknown": 2, "Prod": 3}
BLOCKING_CLASSES = ("Unsupported", "Deferred")
PINNABLE_WAVES = (0, 1, 2, 3, 4)
UNKNOWN_WORKSPACE = "unknown"


NOT_AN_ID = {"", "na", "n/a", "n.a.", "n.a", "null", "none", "nan", "nil", "-", "--", "?", "0"}


def index_migrations(rows, known_gen2=()):
    """Save-As accelerator results (table DataflowMigrations) -> {gen1 id: {"ok", "gen2", "copies", "verified"}}.
    A success is never overwritten by a failure; placeholders such as NA are not ids. Save-As is not idempotent, so one Gen1
    dataflow can have several successful copies: the primary one is chosen deterministically (a copy found in the Gen2 inventory
    first, then the smallest id) and all of them are listed. `verified` tells whether a copy exists in the inventory."""
    out = {}
    for raw in rows or []:
        rec = {str(k).lower(): v for k, v in raw.items()}
        gen1, gen2 = gid(rec.get("dfg1_id")), gid(rec.get("dfg2_id"))
        if not gen1:
            continue
        entry = out.setdefault(gen1, {"ok": False, "gen2": None, "copies": set(), "verified": False})
        if str(rec.get("result") or "").strip().lower() in ("succeed", "success", "succeeded") and gen2 and gen2 not in NOT_AN_ID and gen2 != gen1:
            entry["copies"].add(gen2)
    known = set(known_gen2)
    for entry in out.values():
        copies = sorted(entry["copies"], key=lambda c: (c not in known, c))
        entry.update(ok=bool(copies), gen2=copies[0] if copies else None, copies=copies, verified=any(c in known for c in copies))
    return out


def resolve_environment(df, ws, gov, patterns):
    """Override (dataflow, then workspace) > workspace environment (deployment pipeline, naming) > dataflow name > Unknown."""
    for scope, key in (("dataflow", df["dataflow_id"]), ("workspace", df["workspace_id"])):
        declared = (gov[scope].get(key) or {}).get("environment")
        if declared:
            return declared
    ws_env = (ws or {}).get("environment")
    if ws_env and ws_env != "Unknown":
        return ws_env
    return detect_environment(df.get("dataflow_name"), patterns)


def governance_value(field, df, ws, gov):
    """Dataflow level override, else workspace level override, else the workspace row."""
    for scope, key in (("dataflow", df["dataflow_id"]), ("workspace", df["workspace_id"])):
        value = (gov[scope].get(key) or {}).get(field)
        if value not in (None, ""):
            return value
    return (ws or {}).get(field)


def resolve_criticality(df, ws, facts, env, gov):
    declared = governance_value("business_criticality", df, ws, gov)
    if declared:
        return declared, "Override"
    return infer_criticality(df, facts, env), "Inferred"


def assessment_confidence(df):
    """Data completeness of the evidence behind a readiness class."""
    inspected = df.get("definition_status") == "Collected"
    engine = compute_engine_state(df.get("compute_engine_behavior")) != "Unknown"
    refresh = df.get("refresh_history_status") == "Collected"
    if inspected and engine and refresh:
        return "High"
    return "Low" if not inspected and not engine else "Medium"


def _name(a):
    return a["df"].get("dataflow_name") or a["df"]["dataflow_id"]


def sequence_graph(A, edges):
    """Dataflow -> dataflow adjacency among the dataflows being planned."""
    succ, preds = defaultdict(set), defaultdict(set)
    for e in edges:
        if e["src_type"] == "Dataflow" and e["dst_type"] == "Dataflow" and e["src_id"] in A and e["dst_id"] in A and e["src_id"] != e["dst_id"]:
            succ[e["src_id"]].add(e["dst_id"])
            preds[e["dst_id"]].add(e["src_id"])
    return succ, preds


def propagate_sequence(A, preds, comps):
    """Rule SEQ-001: upstream dataflows migrate first, so a dataflow reading an Unsupported or Deferred one is Deferred."""
    comp_of = {m: i for i, comp in enumerate(comps) for m in comp}
    for i, comp in enumerate(comps):
        blockers = {p: A[p]["klass"] for m in comp for p in preds.get(m, ()) if comp_of[p] != i and A[p]["klass"] in BLOCKING_CLASSES}
        if len(comp) > 1:
            blockers.update({m: A[m]["klass"] for m in comp if A[m]["klass"] in BLOCKING_CLASSES})
        for m in comp:
            others = {p: k for p, k in blockers.items() if p != m}
            if others and A[m]["klass"] not in BLOCKING_CLASSES:
                names = ", ".join(f"{_name(A[p])} ({k})" for p, k in sorted(others.items())[:4])
                A[m]["hits"].append(make_hit("SEQ-001", evidence=f"blocked by {names}", count=len(others)))
                A[m]["klass"], A[m]["reason"], A[m]["rec_path"] = "Deferred", f"Blocked by upstream: {names}", "FixThenReassess"


def dependency_levels(comps, preds):
    comp_of = {m: i for i, comp in enumerate(comps) for m in comp}
    level = {}
    for i, comp in enumerate(comps):
        ups = {comp_of[p] for m in comp for p in preds.get(m, ()) if comp_of[p] != i}
        level[i] = 1 + max((level[u] for u in ups), default=-1)
    return {m: level[i] for i, comp in enumerate(comps) for m in comp}


def base_wave(a):
    klass, band, crit = a["klass"], a["band"], a["crit"]
    if klass == "Deferred":
        return 99, f"Deferred: {a['reason']}"
    if klass == "Unsupported":
        return 4, f"Remediate or rebuild: {a['reason']}"
    heavy = band in ("High", "Very High")
    if klass == "Ready":
        if heavy:
            return 3, f"Ready but {band} complexity"
        if band == "Low" and crit != "High":
            return 1, "Ready, low complexity and not business critical: quick win"
        return 2, "Ready, business critical" if crit == "High" else f"Ready, {band} complexity"
    if heavy or crit == "High":
        return 3, f"Needs attention, {band} complexity" + (", High criticality" if crit == "High" else "")
    return 2, f"Needs attention, {band} complexity"


def pick_pilot(cfg, A, succ, preds):
    """Representative low-risk quick wins: isolated, never Prod, a verified healthy refresh baseline first (then the unverified ones to
    fill the pilot), few consumers, spread over workspaces (at most a third of the pilot per workspace)."""
    if cfg.pilot_size <= 0:
        return []
    cap = max(1, cfg.pilot_size // 3)
    candidates = []
    for did, a in A.items():
        df = a["df"]
        failing = df.get("last_refresh_status") == "Failed" or bool(df.get("consecutive_failures"))
        if a["base_wave"] == 1 and not succ.get(did) and not preds.get(did) and a["env"] != "Prod" and not failing:
            consumers = len(a["facts"]["datasets_all"]) + len(a["facts"]["reports_all"]) + len(a["facts"]["dashboards_all"])
            verified = df.get("last_success_at") is not None
            candidates.append(((0 if verified else 1, ENV_RANK.get(a["env"], 2), 0 if 1 <= consumers <= 5 else 1, a["score"], a["ws_name"] or "", _name(a), did), did))
    chosen, per_workspace = [], Counter()
    for _, did in sorted(candidates):
        wid = A[did]["df"]["workspace_id"]
        if len(chosen) < cfg.pilot_size and per_workspace[wid] < cap:
            chosen.append(did)
            per_workspace[wid] += 1
    return chosen


def plan_waves(cfg, A, succ, preds, comps, pins):
    """Assign a wave to every planned dataflow. Planned waves (0-4) respect the dependency order; Deferred (99) is the backlog,
    outside the order (a blocked dataflow waits there until its upstream is fixed and the assessment re-run)."""
    for a in A.values():
        a["base_wave"], a["wave_reason"] = base_wave(a)
    for did in pick_pilot(cfg, A, succ, preds):
        a = A[did]
        a["base_wave"] = 0
        baseline = "" if a["df"].get("last_success_at") is not None else " (refresh baseline not verified)"
        a["wave_reason"] = (f"Pilot: low-risk {a['env']} dataflow with {len(a['facts']['datasets_all']) + len(a['facts']['reports_all'])} "
                            f"downstream model(s) / report(s){baseline}")
    comp_of = {m: i for i, comp in enumerate(comps) for m in comp}
    for i, comp in enumerate(comps):
        floor, pusher = -1, None
        for m in comp:
            for p in sorted(preds.get(m, ())):
                if comp_of[p] != i and A[p]["wave"] != 99 and A[p]["wave"] > floor:
                    floor, pusher = A[p]["wave"], p
        movable = [m for m in comp if A[m]["klass"] not in BLOCKING_CLASSES]
        group_wave = max([A[m]["base_wave"] for m in movable] + [floor]) if movable else None
        for m in comp:
            a = A[m]
            if a["klass"] in BLOCKING_CLASSES:
                a["wave"] = a["base_wave"]
                continue
            wave, reason = group_wave, a["wave_reason"]
            if wave > a["base_wave"]:
                reason += (f"; moved from wave {a['base_wave']} to {wave} because upstream '{_name(A[pusher])}' is planned in wave {floor}"
                           if floor >= max(A[x]["base_wave"] for x in movable) else f"; moved from wave {a['base_wave']} to {wave} to migrate linked dataflows together")
            pin = pins.get(m)
            if pin in PINNABLE_WAVES:
                reason = f"Pinned to wave {pin} by governance" + (f" (upstream '{_name(A[pusher])}' is planned later, in wave {floor})" if pin < floor else "")
                wave = pin
            a["wave"], a["wave_reason"] = wave, reason
    for a in A.values():
        a["wave_label"] = WAVE_LABELS.get(a["wave"], f"Wave {a['wave']}")
        a["batch_id"] = f"W{a['wave']:02d}-{a['df']['workspace_id']}"

## 7.4 Analysis engine
`run_analysis` turns the Silver rows into every Gold table. It is deterministic and side-effect free (inputs in, rows out),
so it is unit-tested without Spark and can be re-run on its own (`p_run_mode = "analysis_only"`).

In [ ]:
FEATURE_FAMILIES = {"CAP": "Capacity", "STO": "AttachedStorage", "ECE": "EnhancedCompute", "DQ": "DirectQuery", "INC": "IncrementalRefresh",
                    "LNK": "LinkedEntities", "CMP": "ComputedEntities", "QRY": "QueryLimit", "CON": "Connections", "GW": "Gateway",
                    "OWN": "Ownership", "NAM": "Naming", "SCH": "Schedule", "TRG": "ExternalTriggers", "PIP": "Pipelines", "REB": "Rebind",
                    "AIX": "AIInsights", "REF": "RefreshFailures", "STA": "Stale", "SEQ": "UpstreamBlocked", "DAT": "DefinitionNotInspected"}
REBIND_ACTIONS = {
    "Dataflow": "Open the downstream dataflow in the editor and save it so that its linked entities bind to the migrated dataflow",
    "SemanticModel": "Re-point the semantic model source to the migrated dataflow (connector PowerPlatform.Dataflows), take over if needed, then refresh",
    "Datamart": "Re-point the datamart source to the migrated dataflow, then refresh",
    "DataPipeline": "Update the Dataflow refresh activity to the id of the migrated dataflow",
}


def dataflow_cycles(edges):
    """Ids of dataflows that are part of a dependency cycle (a dataflow that reads itself included)."""
    succ, ids, reads_itself = defaultdict(set), set(), set()
    for e in edges:
        if e.get("src_type") == "Dataflow" and e.get("dst_type") == "Dataflow":
            if e["src_id"] == e["dst_id"]:
                reads_itself.add(e["src_id"])
            else:
                succ[e["src_id"]].add(e["dst_id"])
                ids.update((e["src_id"], e["dst_id"]))
    return {m for comp in strongly_connected_components(sorted(ids), succ) if len(comp) > 1 for m in comp} | reads_itself


def migration_state(df, migrations, gen2_targets):
    """(scope, migration_status, gen2_dataflow_id) of a dataflow."""
    did, kind = df["dataflow_id"], df.get("dataflow_type")
    if kind == "Gen2":
        if did in gen2_targets:
            return SCOPE_OUT, "SaveAsCopy", None
        return SCOPE_OUT, "Migrated" if df.get("first_seen_dataflow_type") == "Gen1" else None, None
    if kind != "Gen1":
        return SCOPE_UNKNOWN, None, None
    attempt = migrations.get(did)
    if attempt and attempt["ok"]:
        return SCOPE_IN, "MigratedBySaveAs", attempt["gen2"]
    return SCOPE_IN, "SaveAsFailed" if attempt else "NotStarted", None


def assess_dataflows(cfg, now, dfs, wss, sources, edges, graph, gov, migrations, cycles):
    """Rules, readiness class, score, effort, recommendation and wave for every Gen1 dataflow that is still to be migrated."""
    weights, bands = resolve_scoring(cfg)
    patterns = {**DEFAULT_ENV_PATTERNS, **(cfg.environment_patterns or {})}
    names = Counter((r["workspace_id"], (r.get("dataflow_name") or "").strip().lower()) for r in dfs.values() if r.get("dataflow_type") == "Gen1")
    A = {}
    for did, df in dfs.items():
        if df.get("dataflow_type") != "Gen1" or migrations.get(did, {}).get("ok"):
            continue
        ws = wss.get(df["workspace_id"])
        facts = direct_facts(df, graph, dfs)
        hits = evaluate_rules(cfg, now, df, ws, sources.get(did, []), facts, graph, dfs, names[(df["workspace_id"], (df.get("dataflow_name") or "").strip().lower())])
        env = resolve_environment(df, ws, gov, patterns)
        crit, crit_source = resolve_criticality(df, ws, facts, env, gov)
        A[did] = {"df": df, "ws": ws, "ws_name": (ws or {}).get("workspace_name"), "facts": facts, "hits": hits, "env": env, "crit": crit, "crit_source": crit_source}
    succ, preds = sequence_graph(A, edges)
    comps = strongly_connected_components(sorted(A), succ)
    for did, a in A.items():
        statuses = {"wizard": path_status(a["hits"], "wizard"), "saveas": path_status(a["hits"], "saveas")}
        a["status"], a["path"] = statuses, choose_path(cfg, statuses, a["hits"])
        a["klass"], a["reason"], a["rec_path"] = classify_own(cfg, a["hits"], statuses, a["path"], gov["dataflow"].get(did))
    propagate_sequence(A, preds, comps)
    for did, a in A.items():
        listed = sources.get(did, [])
        a["components"] = {"dependencies": score_dependencies(a["facts"], did in cycles), "refresh_criticality": score_refresh(a["df"], a["crit"]),
                           "gateway": score_gateway(a["df"], listed), "source_complexity": score_sources(a["df"], listed),
                           "unsupported": score_unsupported(a["hits"], a["path"]), "technical": score_technical(a["df"])}
        a["score"], a["band"] = complexity(weights, a["components"], bands)
        a["effort"] = 0.25 if a["rec_path"] == "Retire" else estimate_effort(a["band"], a["hits"], a["path"])
        a["actions"] = build_actions(a["hits"], a["path"], a["klass"])
    pins = {}
    for did, a in A.items():
        pin = coerce_value("INT", governance_value("pin_wave", a["df"], a["ws"], gov))
        if pin is not None:
            pins[did] = pin
    plan_waves(cfg, A, succ, preds, comps, pins)
    levels = dependency_levels(comps, preds)
    groups = {}
    for members in weak_components(sorted(A), succ):
        if len(members) > 1:
            gid_ = "G" + short_hash("|".join(sorted(members)), 10)
            groups.update({m: (gid_, len(members)) for m in members})
    for did, a in A.items():
        a["level"], (a["group_id"], a["group_size"]) = levels.get(did, 0), groups.get(did, (None, 1))
        a["recommendation"] = build_recommendation(a["klass"], a["rec_path"], a["reason"], a["wave_label"])
    return A, succ, preds

In [ ]:
def unsupported_rows(a, now):
    """One row per (rule, entity). Hits that share both are merged (evidence joined, effort added) instead of overwriting each other."""
    rows = {}
    df = a["df"]
    for h in a["hits"]:
        rule = RULES[h["rule_id"]]
        effort = rule.effort_days * (h.get("count", 1) if rule.rule_id in PER_ITEM_RULES else 1)
        key = (rule.rule_id, h["entity_name"])
        if key in rows:
            row = rows[key]
            row["effort_days"] = round(row["effort_days"] + effort, 2)
            if h["evidence"] and h["evidence"] not in row["evidence"]:
                row["evidence"] = truncate(f"{row['evidence']}; {h['evidence']}" if row["evidence"] else h["evidence"], 1000)
            continue
        rows[key] = {
            "dataflow_id": df["dataflow_id"], "workspace_id": df["workspace_id"], "rule_id": rule.rule_id,
            "feature_code": FEATURE_FAMILIES.get(rule.rule_id.split("-")[0], "Other"), "feature_name": rule.name, "category": rule.category,
            "severity": rule.severity, "impact_wizard": rule.impact_wizard, "impact_saveas": rule.impact_saveas, "entity_name": h["entity_name"],
            "evidence": h["evidence"], "remediation": rule.remediation, "effort_days": round(effort, 2), "weight": rule.weight,
            "confidence": rule.confidence, "detected_at": now}
    return list(rows.values())


def readiness_row(now, a, migration):
    df, facts, comps = a["df"], a["facts"], a["components"]
    blocking = sorted({h["rule_id"] for h in a["hits"] if path_impact(RULES[h["rule_id"]], a["path"]) == "Unsupported"})
    attention = sorted({h["rule_id"] for h in a["hits"] if path_impact(RULES[h["rule_id"]], a["path"]) == "NeedsAttention"})
    cu_run, cu_month = estimate_cu(df)
    before, after = a["actions"]
    return {
        "dataflow_id": df["dataflow_id"], "workspace_id": df["workspace_id"], "dataflow_type": "Gen1", "migration_scope": SCOPE_IN,
        "migration_status": migration[0], "gen2_dataflow_id": migration[1], "readiness": READINESS_LABEL[a["klass"]], "readiness_reason": a["reason"],
        "status_wizard": STATUS_LABEL[a["status"]["wizard"]], "status_saveas": STATUS_LABEL[a["status"]["saveas"]],
        "predicted_wizard_status": WIZARD_NAME[a["status"]["wizard"]], "recommended_path": a["rec_path"], "recommendation": a["recommendation"],
        "top_actions": (before + after)[:6], "blocking_rules": blocking, "attention_rules": attention, "complexity_score": a["score"],
        "complexity_band": a["band"], "score_dependencies": comps["dependencies"], "score_refresh_criticality": comps["refresh_criticality"],
        "score_gateway": comps["gateway"], "score_source_complexity": comps["source_complexity"], "score_unsupported": comps["unsupported"],
        "score_technical": comps["technical"], "effort_days": a["effort"], "est_gen2_cu_seconds_per_run": cu_run, "est_gen2_cu_hours_per_month": cu_month,
        "wave": a["wave"], "wave_label": a["wave_label"], "wave_rationale": a["wave_reason"], "batch_id": a["batch_id"],
        "migration_group_id": a["group_id"], "migration_group_size": a["group_size"], "dependency_level": a["level"],
        "upstream_dataflow_count": len(facts["upstream_dataflows"]), "downstream_dataflow_count": len(facts["downstream_dataflows"]),
        "downstream_dataset_count": len(facts["datasets_all"]), "downstream_report_count": len(facts["reports_all"]),
        "downstream_dashboard_count": len(facts["dashboards_all"]), "downstream_other_count": len(facts["other_all"]),
        "rebind_item_count": len(facts["rebind_items"]), "business_criticality": a["crit"], "criticality_source": a["crit_source"],
        "environment": a["env"], "assessment_confidence": assessment_confidence(df), "rules_version": RULES_VERSION, "assessed_at": now}


def passive_row(now, df, facts, scope, migration, env, crit, entry=None):
    """Readiness row for dataflows that are not planned: already Gen2, unknown generation or already copied with Save-As."""
    if migration[0] == "MigratedBySaveAs":
        copies = (entry or {}).get("copies") or []
        note = f"Copied with Save-As to {migration[1]}: re-point the downstream items (Assessment_RebindWorklist), validate, then retire this Gen1 dataflow."
        if len(copies) > 1:
            note += f" {len(copies)} Gen2 copies exist ({', '.join(copies[:4])}): keep one and delete the others."
        if entry and not entry.get("verified"):
            note += " No Gen2 copy was found in the inventory (deleted, or not scanned yet)."
    elif migration[0] == "SaveAsCopy":
        note = "Gen2 copy created with Save-As: validate it, re-enable its schedule and re-point the downstream items."
    elif migration[0] == "Migrated":
        note = "Already Gen2 (migrated)."
    elif scope == SCOPE_UNKNOWN:
        note = "Generation could not be determined: check access to the workspace and re-run discovery."
    else:
        note = "Native Dataflow Gen2: out of scope."
    return {"dataflow_id": df["dataflow_id"], "workspace_id": df["workspace_id"], "dataflow_type": df.get("dataflow_type") or "Unknown",
            "migration_scope": scope, "migration_status": migration[0], "gen2_dataflow_id": migration[1], "recommended_path": "None",
            "recommendation": note, "upstream_dataflow_count": len(facts["upstream_dataflows"]), "downstream_dataflow_count": len(facts["downstream_dataflows"]),
            "downstream_dataset_count": len(facts["datasets_all"]), "downstream_report_count": len(facts["reports_all"]),
            "downstream_dashboard_count": len(facts["dashboards_all"]), "downstream_other_count": len(facts["other_all"]),
            "rebind_item_count": len(facts["rebind_items"]), "business_criticality": crit, "environment": env, "rules_version": RULES_VERSION, "assessed_at": now}


def summary_row(df, ws, ready, unsupported_count, gov):
    ws = ws or {}
    ok = None if df.get("last_refresh_status") is None else (df["last_refresh_status"] == "Success" and not (df.get("consecutive_failures") or 0))
    keep = ("upstream_dataflow_count", "downstream_dataflow_count", "downstream_dataset_count", "downstream_report_count", "downstream_dashboard_count",
            "migration_scope", "readiness", "predicted_wizard_status", "complexity_score", "complexity_band", "effort_days", "recommended_path",
            "wave", "wave_label", "batch_id", "migration_status", "assessment_confidence", "business_criticality")
    row = {
        "dataflow_id": df["dataflow_id"], "workspace_id": df["workspace_id"], "workspace_name": ws.get("workspace_name"), "capacity_id": ws.get("capacity_id"),
        "capacity_name": ws.get("capacity_name"), "capacity_sku": ws.get("capacity_sku"), "domain_name": ws.get("domain_name"),
        "environment": ready.get("environment") or ws.get("environment"), "workspace_classification": ws.get("workspace_classification"),
        "support_team": governance_value("support_team", df, ws, gov), "business_owner": governance_value("business_owner", df, ws, gov),
        "dataflow_name": df.get("dataflow_name"), "dataflow_type": df.get("dataflow_type"),
        "owner_upn": (gov["dataflow"].get(df["dataflow_id"]) or {}).get("technical_owner") or df.get("owner_upn"), "created_at": df.get("created_at"),
        "modified_at": df.get("modified_at"), "description": df.get("description"), "tags": df.get("tags"), "last_refresh_at": df.get("last_refresh_at"),
        "last_refresh_status": df.get("last_refresh_status"), "last_refresh_duration_sec": df.get("last_refresh_duration_sec"),
        "refresh_success_rate": df.get("refresh_success_rate"), "refresh_ok": ok, "schedule_enabled": df.get("schedule_enabled"),
        "schedule_runs_per_day": df.get("schedule_runs_per_day"), "source_count": df.get("source_count"), "uses_gateway": df.get("uses_gateway"),
        "query_count": df.get("query_count"), "entity_count": df.get("entity_count"), "custom_function_count": df.get("custom_function_count"),
        "parameter_count": df.get("parameter_count"), "incremental_refresh_count": df.get("incremental_refresh_count"),
        "computed_entity_count": df.get("computed_entity_count"), "linked_entity_count": df.get("linked_entity_count"),
        "unsupported_feature_count": unsupported_count, "wizard_status_actual": governance_value("wizard_status_actual", df, ws, gov)}
    row.update({k: ready.get(k) for k in keep})
    return row

In [ ]:
MAX_DEPENDENCY_TOTAL = 500000


def dependency_rows(graph, dfs, wss, cycles):
    """Closure rows per dataflow and direction: at most MAX_DEPENDENCY_ROWS each (nearest first) and MAX_DEPENDENCY_TOTAL overall; once
    the overall budget is spent only the direct (hop 1) relations are listed. Returns (rows, number of closures that may be incomplete)."""
    rows, truncated = [], 0
    for did in sorted(d for d in dfs if d in graph.nodes):
        df = dfs[did]
        anchor = _label(did, graph, dfs)
        for direction in ("UPSTREAM", "DOWNSTREAM"):
            arrow = " <- " if direction == "UPSTREAM" else " -> "
            if len(rows) >= MAX_DEPENDENCY_TOTAL:
                reached, _ = graph.closure_info(did, direction, max_depth=1, limit=MAX_DEPENDENCY_ROWS)
                cut = True
            else:
                reached, cut = graph.closure_info(did, direction, limit=min(MAX_DEPENDENCY_ROWS, MAX_DEPENDENCY_TOTAL - len(rows)))
            truncated += bool(cut)
            for nid, hop, edge, path in reached:
                node = graph.nodes.get(nid, {})
                rel_ws = node.get("workspace_id")
                rows.append({
                    "dataflow_id": did, "workspace_id": df["workspace_id"], "direction": direction, "hop": hop,
                    "related_asset_type": node.get("type") or "Other", "related_asset_id": nid, "related_asset_name": _label(nid, graph, dfs),
                    "related_workspace_id": rel_ws, "related_workspace_name": (wss.get(rel_ws) or {}).get("workspace_name"),
                    "relationship": edge["relationship"], "crosses_workspace": bool(rel_ws and rel_ws != df["workspace_id"]),
                    "related_generation": (dfs.get(nid) or {}).get("generation"),
                    "path": truncate(arrow.join([anchor] + [_label(p, graph, dfs) for p in path[1:]]), 1000), "is_cyclic": did in cycles})
    return rows, truncated


def node_rows(graph, dfs, wss, ready_by_id):
    rows = []
    for nid in sorted(set(graph.nodes) | set(dfs)):
        df, node = dfs.get(nid), graph.nodes.get(nid, {})
        ws_id = (df or {}).get("workspace_id") or node.get("workspace_id")
        ready = ready_by_id.get(nid) or {}
        rows.append({
            "node_id": nid, "node_type": "Dataflow" if df else (node.get("type") or "Other"), "node_name": _label(nid, graph, dfs), "workspace_id": ws_id,
            "workspace_name": (wss.get(ws_id) or {}).get("workspace_name"), "generation": (df or {}).get("generation") if df else None,
            "readiness": ready.get("readiness"), "wave": ready.get("wave"), "in_degree": len({e["src_id"] for e in graph.up.get(nid, [])}),
            "out_degree": len({e["dst_id"] for e in graph.down.get(nid, [])})})
    return rows


def batch_environment(members):
    """Environment of a batch: the one its dataflows share (Unknown ignored when another is known), Mixed when they disagree."""
    envs = {m["env"] for m in members} - {"Unknown"}
    if not envs:
        return "Unknown"
    return next(iter(envs)) if len(envs) == 1 else "Mixed"


def wave_plan_rows(A, wss):
    batches = defaultdict(list)
    for a in A.values():
        batches[(a["wave"], a["df"]["workspace_id"])].append(a)
    rows = []
    for (wave, wid), members in sorted(batches.items(), key=lambda kv: (kv[0][0], str(kv[0][1]))):
        ws = wss.get(wid) or {}
        consumers = set()
        for m in members:
            consumers |= m["facts"]["datasets_all"] | m["facts"]["reports_all"] | m["facts"]["dashboards_all"]
        count = Counter(m["klass"] for m in members)
        rows.append({
            "wave": wave, "wave_label": WAVE_LABELS.get(wave, f"Wave {wave}"), "workspace_id": wid, "workspace_name": ws.get("workspace_name"),
            "capacity_name": ws.get("capacity_name"), "environment": batch_environment(members), "dataflow_count": len(members),
            "ready_count": count["Ready"], "needs_attention_count": count["NeedsAttention"], "unsupported_count": count["Unsupported"],
            "deferred_count": count["Deferred"], "total_effort_days": round(sum(m["effort"] for m in members), 2),
            "max_complexity_score": max(m["score"] for m in members), "downstream_item_count": len(consumers),
            "path_mix": ", ".join(f"{p}:{n}" for p, n in sorted(Counter(m["rec_path"] for m in members).items()))})
    return rows


def backlog_rows(A, preds, migrations):
    ordered = sorted(A.values(), key=lambda a: (a["wave"], a["level"], a["ws_name"] or "", _name(a), a["df"]["dataflow_id"]))
    rows = []
    for seq, a in enumerate(ordered, start=1):
        did = a["df"]["dataflow_id"]
        blocked = sorted(preds.get(did, ()))
        before, after = a["actions"]
        rows.append({
            "seq": seq, "wave": a["wave"], "wave_label": a["wave_label"], "batch_id": a["batch_id"], "workspace_id": a["df"]["workspace_id"],
            "workspace_name": a["ws_name"], "dataflow_id": did, "dataflow_name": a["df"].get("dataflow_name"), "owner_upn": a["df"].get("owner_upn"),
            "readiness": READINESS_LABEL[a["klass"]], "recommended_path": a["rec_path"], "can_start_now": a["klass"] == "Ready" and not blocked,
            "blocked_by": blocked, "pre_migration_actions": before, "post_migration_actions": after, "effort_days": a["effort"],
            "complexity_band": a["band"], "business_criticality": a["crit"], "environment": a["env"],
            "migration_status": "SaveAsFailed" if did in migrations else "NotStarted", "gen2_dataflow_id": None})
    return rows


def rebind_rows(A, dfs, graph, wss, migrations):
    """Consumers to re-point: planned dataflows on their recommended path, plus Gen1 dataflows already copied with Save-As."""
    targets = [(did, a["df"], a["facts"], a["wave"], a["rec_path"], a["ws_name"]) for did, a in A.items()
               if a["klass"] in ("Ready", "NeedsAttention") and a["rec_path"] in ("UpgradeWizard", "SaveAs")]
    for did, attempt in sorted(migrations.items()):
        df = dfs.get(did)
        if attempt["ok"] and df and df.get("dataflow_type") == "Gen1":
            targets.append((did, df, direct_facts(df, graph, dfs), None, "SaveAs", (wss.get(df["workspace_id"]) or {}).get("workspace_name")))
    rows = []
    for did, df, facts, wave, path, ws_name in targets:
        for e in facts["rebind_items"]:
            if path == "UpgradeWizard" and e["dst_type"] != "Dataflow":
                continue
            rows.append({
                "dataflow_id": did, "dataflow_name": df.get("dataflow_name"), "workspace_id": df["workspace_id"], "workspace_name": ws_name, "wave": wave,
                "path": path, "consumer_type": e["dst_type"], "consumer_id": e["dst_id"], "consumer_name": e.get("dst_name"),
                "consumer_workspace_id": e.get("dst_workspace_id"), "consumer_workspace_name": (wss.get(e.get("dst_workspace_id")) or {}).get("workspace_name"),
                "uses_legacy_connector": bool(e["attrs"].get("legacy_connector")), "storage_mode": e["attrs"].get("storage_mode"),
                "action": REBIND_ACTIONS[e["dst_type"]], "effort_days": RULES["REB-001"].effort_days, "gen2_dataflow_id": (migrations.get(did) or {}).get("gen2")})
    return rows

In [ ]:
def conform(table_name, rows):
    """Full column set per row (missing columns become NULL). Unknown keys are a programming error: they would be dropped silently."""
    table = TABLES[table_name]
    known = {c[0] for c in table.cols}
    out = []
    for row in rows:
        stray = set(row) - known
        if stray:
            raise ValueError(f"{table_name}: unknown columns {sorted(stray)}")
        out.append({c: row.get(c) for c in known})
    return out


def run_analysis(cfg, now, data):
    """Silver rows in, Gold rows out. data: dataflows, workspaces, sources, edges, overrides, migrations (lists of dict rows).
    Defensive on purpose: rows without a key are skipped, a dataflow without a workspace is planned under UNKNOWN_WORKSPACE, edges without
    both endpoints are ignored (counted in the stats), and every input is sorted by its key so that row order never changes a result."""
    raw_dataflows = [r for r in data["dataflows"] if r.get("dataflow_id")]
    dfs = {}
    for r in sorted(raw_dataflows, key=lambda r: r["dataflow_id"]):
        dfs[r["dataflow_id"]] = r if r.get("workspace_id") else {**r, "workspace_id": UNKNOWN_WORKSPACE}
    wss = {r["workspace_id"]: r for r in sorted((r for r in data["workspaces"] if r.get("workspace_id")), key=lambda r: r["workspace_id"])}
    sources = defaultdict(list)
    for s in sorted((s for s in data["sources"] if s.get("dataflow_id")), key=lambda s: (s["dataflow_id"], str(s.get("source_key")))):
        sources[s["dataflow_id"]].append(s)
    edges = dedupe_edges(data["edges"])
    incomplete_edges = sum(1 for e in data["edges"] if not (e.get("src_id") and e.get("dst_id")))
    graph = LineageGraph(edges)
    gov = governance_index(data["overrides"])
    known_gen2 = {did for did, r in dfs.items() if r.get("dataflow_type") == "Gen2"}
    migrations = index_migrations(data.get("migrations"), known_gen2)
    gen2_targets = {c for m in migrations.values() for c in m["copies"]}
    cycles = dataflow_cycles(edges)
    A, succ, preds = assess_dataflows(cfg, now, dfs, wss, sources, edges, graph, gov, migrations, cycles)
    patterns = {**DEFAULT_ENV_PATTERNS, **(cfg.environment_patterns or {})}

    readiness, unsupported = [], []
    for did, df in dfs.items():
        scope, status, gen2_id = migration_state(df, migrations, gen2_targets)
        if did in A:
            readiness.append(readiness_row(now, A[did], (status, gen2_id)))
            unsupported.extend(unsupported_rows(A[did], now))
        else:
            facts = direct_facts(df, graph, dfs)
            ws = wss.get(df["workspace_id"])
            env = resolve_environment(df, ws, gov, patterns)
            readiness.append(passive_row(now, df, facts, scope, (status, gen2_id), env, governance_value("business_criticality", df, ws, gov), migrations.get(did)))
    ready_by_id = {r["dataflow_id"]: r for r in readiness}
    feature_counts = Counter(r["dataflow_id"] for r in unsupported if "NeedsAttention" in (r["impact_wizard"], r["impact_saveas"]) or "Unsupported" in (r["impact_wizard"], r["impact_saveas"]))
    summary = [summary_row(df, wss.get(df["workspace_id"]), ready_by_id[did], feature_counts[did], gov) for did, df in dfs.items()]
    dependencies, truncated = dependency_rows(graph, dfs, wss, cycles)
    tables = {
        "Assessment_MigrationReadiness": readiness, "Assessment_UnsupportedFeatures": unsupported, "Assessment_DataflowSummary": summary,
        "Assessment_DataflowDependencies": dependencies, "Assessment_LineageNodes": node_rows(graph, dfs, wss, ready_by_id),
        "Assessment_WavePlan": wave_plan_rows(A, wss), "Assessment_MigrationBacklog": backlog_rows(A, preds, {k: v for k, v in migrations.items() if not v["ok"]}),
        "Assessment_RebindWorklist": rebind_rows(A, dfs, graph, wss, migrations), "Assessment_RuleCatalog": rule_catalog_rows()}
    tables = {name: conform(name, rows) for name, rows in tables.items()}
    classes = Counter(r["readiness"] for r in readiness if r.get("readiness"))
    stats = {"dataflows": len(dfs), "planned": len(A), "classes": dict(classes), "cycles": len(cycles), "dependency_truncated": truncated,
             "incomplete_edges": incomplete_edges, "dataflows_without_workspace": sum(1 for r in raw_dataflows if not r.get("workspace_id")),
             "migrated": sum(1 for r in readiness if r.get("migration_status") in ("Migrated", "MigratedBySaveAs")),
             "effort_days": round(sum(a["effort"] for a in A.values()), 2)}
    return {"tables": tables, "stats": stats}


GOLD_TABLES = ("Assessment_MigrationReadiness", "Assessment_UnsupportedFeatures", "Assessment_DataflowSummary", "Assessment_DataflowDependencies",
               "Assessment_LineageNodes", "Assessment_WavePlan", "Assessment_MigrationBacklog", "Assessment_RebindWorklist", "Assessment_RuleCatalog")


def load_analysis_inputs(ctx):
    lake = ctx.lake
    return {"dataflows": lake.read("Assessment_Dataflows"), "workspaces": lake.read("Assessment_Workspaces"),
            "sources": lake.read("Assessment_DataflowSources"), "edges": lake.read("Assessment_Lineage"),
            "overrides": lake.read("Assessment_GovernanceOverrides"), "migrations": lake.read_external(ctx.cfg.migration_results_table)}


def phase_analysis(ctx):
    data = load_analysis_inputs(ctx)
    if data["migrations"] is None:
        ctx.logger.info("no Save-As results table %r: migration status derives from type changes only", ctx.cfg.migration_results_table)
    result = run_analysis(ctx.cfg, ctx.clock.now(), data)
    ctx.state["analysis"] = result
    stats = result["stats"]
    if stats["dependency_truncated"]:
        ctx.logger.warning("%d dependency closures may be incomplete (at most %d rows per dataflow and direction, %d overall, nearest assets first; "
                           "the counts in Assessment_MigrationReadiness stay exact)", stats["dependency_truncated"], MAX_DEPENDENCY_ROWS, MAX_DEPENDENCY_TOTAL)
    if stats["incomplete_edges"]:
        ctx.logger.warning("%d lineage rows without a source or a target id were ignored", stats["incomplete_edges"])
    if stats["dataflows_without_workspace"]:
        ctx.logger.warning("%d dataflows have no workspace id: they are planned under workspace %r", stats["dataflows_without_workspace"], UNKNOWN_WORKSPACE)
    classes = ", ".join(f"{k}={v}" for k, v in sorted(stats["classes"].items())) or "no in-scope dataflow"
    return {"rows_in": len(data["dataflows"]), "rows_out": sum(len(v) for v in result["tables"].values()),
            "message": f"{stats['planned']} planned of {stats['dataflows']} dataflows: {classes}; effort {stats['effort_days']} d"}

### Run - section 7: migration readiness analysis

In [ ]:
run_steps(ctx, sections={7})

# 8. Delta persistence
Silver tables are written by the collection phases (partial-column `MERGE` + soft deletes). This section publishes the **Gold**
tables produced by the analysis:

* full snapshot per run (`INSERT OVERWRITE`): Gold is derived data, always consistent with the latest Silver and rules,
* row counts are verified after each write (a mismatch fails the phase),
* `OPTIMIZE ... ZORDER BY` (+ optional `VACUUM`) keeps files compact; V-Order and optimized writes are enabled by the storage layer.

| Layer | Tables | Write pattern |
|---|---|---|
| Silver (current state) | Capacities, Workspaces, Dataflows, DataflowSources, Lineage | `MERGE` by business key, soft delete |
| Silver (history) | RefreshHistory (partitioned by month) | `MERGE` by (dataflow, refresh id) |
| Gold (assessment) | MigrationReadiness, UnsupportedFeatures, DataflowDependencies, LineageNodes, DataflowSummary, WavePlan, MigrationBacklog, RebindWorklist, RuleCatalog | snapshot per run |
| Control | Checkpoint, RunLog, ErrorLog, RunSummary, GovernanceOverrides | merge / append |

In [ ]:
def expected_row_count(table_name, rows):
    keys = TABLES[table_name].keys
    return len({tuple(r.get(k) for k in keys) for r in rows})


def phase_publish(ctx):
    analysis = ctx.state.get("analysis")
    if not analysis:
        raise DeaError("no analysis result to publish: the analysis phase did not run or failed")
    written, problems = {}, []
    for name in GOLD_TABLES:
        rows = analysis["tables"][name]
        ctx.lake.replace(name, rows)
        written[name] = ctx.lake.count(name)
        expected = expected_row_count(name, rows)
        ctx.logger.info("%-34s %7d rows", name, written[name])
        if written[name] != expected:
            problems.append(f"{name}: {written[name]} rows stored, {expected} expected")
            ctx.record_error("publish", "Table", name, None, message=problems[-1])
    ctx.state["published"] = written
    metrics = {"rows_in": sum(expected_row_count(n, analysis["tables"][n]) for n in GOLD_TABLES), "rows_out": sum(written.values()),
               "message": f"{len(written)} Gold tables published"}
    if problems:
        metrics.update(status="Failed", message="; ".join(problems))
    return metrics


def phase_maintenance(ctx):
    """OPTIMIZE (Z-ORDER where defined) and optional VACUUM. Maintenance never fails the run; the message states what really ran
    (Partial when a table could not be optimized)."""
    hours = ctx.cfg.vacuum_retention_hours
    optimized = vacuumed = failed = 0
    for name in TABLES:
        done = ctx.lake.maintain(name, hours)
        if done is None:                       # this lake has nothing to maintain (in-memory)
            continue
        optimized += "OPTIMIZE" in done
        vacuumed += "VACUUM" in done
        failed += "OPTIMIZE" not in done
    return {"rows_in": len(TABLES), "rows_out": optimized, "status": "Partial" if failed else "Succeeded",
            "message": f"{optimized} of {len(TABLES)} tables optimized, {vacuumed} vacuumed" + (f" (retention {hours} h)" if hours else "")}

### Run - section 8: Delta persistence

In [ ]:
run_steps(ctx, sections={8})

# 9. Reporting output
Everything is derived from the Gold / Silver tables, so it also works after `p_run_mode = "analysis_only"`.

| Output | Where | Purpose |
|---|---|---|
| KPI summary (Markdown, also displayed in the notebook) | `Files/dea/reports/<run>/summary.md`, `kpis.json` | steering committee view: inventory, readiness, waves, top findings |
| Excel workbook | `Files/dea/reports/<run>/dataflow_assessment.xlsx` (CSV files when pandas / openpyxl are missing) | working file for the migration team |
| Lineage graph | `lineage.mmd` (Mermaid), `lineage.dot` (Graphviz), `lineage.svg` when the `dot` binary exists | the most connected linked dataflows and their consumers, coloured by readiness |

The Delta tables are the system of record: point a Power BI semantic model (Direct Lake) at `Assessment_DataflowSummary`,
`Assessment_MigrationReadiness`, `Assessment_UnsupportedFeatures`, `Assessment_WavePlan` and `Assessment_LineageNodes` / `Assessment_Lineage`.

In [ ]:
import shutil
import subprocess

READINESS_ORDER = ["Ready", "Needs Attention", "Unsupported", "Deferred"]
WIZARD_ORDER = ["Ready to migrate", "Needs Attention", "Upgrade unavailable"]
BAND_ORDER = ["Low", "Medium", "High", "Very High"]
LEVEL_ORDER = ["High", "Medium", "Low"]
EXPORT_SHEETS = [("Dataflows", "Assessment_DataflowSummary"), ("Readiness", "Assessment_MigrationReadiness"), ("Findings", "Assessment_UnsupportedFeatures"),
                 ("Waves", "Assessment_WavePlan"), ("Backlog", "Assessment_MigrationBacklog"), ("Rebind", "Assessment_RebindWorklist"),
                 ("Sources", "Assessment_DataflowSources"), ("Lineage", "Assessment_Lineage"), ("Workspaces", "Assessment_Workspaces"),
                 ("Rules", "Assessment_RuleCatalog")]
REPORT_TABLES = sorted({name for _, name in EXPORT_SHEETS} | {"Assessment_Capacities", "Assessment_LineageNodes"})


def count_by(rows, key, order=None):
    counts = Counter(r.get(key) if r.get(key) not in (None, "") else "Unknown" for r in rows)
    ordered = list(order or [])
    return {k: counts.get(k, 0) for k in ordered + sorted(k for k in counts if k not in ordered)}


def pct(part, total):
    return round(100.0 * part / total, 1) if total else 0.0


def mean(values):
    values = [v for v in values if v is not None]
    return round(sum(values) / len(values), 1) if values else None


def compute_kpis(tables, top_n=25, run_id=None, now=None):
    summary, ready, findings = tables["Assessment_DataflowSummary"], tables["Assessment_MigrationReadiness"], tables["Assessment_UnsupportedFeatures"]
    rules = {r["rule_id"]: r for r in tables["Assessment_RuleCatalog"]}
    planned = [r for r in ready if r.get("readiness")]
    planned_ids = {r["dataflow_id"] for r in planned}
    summary_by_id = {r["dataflow_id"]: r for r in summary}
    planned_summary = [summary_by_id[i] for i in planned_ids if i in summary_by_id]
    migrated = [r for r in ready if r.get("migration_status") in ("Migrated", "MigratedBySaveAs")]
    families = defaultdict(set)
    for f in findings:
        if f["impact_wizard"] != "None" or f["impact_saveas"] != "None":
            families[f["rule_id"]].add(f["dataflow_id"])
    waves = defaultdict(lambda: {"dataflows": 0, "workspaces": 0, "ready": 0, "needs_attention": 0, "unsupported": 0, "deferred": 0, "effort_days": 0.0, "label": ""})
    for w in tables["Assessment_WavePlan"]:
        agg = waves[w["wave"]]
        agg["label"] = w["wave_label"]
        agg["workspaces"] += 1
        for key, col in (("dataflows", "dataflow_count"), ("ready", "ready_count"), ("needs_attention", "needs_attention_count"),
                         ("unsupported", "unsupported_count"), ("deferred", "deferred_count")):
            agg[key] += w[col] or 0
        agg["effort_days"] = round(agg["effort_days"] + (w["total_effort_days"] or 0.0), 2)
    consumers = lambda s: (s.get("downstream_dataset_count") or 0) + (s.get("downstream_report_count") or 0) + (s.get("downstream_dashboard_count") or 0)  # noqa: E731
    total_program = len(planned) + len(migrated)
    return {
        "run_id": run_id, "generated_at": now.isoformat(timespec="seconds") if now else None,
        "inventory": {"capacities": len(tables["Assessment_Capacities"]), "workspaces": len(tables["Assessment_Workspaces"]),
                      "workspaces_with_dataflows": len({s["workspace_id"] for s in summary}), "dataflows": len(summary),
                      "gen1": sum(1 for s in summary if s.get("dataflow_type") == "Gen1"), "gen2": sum(1 for s in summary if s.get("dataflow_type") == "Gen2"),
                      "unknown_generation": sum(1 for s in summary if s.get("dataflow_type") not in ("Gen1", "Gen2"))},
        "readiness": count_by(planned, "readiness", READINESS_ORDER), "wizard_prediction": count_by(planned, "predicted_wizard_status", WIZARD_ORDER),
        "paths": count_by(planned, "recommended_path"), "complexity": count_by(planned, "complexity_band", BAND_ORDER),
        "avg_complexity_score": mean(r.get("complexity_score") for r in planned), "effort_days_total": round(sum(r.get("effort_days") or 0.0 for r in planned), 1),
        "migration": {"in_program": total_program, "migrated": len(migrated), "remaining": len(planned), "progress_pct": pct(len(migrated), total_program),
                      "save_as_failed": sum(1 for r in ready if r.get("migration_status") == "SaveAsFailed")},
        "waves": [{"wave": k, **v} for k, v in sorted(waves.items())],
        "top_findings": [{"rule_id": rid, "name": rules.get(rid, {}).get("name", rid), "severity": rules.get(rid, {}).get("severity"),
                          "category": rules.get(rid, {}).get("category"), "dataflows": len(ids), "share_pct": pct(len(ids), len(planned))}
                         for rid, ids in sorted(families.items(), key=lambda kv: (-len(kv[1]), kv[0]))[:top_n]],
        "top_complex": [{"dataflow": r.get("dataflow_id"), "name": summary_by_id.get(r["dataflow_id"], {}).get("dataflow_name"),
                         "workspace": summary_by_id.get(r["dataflow_id"], {}).get("workspace_name"), "score": r["complexity_score"], "band": r["complexity_band"],
                         "readiness": r["readiness"], "wave": r["wave"]}
                        for r in sorted(planned, key=lambda r: (-(r.get("complexity_score") or 0), r["dataflow_id"]))[:top_n]],
        "top_impact": [{"dataflow": s["dataflow_id"], "name": s.get("dataflow_name"), "workspace": s.get("workspace_name"), "models": s.get("downstream_dataset_count"),
                        "reports": s.get("downstream_report_count"), "dashboards": s.get("downstream_dashboard_count"), "readiness": s.get("readiness")}
                       for s in sorted(planned_summary, key=lambda s: (-consumers(s), s["dataflow_id"]))[:top_n] if consumers(s)],
        "environment": count_by(planned, "environment"), "criticality": count_by(planned, "business_criticality", LEVEL_ORDER),
        "capacity": count_by([{"capacity": s.get("capacity_name") or "(shared or unresolved)"} for s in planned_summary], "capacity"),
        "refresh": {"failing": sum(1 for s in planned_summary if s.get("refresh_ok") is False), "never_refreshed": sum(1 for s in planned_summary if s.get("last_refresh_status") is None),
                    "scheduled": sum(1 for s in planned_summary if s.get("schedule_enabled")), "avg_last_duration_min": mean((s.get("last_refresh_duration_sec") or 0) / 60.0
                                                                                                                     for s in planned_summary if s.get("last_refresh_duration_sec") is not None)},
        "connectivity": {"dataflows_with_gateway": sum(1 for s in planned_summary if s.get("uses_gateway")), "source_categories": count_by(tables["Assessment_DataflowSources"], "source_category")},
        "data_quality": {"confidence": count_by(planned, "assessment_confidence", LEVEL_ORDER), "definition_not_inspected": len(families.get("DAT-001", ())),
                         "unknown_generation": sum(1 for s in summary if s.get("dataflow_type") not in ("Gen1", "Gen2"))},
    }


def _cell(value):
    text = "" if value is None else (f"{value:.1f}" if isinstance(value, float) else str(value))
    return text.replace("|", "\\|").replace("\n", " ")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    lines += ["| " + " | ".join(_cell(v) for v in row) + " |" for row in rows]
    return "\n".join(lines)


def dict_table(title, counts, total=None):
    total = total if total is not None else sum(counts.values())
    return md_table([title, "Dataflows", "Share %"], [(k, v, pct(v, total)) for k, v in counts.items()])


def render_markdown(k):
    inv, mig = k["inventory"], k["migration"]
    parts = [f"# Dataflow Gen1 to Gen2 migration assessment\nRun `{k['run_id']}` - generated {k['generated_at']}",
             "## Inventory\n" + md_table(["Capacities", "Workspaces", "With dataflows", "Dataflows", "Gen1", "Gen2", "Unknown"],
                                         [(inv["capacities"], inv["workspaces"], inv["workspaces_with_dataflows"], inv["dataflows"], inv["gen1"], inv["gen2"], inv["unknown_generation"])]),
             f"## Migration progress\n{mig['migrated']} migrated, {mig['remaining']} remaining ({mig['progress_pct']} % of {mig['in_program']}); "
             f"{mig['save_as_failed']} Save-As failures. Estimated remaining effort: **{k['effort_days_total']} person-days**, average complexity {k['avg_complexity_score']}.",
             "## Readiness of the remaining Gen1 dataflows\n" + dict_table("Readiness", k["readiness"]),
             "## Predicted Upgrade Wizard assessment\n" + dict_table("Wizard status", k["wizard_prediction"]),
             "## Recommended path\n" + dict_table("Path", k["paths"]), "## Complexity\n" + dict_table("Band", k["complexity"]),
             "## Migration waves\n" + md_table(["Wave", "Label", "Dataflows", "Workspaces", "Ready", "Needs Attention", "Unsupported", "Deferred", "Effort (days)"],
                                                [(w["wave"], w["label"], w["dataflows"], w["workspaces"], w["ready"], w["needs_attention"], w["unsupported"], w["deferred"], w["effort_days"]) for w in k["waves"]]),
             "## Most frequent findings\n" + md_table(["Rule", "Finding", "Severity", "Category", "Dataflows", "Share %"],
                                                      [(f["rule_id"], f["name"], f["severity"], f["category"], f["dataflows"], f["share_pct"]) for f in k["top_findings"]]),
             "## Most complex dataflows\n" + md_table(["Dataflow", "Workspace", "Score", "Band", "Readiness", "Wave"],
                                                      [(t["name"] or t["dataflow"], t["workspace"], t["score"], t["band"], t["readiness"], t["wave"]) for t in k["top_complex"]]),
             "## Highest downstream impact\n" + md_table(["Dataflow", "Workspace", "Models", "Reports", "Dashboards", "Readiness"],
                                                         [(t["name"] or t["dataflow"], t["workspace"], t["models"], t["reports"], t["dashboards"], t["readiness"]) for t in k["top_impact"]]),
             "## Governance\n" + dict_table("Environment", k["environment"]) + "\n\n" + dict_table("Business criticality", k["criticality"]) + "\n\n" + dict_table("Capacity", k["capacity"]),
             f"## Refresh and connectivity\nFailing refresh: {k['refresh']['failing']}; never refreshed: {k['refresh']['never_refreshed']}; scheduled: {k['refresh']['scheduled']}; "
             f"average last duration: {k['refresh']['avg_last_duration_min']} min; dataflows behind a gateway: {k['connectivity']['dataflows_with_gateway']}.\n\n"
             + dict_table("Source category", k["connectivity"]["source_categories"]),
             "## Data quality\n" + dict_table("Assessment confidence", k["data_quality"]["confidence"])
             + f"\n\nDefinitions not inspected: {k['data_quality']['definition_not_inspected']}; unknown generation: {k['data_quality']['unknown_generation']}."]
    return "\n\n".join(parts) + "\n"


def show_markdown(text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except Exception:  # noqa: BLE001 - outside a notebook the text is simply printed
        print(text)

In [ ]:
def kpi_rows(kpis):
    """Flatten the KPI dictionary to (section, metric, value) rows for the Summary sheet."""
    rows = []
    for section, value in kpis.items():
        if isinstance(value, dict):
            for key, inner in value.items():
                if isinstance(inner, dict):
                    rows += [(f"{section}.{key}", k2, v2) for k2, v2 in inner.items()]
                else:
                    rows.append((section, key, inner))
        elif not isinstance(value, list):
            rows.append(("run", section, value))
    return rows


def excel_value(value):
    if isinstance(value, (list, tuple, set)):
        value = "; ".join(str(v) for v in value)
    if isinstance(value, dt.datetime) and value.tzinfo is not None:
        value = value.astimezone(dt.timezone.utc).replace(tzinfo=None)
    if isinstance(value, str):
        value = value[:32000]
        if value[:1] in ("=", "+", "-", "@", "\t", "\r"):
            value = "'" + value   # spreadsheet formula injection: tenant-controlled names must never become formulas
    return value


MAX_SHEET_ROWS = 200000   # larger detail tables go to a CSV file next to the workbook (Excel itself stops at 1 048 575 rows)


def sheet_view(tables, name):
    cols = [c[0] for c in TABLES[name].cols]
    return cols, [[excel_value(r.get(c)) for c in cols] for r in tables[name]]


def oversized_sheets(tables, limit=None):
    limit = MAX_SHEET_ROWS if limit is None else limit
    return [sheet for sheet, name in EXPORT_SHEETS if len(tables[name]) > limit]


def build_workbook(tables, kpis):
    """Excel workbook as bytes, or None when pandas / openpyxl are unavailable. A sheet with more than MAX_SHEET_ROWS rows only carries
    a note: its rows are written as CSV by export_reports."""
    try:
        import pandas as pd
        import openpyxl  # noqa: F401
    except ImportError:
        return None
    too_big = set(oversized_sheets(tables))
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        pd.DataFrame([(a, b, excel_value(c)) for a, b, c in kpi_rows(kpis)], columns=["Section", "Metric", "Value"]).to_excel(writer, sheet_name="Summary", index=False)
        for sheet, name in EXPORT_SHEETS:
            if sheet in too_big:
                note = f"{len(tables[name])} rows: too many for a worksheet, see {sheet.lower()}.csv next to this workbook"
                pd.DataFrame([(note,)], columns=["Note"]).to_excel(writer, sheet_name=sheet, index=False)
                continue
            cols, data = sheet_view(tables, name)
            pd.DataFrame(data, columns=cols).to_excel(writer, sheet_name=sheet, index=False)
        for sheet in writer.book.worksheets:
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            for column in sheet.iter_cols(min_row=1, max_row=200):
                width = max((len(str(c.value)) for c in column if c.value is not None), default=8)
                sheet.column_dimensions[column[0].column_letter].width = min(60, width + 2)
    return buffer.getvalue()


def write_csv(files, rel, cols, data):
    out = io.StringIO()
    writer = csv.writer(out)
    writer.writerow(cols)
    writer.writerows(data)
    return files.write_bytes(rel, ("\ufeff" + out.getvalue()).encode("utf-8"))


def export_reports(ctx, tables, kpis, markdown):
    base = f"reports/{ctx.run_id}"
    written = [ctx.files.write_text(f"{base}/summary.md", markdown), ctx.files.write_json(f"{base}/kpis.json", kpis, compress=False)]
    workbook = build_workbook(tables, kpis)
    if workbook is not None:
        written.append(ctx.files.write_bytes(f"{base}/dataflow_assessment.xlsx", workbook))
        csv_sheets = oversized_sheets(tables)
    else:
        ctx.logger.warning("pandas / openpyxl not available: writing CSV files instead of the Excel workbook")
        csv_sheets = [sheet for sheet, _ in EXPORT_SHEETS]
    for sheet, name in EXPORT_SHEETS:
        if sheet in csv_sheets:
            cols, data = sheet_view(tables, name)
            written.append(write_csv(ctx.files, f"{base}/{sheet.lower()}.csv", cols, data))
            if workbook is not None:
                ctx.logger.warning("%s has %d rows: written as %s.csv, the workbook sheet only carries a note", sheet, len(data), sheet.lower())
    return written

In [ ]:
NODE_FILL = {"Ready": "#c8e6c9", "Needs Attention": "#fff3cd", "Unsupported": "#f8d7da", "Deferred": "#e0e0e0"}
TYPE_FILL = {"Dataflow": "#e3f2fd", "SemanticModel": "#ede7f6", "Report": "#fce4ec", "Dashboard": "#f3e5f5", "DataPipeline": "#e8f5e9", "Datamart": "#fff8e1"}
CLASS_OF = {"Ready": "ready", "Needs Attention": "attention", "Unsupported": "unsupported", "Deferred": "deferred"}


def lineage_subgraph(nodes, edges, max_nodes):
    """The most connected linked dataflows first, then their neighbours and the reports behind their models, up to max_nodes.
    Data sources are left out (they are in Assessment_DataflowSources)."""
    by_id = {n["node_id"]: n for n in nodes}
    pairs = sorted({(e["src_id"], e["dst_id"]) for e in edges if e["src_type"] != "Datasource" and e["src_id"] in by_id and e["dst_id"] in by_id})
    neighbours, successors, linked = defaultdict(set), defaultdict(set), Counter()
    for src, dst in pairs:
        neighbours[src].add(dst)
        neighbours[dst].add(src)
        successors[src].add(dst)
        if by_id[src]["node_type"] == "Dataflow" and by_id[dst]["node_type"] == "Dataflow":
            linked[src] += 1
            linked[dst] += 1
    seeds = sorted((i for i, n in by_id.items() if n["node_type"] == "Dataflow" and neighbours[i]), key=lambda i: (-linked[i], -len(neighbours[i]), i))
    chosen = []
    for seed in seeds:
        group = [seed] + sorted(neighbours[seed])
        for hop in sorted(successors[seed]):
            if by_id[hop]["node_type"] != "Dataflow":
                group += sorted(successors[hop])
        for node_id in group:
            if node_id not in chosen and len(chosen) < max_nodes:
                chosen.append(node_id)
        if len(chosen) >= max_nodes:
            break
    keep = set(chosen)
    return [by_id[i] for i in chosen], [(s, d) for s, d in pairs if s in keep and d in keep]


def _safe(text, limit=44):
    return truncate(re.sub(r'["<>\[\]{}|`\\]', "'", str(text or "")), limit)


def _detail(node):
    detail = node["node_type"]
    if node.get("readiness"):
        detail += f" - {node['readiness']}" + (f" - wave {node['wave']}" if node.get("wave") is not None else "")
    elif node["node_type"] == "Dataflow" and node.get("generation") == 2:
        detail += " - Gen2"
    return detail


def to_mermaid(nodes, edges):
    ids = {n["node_id"]: f"n{i}" for i, n in enumerate(nodes)}
    lines = ["flowchart LR"]
    lines += [f'  {ids[n["node_id"]]}["{_safe(n["node_name"])}<br/><i>{_safe(_detail(n), 60)}</i>"]' for n in nodes]
    lines += [f"  {ids[s]} --> {ids[d]}" for s, d in edges]
    lines += [f"  classDef {c} fill:{NODE_FILL[r]},stroke:#555" for r, c in CLASS_OF.items()]
    for readiness, klass in CLASS_OF.items():
        members = [ids[n["node_id"]] for n in nodes if n.get("readiness") == readiness]
        if members:
            lines.append(f"  class {','.join(members)} {klass}")
    return "\n".join(lines) + "\n"


def to_dot(nodes, edges):
    ids = {n["node_id"]: f"n{i}" for i, n in enumerate(nodes)}
    lines = ["digraph lineage {", "  rankdir=LR;", '  node [shape=box, style="rounded,filled", fontname="Segoe UI", fontsize=10];']
    for n in nodes:
        fill = NODE_FILL.get(n.get("readiness")) or TYPE_FILL.get(n["node_type"], "#ffffff")
        lines.append(f'  {ids[n["node_id"]]} [label="{_safe(n["node_name"])}\\n{_safe(_detail(n), 60)}", fillcolor="{fill}"];')
    lines += [f"  {ids[s]} -> {ids[d]};" for s, d in edges]
    return "\n".join(lines + ["}"]) + "\n"


def render_svg(dot_text):
    """SVG bytes through the Graphviz `dot` binary when it exists in the runtime, else None."""
    binary = shutil.which("dot")
    if not binary:
        return None
    try:
        done = subprocess.run([binary, "-Tsvg"], input=dot_text.encode("utf-8"), capture_output=True, timeout=60, check=True)
        return done.stdout
    except (OSError, subprocess.SubprocessError):
        return None


def render_lineage(ctx, tables):
    nodes, edges = lineage_subgraph(tables["Assessment_LineageNodes"], tables["Assessment_Lineage"], ctx.cfg.lineage_graph_max_nodes)
    if not nodes:
        ctx.logger.info("no lineage to draw")
        return []
    base = f"reports/{ctx.run_id}"
    dot_text, mermaid = to_dot(nodes, edges), to_mermaid(nodes, edges)
    written = [ctx.files.write_text(f"{base}/lineage.mmd", mermaid), ctx.files.write_text(f"{base}/lineage.dot", dot_text)]
    svg = render_svg(dot_text)
    if svg:
        written.append(ctx.files.write_bytes(f"{base}/lineage.svg", svg))
        try:
            from IPython.display import SVG, display
            display(SVG(data=svg))
        except Exception:  # noqa: BLE001
            ctx.logger.info("lineage graph rendered to lineage.svg")
    else:
        ctx.logger.info("Graphviz `dot` binary not found: lineage written as Mermaid (lineage.mmd) and DOT (lineage.dot) only")
        show_markdown("### Lineage (Mermaid source)\n```mermaid\n" + mermaid + "```")
    return written


def phase_report(ctx):
    tables = {name: ctx.lake.read(name) for name in REPORT_TABLES}
    kpis = compute_kpis(tables, ctx.cfg.top_n, ctx.run_id, ctx.clock.now())
    markdown = render_markdown(kpis)
    ctx.state["kpis"] = kpis
    show_markdown(markdown)
    files = []
    if ctx.cfg.export_reports:
        files += export_reports(ctx, tables, kpis, markdown)
    if ctx.cfg.render_lineage_graph:
        files += render_lineage(ctx, tables)
    ctx.state["report_files"] = files
    return {"rows_in": kpis["inventory"]["dataflows"], "rows_out": len(files), "message": f"{len(files)} files written under reports/{ctx.run_id}"}

### Run - section 9: reporting output

In [ ]:
run_steps(ctx, sections={9})

# 10. Logging and monitoring
| Signal | Where | Notes |
|---|---|---|
| Structured console log | notebook output | `time | level | run id | phase | message`; run id and phase come from a logging filter |
| Phase log | `Assessment_RunLog` | status, duration, rows, API calls / retries / throttling, errors per phase |
| Error log | `Assessment_ErrorLog` | endpoint template, HTTP status, error code, truncated body; never tokens or secrets |
| Run KPIs | `Assessment_RunSummary` | one row per run: counts by readiness, effort, API usage - ready for trend and burn-down charts |
| Run report | `Files/dea/runs/<run id>.json` | everything above plus API statistics per bucket, access preflight and recent warnings |
| Pipeline signal | notebook exit value, `p_fail_on_partial` | a pipeline can branch on `status` or fail the activity |

Status semantics: **Succeeded** (all phases fine), **Partial** (a phase stopped early on the deadline, API budget or throttling:
the next run resumes from the checkpoints), **Failed** (a phase raised). A critical failure (authentication, preflight) aborts the run.

In [ ]:
class RunFailed(DeaError):
    """Raised at the very end of a run, after the logs are persisted, so pipelines see the failure."""


def build_run_summary(ctx, status):
    kpis = ctx.state.get("kpis") or {}
    stats = (ctx.state.get("analysis") or {}).get("stats") or {}
    inventory, readiness = kpis.get("inventory") or {}, kpis.get("readiness") or stats.get("classes") or {}
    totals = ctx.stats.totals()
    return {
        "run_id": ctx.run_id, "started_at": ctx.started_at, "ended_at": utcnow(), "status": status, "mode": ctx.cfg.run_mode,
        "workspaces_total": inventory.get("workspaces"), "workspaces_scanned": len(ctx.state.get("scanned_ws_ids") or ()),
        "dataflows_total": inventory.get("dataflows") if inventory else stats.get("dataflows"), "gen1_count": inventory.get("gen1"), "gen2_count": inventory.get("gen2"),
        "ready_count": readiness.get("Ready"), "needs_attention_count": readiness.get("Needs Attention"), "unsupported_count": readiness.get("Unsupported"),
        "deferred_count": readiness.get("Deferred"), "migrated_count": (kpis.get("migration") or {}).get("migrated", stats.get("migrated")),
        "avg_complexity_score": kpis.get("avg_complexity_score"), "total_effort_days": kpis.get("effort_days_total", stats.get("effort_days")),
        "api_calls": totals["calls"], "api_throttled": totals["throttled"], "error_count": len(ctx.error_rows), "notebook_version": NOTEBOOK_VERSION}


def run_log_rows(ctx):
    rows = [dict(r) for r in ctx.phase_rows]
    if rows:
        rows[0]["params_json"] = json.dumps(ctx.cfg.public_dict(), default=str, sort_keys=True)
    return rows


def run_report(ctx, status, summary):
    capture = getattr(ctx, "log_capture", None)
    return {"run_id": ctx.run_id, "status": status, "mode": ctx.cfg.run_mode, "dry_run": ctx.cfg.dry_run, "notebook_version": NOTEBOOK_VERSION,
            "rules_version": RULES_VERSION, "summary": summary, "phases": ctx.phase_rows, "access": ctx.state.get("access"),
            "api": {"totals": ctx.stats.totals(), "by_bucket": ctx.stats.by_bucket()}, "errors": ctx.error_rows[:200],
            "warnings": [list(w) for w in list(capture.records)[-100:]] if capture else [], "parameters": ctx.cfg.public_dict()}


def log_banner(ctx, status):
    log = ctx.logger.info
    log("=" * 98)
    log("run %s finished: %s (mode=%s%s)", ctx.run_id, status.upper(), ctx.cfg.run_mode, ", dry run" if ctx.cfg.dry_run else "")
    log("%-18s %-10s %8s %9s %9s %6s %5s %4s", "phase", "status", "sec", "rows in", "rows out", "api", "429", "err")
    for r in ctx.phase_rows:
        log("%-18s %-10s %8.1f %9s %9s %6s %5s %4s", r["phase"], r["status"], r.get("duration_sec") or 0.0, r.get("rows_in") or "-", r.get("rows_out") or "-",
            r.get("api_calls"), r.get("api_throttled"), r.get("errors"))
    totals = ctx.stats.totals()
    log("API: %s calls, %s retries, %s throttled (HTTP 429), %s failed; %s errors logged", totals["calls"], totals["retries"], totals["throttled"], totals["failures"], len(ctx.error_rows))
    for bucket, s in ctx.stats.by_bucket().items():
        log("   %-32s calls=%-5s retries=%-4s throttled=%-4s failed=%-4s avg=%sms", bucket, s["calls"], s["retries"], s["throttled"], s["failures"], s["avg_ms"])
    log("=" * 98)


def _monitor_state(ctx):
    """What finalize_run already did in this context, so that running it again (a retry of the last cell) completes the work instead of repeating it."""
    return ctx.state.setdefault("monitor", {"run_log": 0, "error_log": 0})


def attempt_number(ctx):
    """1 for a normal run, then 2, 3 ... for each resume of the same run id (counted from the reports already written); fixed per context."""
    state = _monitor_state(ctx)
    if "attempt" not in state:
        try:
            written = [f for f in ctx.files.list("runs") if os.path.basename(f).startswith(ctx.run_id)]
        except Exception:  # noqa: BLE001 - the number only names the report file
            written = []
        state["attempt"] = len(written) + 1
    return state["attempt"]


def cumulative_summary(ctx, summary):
    """A resumed run continues one KPI row: earliest start, cumulative API usage and error count. The earlier attempts are read once per
    context, so persisting again replaces this attempt's share instead of adding it a second time."""
    if ctx.cfg.run_mode != "resume":
        return summary
    state = _monitor_state(ctx)
    if "prior" not in state:
        try:
            rows = ctx.lake.read("Assessment_RunSummary", where={"run_id": ctx.run_id})
        except Exception as exc:  # noqa: BLE001
            ctx.logger.warning("could not read the previous KPI row of the resumed run: %s", exc)
            return summary
        state["prior"] = rows[0] if rows else None
    prior = state["prior"]
    if not prior:
        return summary
    merged = dict(summary)
    merged["started_at"] = prior.get("started_at") or summary["started_at"]
    for key in ("api_calls", "api_throttled", "error_count"):
        merged[key] = (prior.get(key) or 0) + (summary.get(key) or 0)
    return merged


def guarded(ctx, label, action, problems):
    """Run one persistence step on its own: a failure is logged and collected, and never prevents the other steps."""
    try:
        action()
    except Exception as exc:  # noqa: BLE001
        problems.append(f"{label}: {type(exc).__name__}: {exc}"[:300])
        ctx.logger.error("could not persist the %s: %s: %s", label, type(exc).__name__, exc)


def append_new(ctx, table, rows, counter):
    """Append the rows that this context has not stored yet (the lists only grow)."""
    state = _monitor_state(ctx)
    fresh = rows[state[counter]:]
    if fresh:
        ctx.lake.append(table, fresh)
    state[counter] = len(rows)


def finalize_run(ctx):
    """Persist checkpoints, run log, error log, KPIs and the JSON report, each step independently. Never raises: monitoring must not hide
    the run outcome; what could not be stored is returned in summary['persistence_problems'] and written to the report. Calling it again
    stores what is still missing, and rewrites the KPI row and the report of the same attempt."""
    status = "Failed" if getattr(ctx, "aborted", False) else ctx.overall_status()
    summary = cumulative_summary(ctx, build_run_summary(ctx, status))
    problems = []
    guarded(ctx, "checkpoints", ctx.checkpoints.flush, problems)
    guarded(ctx, "run log", lambda: append_new(ctx, "Assessment_RunLog", run_log_rows(ctx), "run_log"), problems)
    guarded(ctx, "error log", lambda: append_new(ctx, "Assessment_ErrorLog", list(ctx.error_rows), "error_log"), problems)
    guarded(ctx, "run summary", lambda: ctx.lake.merge("Assessment_RunSummary", [summary]), problems)
    attempt = attempt_number(ctx)
    report = {**run_report(ctx, status, summary), "attempt": attempt, "persistence_problems": problems}
    guarded(ctx, "run report", lambda: ctx.files.write_json(f"runs/{ctx.run_id}{'' if attempt == 1 else f'.attempt{attempt}'}.json", report, compress=False), problems)
    log_banner(ctx, status)
    return {**summary, "persistence_problems": problems}


def finish_run(ctx):
    """Section 10: persist everything, then fail the notebook if the run aborted (or, with p_fail_on_partial, did not fully succeed or
    could not be recorded)."""
    summary = finalize_run(ctx)
    if getattr(ctx, "aborted", False):
        raise RunFailed(f"run {ctx.run_id} aborted: {getattr(ctx, 'abort_reason', 'critical phase failed')}")
    if ctx.cfg.fail_on_partial and summary["status"] != "Succeeded":
        raise RunFailed(f"run {ctx.run_id} finished {summary['status']}: see Assessment_RunLog and Assessment_ErrorLog")
    if ctx.cfg.fail_on_partial and summary["persistence_problems"]:
        raise RunFailed(f"run {ctx.run_id} could not be fully recorded: " + "; ".join(summary["persistence_problems"]))
    return summary

### Run - section 10: logging and monitoring
Persists the run log, error log, KPIs and the JSON report, prints the run banner, returns the outcome to a calling pipeline
and fails the notebook when the run aborted (or, with `p_fail_on_partial`, did not fully succeed).

In [ ]:
SUMMARY = finish_run(ctx)
exit_notebook(globals(), SUMMARY)